# Classification of biosynthetic gene clusters (BGCs) by their Pfam domain profile

This notebook starts from **a single table** (`tabla_bgc-5.csv`, one row per BGC region) and, with no other input file:

1. **explores** the BGC collection and its Pfam domains;
2. **trains** a classifier (linear SVM with recursive feature elimination, RFE) that predicts each BGC's product category (`product_primary`) from its domains;
3. **evaluates** the model on BGCs it never saw during training;
4. **explores the results**: groups the BGCs, identifies the domains responsible for each group and compares the gene-by-gene architecture of a representative BGC of each group, with dedicated siderophore and sequence-similarity analyses;
5. **compares the clusters with one another**: heatmap of clusters against domains, recurrent domains and genes, synteny, and all clusters aligned in order.

## Structure

| Part | Question it answers | Sections | Results folder |
|---|---|---|---|
| **0 · Setup** | Installation, configuration and helper functions | 0.1 – 0.4 | — |
| **1 · Data analysis and exploration** | What is in the table? Can the classes be told apart by their domains? | 1.1 – 1.8 | `results/1_data/` |
| **2 · Model training** | How many domains does the model need? Is it overfitting? | 2.1 – 2.7 | `results/2_training/` |
| **3 · Analysis of the model results** | How well does it classify new BGCs, and where does it fail? | 3.1 – 3.5 | `results/3_evaluation/` |
| **4 · Results exploration: groups and architecture** | Which domains separate the BGC groups, and how does their architecture change? | 4.1 – 4.6 | `results/4_groups_architecture/` |
| **5 · Cluster comparison** | Which domains and genes recur, and how conserved is gene order? | 5.1 – 5.5 | `results/5_cluster_comparison/` |
| **Annex** | Exported model and list of generated files | A.1 – A.3 | `results/annex_model/` |

Every file starts with the number of the section that produces it (e.g. `2.2_feature_selection_curve.png`), so it maps straight back to the notebook. Every figure has a CSV table next to it with the values it shows.

## Input data

| Column | Use |
|---|---|
| `product_primary` | Class to predict: the BGC's product category assigned by antiSMASH |
| `PF…:…` (one per Pfam domain) | Domain score in the BGC; 0 = domain absent. These are the model's features |
| `assembly_accession`, `region_number`, `location` | Identify each region and let the GenBank (`GCA_`) and RefSeq (`GCF_`) copies of the same genome be merged |
| `protein_sequences` | BGC proteins separated by `\|` (sections 4.3 – 4.6 and Part 5) |
| `protein_seq_pfam_map` | The same proteins with every domain replaced by `[Domain(value)]`: the source of the gene-by-gene architecture (Part 4) and of gene order (Part 5) |

## How to run it

- **Google Colab:** upload this notebook and `tabla_bgc-5.csv` to the same folder and use *Runtime → Run all*. Cell 0.1 installs anything missing.
- **Locally:** `pip install -r requirements.txt` and open the notebook with Jupyter.
- **Run time:** about 8–11 minutes with 2 cores. The slowest parts are the leave-one-out cross-validation (Part 2), the alignments of section 4.6 and the protein clustering of section 5.3. For a quick run set `"cv_strategy"` to `"kfold"` in cell 0.2.

## Reproducibility

Everything random depends on `CONFIG["random_state"]`, so two runs on the same table give exactly the same numbers and figures.

## Limitations (worth reading before interpreting)

1. The train/test split is made per BGC, not per genome: near-identical BGCs from closely related strains can end up on both sides. Section 3.1 measures how many test BGCs have an identical domain profile in the training set, and the accuracy on those that do not.
2. antiSMASH assigns `product_primary` with domain-based rules; the model learns to reproduce that assignment from the Pfam domains.
3. The table does not store each gene's strand: in the architecture figures genes follow table order and every arrow points right. Functional categories are assigned per Pfam family from an editable list (section 4.2), not by manual annotation. In addition, the `protein_seq_pfam_map` column does not contain every domain recorded in the `PF…` columns (section 4.3 measures how many are missing): gene-by-gene architectures and gene names only include the domains in the map. The table does not provide per-gene coordinates either: the synteny and the alignments of Part 5 assume that the genes of each BGC are listed in their order in the genome, and their results depend on the similarity threshold used to decide that two genes are the same gene (section 5.3).
4. t-SNE is a descriptive tool: distances between groups and their size on the map are not interpreted.
5. Some CSV rows describe overlapping stretches of the same genome with different classes (e.g. an `nrps` BGC inside the coordinates of an `nrp-metallophore` BGC). Their domain profiles are almost identical, so the model cannot tell them apart. Section 1.3 counts them and section 3.4 flags the errors that fall in these cases.

## References

- Blin K, Shaw S, Augustijn HE, *et al.* (2023). antiSMASH 7.0: new and improved predictions for detection, regulation, chemical structures and visualisation. *Nucleic Acids Research* 51: W46–W50. doi:10.1093/nar/gkad344
- Mistry J, *et al.* (2021). Pfam: The protein families database in 2021. *Nucleic Acids Research* 49: D412–D419.
- Guyon I, Weston J, Barnhill S, Vapnik V (2002). Gene selection for cancer classification using support vector machines. *Machine Learning* 46: 389–422.
- Pedregosa F, *et al.* (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research* 12: 2825–2830.
- van der Maaten L, Hinton G (2008). Visualizing data using t-SNE. *Journal of Machine Learning Research* 9: 2579–2605.
- Cock PJA, *et al.* (2009). Biopython: freely available Python tools for computational molecular biology and bioinformatics. *Bioinformatics* 25: 1422–1423.
- Kralova S, Spacek P, Gafriller J, *et al.* (2026). Kineochelins—A new group of siderophores from an Antarctic bacterium. *Microbial Biotechnology* 19: e70386. doi:10.1111/1751-7915.70386 (reference for the architecture figure of section 4.3)
- Navarro-Muñoz JC, Selem-Mojica N, Mullowney MW, *et al.* (2020). A computational framework to explore large-scale biosynthetic diversity. *Nature Chemical Biology* 16: 60–68. doi:10.1038/s41589-019-0400-9 (BiG-SCAPE; formula of the adjacency index used in section 5.4)

---
# Part 0 · Setup

Installation, configuration and helper functions. **The only cell you normally need to edit is 0.2.**

## 0.1 Installation

Installs only the missing packages (in Google Colab usually just Biopython) without printing the `pip` log.

In [ ]:
import importlib.util, subprocess, sys

REQUIRED = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "seaborn": "seaborn", "joblib": "joblib", "Bio": "biopython"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Installed packages:", ", ".join(missing) if missing else "none (all present)")

## 0.2 Configuration

All analysis parameters, grouped by the notebook part that uses them. The defaults reproduce the study protocol (linear SVM, RFE, leave-one-out validation, one-standard-error rule).

In [ ]:
import numpy as np

CONFIG = {
    # ---- Input, output and reproducibility -----------------
    "input_csv":      "tabla_bgc-5.csv",     # the only input file
    "output_dir":     "results",  # one subfolder per notebook part
    "save_figures":   True,
    "figure_formats": ("png", "pdf"),
    "figure_dpi":     300,                   # resolution of the saved files
    "inline_max_width_px": 1400,             # maximum width (px) of a figure inside the notebook; saved files are not affected
    "show_individual_panels": False,         # True = also display the individual panels here (they are always saved)
    "random_state":   42,                    # single seed for everything random
    "n_jobs":         -1,                    # CPU cores (-1 = all)

    # ---- CSV columns --------------------------------------------------------
    "target_column":   "product_primary",          # class to predict
    "feature_prefix":  "PF",                       # numeric columns starting with this = Pfam domains
    "assembly_column": "assembly_accession",
    "region_column":   "region_number",
    "location_column": "location",
    "sequence_column": "protein_sequences",        # BGC proteins separated by "|"
    "pfam_map_column": "protein_seq_pfam_map",     # the same proteins with [Domain(value)]

    # ---- Part 1 · Data ------------------------------------------------------
    "collapse_assembly_redundancy": True,    # merge GenBank (GCA_) and RefSeq (GCF_) copies of the same genome
    "min_class_size":  10,                   # classes with fewer BGCs are excluded
    "drop_zero_variance": True,              # drop domains that never vary
    "tsne_perplexity_main": 30,              # perplexity of the main t-SNE maps
    "tsne_perplexities": (5, 15, 30),        # perplexities of the sensitivity figure (1.5c)
    "tsne_pca_components": 50,               # internal compression before t-SNE (None = none)
    "tsne_ellipses":   True,                 # draw ellipses on the t-SNE maps
    "ellipse_level":   0.95,                 # confidence level of each ellipse
    "corr_circle_fraction": 0.25,            # stratified fraction for the correlation circle
    "corr_circle_top_features": 15,          # domains (arrows) drawn in the circle

    # ---- Part 2 · Training ------------------------------------------
    "test_size":       0.30,                 # held-out test: not used until Part 3
    "validation_size": 0.20,                 # inner validation (taken from training)
    "svm_c":           1.0,                  # initial C of the linear SVM
    "rfe_step":        0.1,                  # fraction of domains removed per RFE round
    "cv_strategy":     "loocv",              # "loocv" (study protocol) or "kfold" (faster)
    "kfold_splits":    10,
    "kfold_repeats":   3,
    "feature_grid":    (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
                        22, 25, 28, 30, 35, 40, 45, 50, 60, 75, 100, 125, 150, 200),
    "tune_c":          True,                 # choose C by cross-validation
    "c_grid":          (0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 100.0),
    "learning_curve_sizes": np.linspace(0.15, 1.0, 8),
    "n_permutations":  100,                  # label permutations (permutation test)
    "max_generalization_gap": 0.05,          # tolerable gap between training and test accuracy

    # ---- Part 4 · Groups and architecture -----------------------------
    "n_groups":        3,                    # groups formed on the t-SNE
    "group_on":        "all",                # "all" = t-SNE with all domains · "signature" = with the model's domains
    "marker_domains_per_group": 5,           # marker domains shown per group
    "tree_max_depth":  3,                    # depth of the rule tree (readable)
    "tree_min_prevalence": 0.05,             # the tree only uses domains present in at least 5 % of BGCs
    "arch_representatives": 1,               # representative BGCs drawn per group
    "siderophore_classes": ("nrp-metallophore", "ni-siderophore"),   # siderophore classes (4.5)
    "siderophore_compare_classes": ("nrps",),                        # classes shown for comparison in 4.5
    "sid_min_bgcs":    3,                    # minimum BGCs of a class within a group to draw its representative
    "seq_per_class":   6,                    # BGCs per class in the alignment matrix (cost grows with n²)
    "seq_max_proteins_per_bgc": 10,          # the N longest proteins of each BGC are aligned
    "seq_min_length":  30,                   # shorter peptides are discarded (aa)
    "mantel_permutations": 999,              # Mantel test permutations

    # ---- Part 5 · Cluster comparison ---------------------------------------
    "heatmap_domains_per_class": 8,          # characteristic domains per class in the heatmap (5.1)
    "recurrent_top_domains": 30,             # most recurrent domains shown (5.2)
    "gene_family_min_similarity": 0.5,       # minimum similarity to the representative to belong to a gene family (5.3)
    "repeat_top_items": 15,                  # repeated genes / domains shown (5.3)
    "synteny_type_threshold": 0.5,           # minimum mean synteny index within a synteny type (5.4)
    "align_representatives": 4,              # synteny types whose representative is drawn to scale (5.5)
    "align_label_min_fraction": 0.25,        # the gene is named for columns shared by at least this fraction of the BGCs (5.5)
    "align_min_bgcs_shown": 3,               # loci with fewer BGCs are saved but not displayed in the notebook (5.5)
}

## 0.3 Libraries and modelling functions

Functions used throughout the notebook: results folders, the model (scaling + linear SVM), the cross-validation strategy, the Wilson interval and the metrics.

In [ ]:
import re, time, warnings
from pathlib import Path

import pandas as pd
from scipy.stats import chi2, fisher_exact, spearmanr
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE
from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report, cohen_kappa_score,
                             confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score,
                             roc_auc_score, silhouette_score)
from sklearn.model_selection import (LeaveOneOut, RepeatedStratifiedKFold, StratifiedKFold, cross_val_score,
                                     learning_curve, permutation_test_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")      # sklearn warnings about classes missing from small folds
RS = CONFIG["random_state"]
np.random.seed(RS)

# ---- Results folders: one per notebook part -------------------
OUT = Path(CONFIG["output_dir"])
PART_DIRS = {"1": OUT / "1_data", "2": OUT / "2_training", "3": OUT / "3_evaluation",
             "4": OUT / "4_groups_architecture", "5": OUT / "5_cluster_comparison",
             "A": OUT / "annex_model"}
for folder in PART_DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)


def out_path(section, name, ext):
    """"2.2", "curve" → results/2_training/2.2_curve.<ext>"""
    return PART_DIRS[section.split(".")[0]] / f"{section}_{name}.{ext}"


def save_table(df, section, name):
    """Saves the table as CSV next to its section's figures and returns it."""
    df.to_csv(out_path(section, name, "csv"), index=False)
    return df


def slug(text):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(text))


# ---- Model and validation -----------------------------------------------------------
def make_estimator(C=None, probability=False):
    """Scaling + linear SVM.
    The scaler sits inside the pipeline so that, under cross-validation, it is fitted only on the training part of
    each fold (no information leak)."""
    return Pipeline([("scaler", StandardScaler()),
                     ("svm", SVC(kernel="linear", C=CONFIG["svm_c"] if C is None else C,
                                 probability=probability, random_state=RS))])


def make_cv():
    """LOOCV (study protocol) or repeated stratified K-fold (faster)."""
    if CONFIG["cv_strategy"] == "loocv":
        return LeaveOneOut()
    return RepeatedStratifiedKFold(n_splits=CONFIG["kfold_splits"], n_repeats=CONFIG["kfold_repeats"], random_state=RS)


def cv_summary(scores):
    """Mean and standard error. Under LOOCV
    each fold scores 0 or 1, so the between-fold SD does not describe the model; the standard error (sd/√n) does."""
    scores = np.asarray(scores, float)
    se = scores.std(ddof=1) / np.sqrt(len(scores)) if len(scores) > 1 else 0.0
    return float(scores.mean()), float(se)


def wilson_interval(successes, n, z=1.96):
    """Wilson interval for a proportion (accuracy); better behaved than the normal one for small n."""
    p, d = successes / n, 1 + z ** 2 / n
    centre = (p + z ** 2 / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
    return max(0.0, centre - half), min(1.0, centre + half)


def compute_metrics(y_true, y_pred, classes, partition):
    """Overall metrics (one row) and per-class metrics (one row per class)."""
    kw = {"zero_division": 0, "labels": list(classes)}
    overall = pd.DataFrame([{
        "partition": partition, "n": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", **kw),
        "recall_macro": recall_score(y_true, y_pred, average="macro", **kw),
        "f1_macro": f1_score(y_true, y_pred, average="macro", **kw),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred)}]).round(4)
    report = classification_report(y_true, y_pred, labels=list(classes), output_dict=True, zero_division=0)
    per_class = pd.DataFrame([{"partition": partition, "class": c, "n": int(report[c]["support"]),
                               "precision": report[c]["precision"], "recall": report[c]["recall"],
                               "f1": report[c]["f1-score"]} for c in classes])
    per_class["errors"] = (per_class["n"] * (1 - per_class["recall"])).round().astype(int)
    per_class["error_rate"] = 1 - per_class["recall"]
    return overall, per_class.round(4)

## 0.4 Figure style and functions

Every figure shares the same style so they all read the same way:

- **Fixed colour per class.** The first 8 classes (largest to smallest) use a categorical palette validated for colour-vision deficiency. If there are more than 8 classes, from the ninth on a colour is reused with a different, hollow marker, so no class depends on colour alone.
- **Direct labels.** On the t-SNE maps every class has its name written over its main group, in addition to the legend.
- **One ellipse per compact group.** A class can appear in several places on the map, so each ellipse encloses a compact group of BGCs of the same class (95 %) instead of one ellipse per class that would cover empty areas.
- **Heatmaps** use a light-to-dark blue scale; for signed values (more or less than the rest), blue → grey → red.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, to_rgba
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse, FancyBboxPatch, Polygon

%matplotlib inline

# ---- Palette and ink (colour-blind validated categorical palette) --
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
CONTEXT = "#d9d8d2"                                         # context points
SEQ_CMAP = LinearSegmentedColormap.from_list("seq_blue", [SURFACE, "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5",
                                                          "#256abf", "#184f95", "#0d366b"])
DIV_CMAP = LinearSegmentedColormap.from_list("div_blue_red", ["#2a78d6", "#f0efec", "#e34948"])
HALO = [pe.withStroke(linewidth=3, foreground=SURFACE)]    # light outline so text reads over points

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "figure.dpi": 110,
    "font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.labelsize": 11, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-", "axes.axisbelow": True,
    "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.frameon": False, "legend.fontsize": 10,
    "lines.linewidth": 2, "lines.solid_capstyle": "round"})


def titles(ax, title, subtitle=None):
    """Title (what the figure shows) and subtitle (how to read it), left-aligned."""
    n_lines = 0 if not subtitle else subtitle.count("\n") + 1
    ax.set_title(title, loc="left", pad=10 + 15 * n_lines)
    if subtitle:
        ax.text(0, 1.012, subtitle, transform=ax.transAxes, ha="left", va="bottom", fontsize=10, color=INK2)


def show_and_save(fig, section, name, individual=False):
    """Saves the figure (png/pdf) in
    its part's folder and displays it. Individual panels are always saved and displayed only if
    CONFIG["show_individual_panels"] is True."""
    if CONFIG["save_figures"]:
        for fmt in CONFIG["figure_formats"]:
            fig.savefig(out_path(section, name, fmt), dpi=CONFIG["figure_dpi"], bbox_inches="tight")
    if individual and not CONFIG["show_individual_panels"]:
        plt.close(fig)
    else:
        if fig.get_figwidth() * fig.dpi > CONFIG["inline_max_width_px"]:        # very wide figures are displayed at a lower resolution
            fig.set_dpi(CONFIG["inline_max_width_px"] / fig.get_figwidth())
        plt.show()


def make_style(order, start=0):
    """Fixed colour and marker per category. Beyond 8
    categories a colour is reused with a different, hollow marker (composite colour × shape encoding)."""
    reuse = [(6, "D"), (0, "s"), (2, "^"), (5, "v"), (1, "P")]
    style = {}
    for i, c in enumerate(order):
        if start + i < len(SERIES):
            style[c] = {"color": SERIES[start + i], "marker": "o", "hollow": False}
        else:
            k, marker = reuse[(start + i - len(SERIES)) % len(reuse)]
            style[c] = {"color": SERIES[k], "marker": marker, "hollow": True}
    return style


def repel(xy, box, n_iter=400):
    """Pushes apart overlapping labels. `box` = a common (width, height) or one per label."""
    pos = np.asarray(xy, float).copy()
    wh = np.broadcast_to(np.asarray(box, float), pos.shape)
    for _ in range(n_iter):
        moved = False
        for i in range(len(pos)):
            for j in range(i + 1, len(pos)):
                bx, by = (wh[i] + wh[j]) / 2
                d = (pos[i] - pos[j]) / (bx, by)
                dist = np.hypot(*d)
                if dist < 1:
                    u = d / dist if dist > 1e-9 else np.array([0.0, 1.0])
                    step = u * (1 - dist) / 2 * np.array([bx, by])
                    pos[i] += step
                    pos[j] -= step
                    moved = True
        if not moved:
            break
    return pos


def compact_groups(pts, span, eps_frac=0.05, min_pts=3):
    """Compact groups of
    same-category points on the map (DBSCAN), largest first."""
    if len(pts) < min_pts:
        return []
    lab = DBSCAN(eps=eps_frac * span, min_samples=min_pts).fit_predict(pts)
    return [pts[lab == k] for k in sorted(set(lab) - {-1}, key=lambda k: -(lab == k).sum())]


def ellipse_patch(pts, span, color, level=None, fill=True):
    """Confidence ellipse (bivariate normal) of a group of points,
    with a minimum size so groups of identical points remain visible."""
    level = CONFIG["ellipse_level"] if level is None else level
    cov = np.cov(pts, rowvar=False) + np.eye(2) * (0.012 * span) ** 2
    vals, vecs = np.linalg.eigh(cov)
    vals, vecs = vals[::-1], vecs[:, ::-1]
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    width, height = 2 * np.sqrt(vals * chi2.ppf(level, df=2))
    return Ellipse(pts.mean(axis=0), width, height, angle=angle, lw=1.2, zorder=1,
                   facecolor=to_rgba(color, 0.08) if fill else "none", edgecolor=to_rgba(color, 0.95))


def cat_scatter(ax, E, labels, order, style, flip=(1, 1), ellipses=None, direct_labels=True, s=26,
                label_size=9.5, alpha=0.9):
    """2D map with one
    colour/marker per category, one ellipse per compact group and direct labels."""
    ellipses = CONFIG["tsne_ellipses"] if ellipses is None else ellipses
    E = np.asarray(E)[:, :2] * np.array(flip)
    span = float(np.ptp(E, axis=0).max())
    anchors = {}
    for c in order:
        m = labels == c
        if not m.any():
            continue
        st = style[c]
        ax.scatter(E[m, 0], E[m, 1], s=s * (1.6 if st["hollow"] else 1), marker=st["marker"], alpha=alpha,
                   facecolors="none" if st["hollow"] else st["color"],
                   edgecolors=st["color"] if st["hollow"] else SURFACE,
                   linewidths=1.3 if st["hollow"] else 0.6, zorder=3 if st["hollow"] else 2)
        groups = compact_groups(E[m], span)
        if ellipses:
            for g in groups:
                ax.add_patch(ellipse_patch(g, span, st["color"]))
        anchors[c] = (groups[0] if groups else E[m]).mean(axis=0)
    if direct_labels and anchors:
        keys = list(anchors)
        xy = np.array([anchors[k] for k in keys])
        rx, ry = np.ptp(E, axis=0)
        pos = repel(xy + [0, 0.045 * ry], (0.15 * rx, 0.05 * ry))
        for k, (x0, y0), (x1, y1) in zip(keys, xy, pos):
            if abs(x1 - x0) > 0.02 * rx or abs(y1 - y0 - 0.045 * ry) > 0.02 * ry:      # leader line only if the label moved
                ax.plot([x0, x1], [y0, y1 - 0.01 * ry], color=INK2, lw=0.6, zorder=4)
            ax.text(x1, y1, k, ha="center", va="bottom", fontsize=label_size, fontweight="bold", color=INK,
                    path_effects=HALO, zorder=5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)


def cat_handles(order, style):
    return [Line2D([], [], ls="", marker=style[c]["marker"], ms=8,
                   mfc="none" if style[c]["hollow"] else style[c]["color"],
                   mec=style[c]["color"] if style[c]["hollow"] else SURFACE, mew=1.3 if style[c]["hollow"] else 0.8)
            for c in order]


def cat_legend(ax, order, style, counts=None, title=None, anchor=(1.01, 0.5)):
    labels = [f"{c} ({counts[c]})" if counts is not None else c for c in order]
    ax.legend(cat_handles(order, style), labels, loc="center left", bbox_to_anchor=anchor, title=title,
              title_fontsize=10, labelcolor=INK2, handletextpad=0.4)


def fig_legend(fig, order, style, counts=None, ncol=5, y=-0.02):
    labels = [f"{c} ({counts[c]})" if counts is not None else c for c in order]
    fig.legend(cat_handles(order, style), labels, loc="upper center", bbox_to_anchor=(0.5, y), ncol=ncol,
               labelcolor=INK2, handletextpad=0.4, columnspacing=1.4)


def highlight_panel(ax, E, labels, c, style, s_fg=16):
    """One category highlighted (with its ellipses) over the rest in grey."""
    m = labels == c
    st = style[c]
    span = float(np.ptp(E[:, :2], axis=0).max())
    ax.scatter(E[~m, 0], E[~m, 1], s=7, c=CONTEXT, linewidths=0, zorder=1)
    ax.scatter(E[m, 0], E[m, 1], s=s_fg * (1.5 if st["hollow"] else 1), marker=st["marker"],
               facecolors="none" if st["hollow"] else st["color"],
               edgecolors=st["color"] if st["hollow"] else SURFACE, linewidths=1.2 if st["hollow"] else 0.5, zorder=3)
    for g in compact_groups(E[m, :2], span):
        ax.add_patch(ellipse_patch(g, span, st["color"], fill=False))
    ax.set_title(f"{c}  (n = {m.sum()})", loc="left", fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)


def hbox(ax, values, labels, color=None):
    """Horizontal box plot
    with the points on top (independent of the matplotlib version)."""
    color = SERIES[0] if color is None else color
    rng = np.random.default_rng(RS)
    for i, v in enumerate(values):
        v = np.asarray(v, float)
        q1, med, q3 = np.percentile(v, [25, 50, 75])
        iqr = q3 - q1
        lo, hi = v[v >= q1 - 1.5 * iqr].min(), v[v <= q3 + 1.5 * iqr].max()
        ax.plot([lo, q1], [i, i], color=color, lw=1)
        ax.plot([q3, hi], [i, i], color=color, lw=1)
        ax.add_patch(FancyBboxPatch((q1, i - 0.28), max(q3 - q1, 1e-9), 0.56, boxstyle="square,pad=0",
                                    facecolor=to_rgba(color, 0.12), edgecolor=color, lw=1.2, zorder=2))
        ax.plot([med, med], [i - 0.28, i + 0.28], color=color, lw=2.2, zorder=3)
        ax.scatter(v, i + rng.uniform(-0.2, 0.2, len(v)), s=9, color=color, alpha=0.35, linewidths=0, zorder=4)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_ylim(len(labels) - 0.5, -0.5)
    ax.grid(axis="y", visible=False)


def heatmap(ax, M, row_labels, col_labels, cmap=None, vmin=None, vmax=None, annot=None, fmt=".2f",
            cbar_label=None, center=None, annot_size=9, cbar=True):
    """Heatmap with a clear gap between cells. With cbar=False the
    colour bar is not drawn (so it can be added separately when the figure has more panels)."""
    sns.heatmap(np.asarray(M, float), ax=ax, cmap=SEQ_CMAP if cmap is None else cmap, vmin=vmin, vmax=vmax,
                center=center, annot=annot if annot is not None else True, fmt=fmt if annot is None else "",
                annot_kws={"fontsize": annot_size}, linewidths=2, linecolor=SURFACE,
                xticklabels=col_labels, yticklabels=row_labels, cbar=cbar,
                cbar_kws={"label": cbar_label, "shrink": 0.75} if cbar_label else {"shrink": 0.75})
    ax.tick_params(length=0)
    ax.grid(False)


def rotate_xlabels(ax, angle=40):
    """Tilted x-axis labels aligned with their column."""
    plt.setp(ax.get_xticklabels(), rotation=angle, ha="right", rotation_mode="anchor")


def tsne_axes(ax, sx=1, sy=1):
    ax.set_xlabel(("−" if sx < 0 else "") + "t-SNE 1")
    ax.set_ylabel(("−" if sy < 0 else "") + "t-SNE 2")


def run_tsne(X_in, perplexity, n_components=2):
    """t-SNE with a fixed seed. The data are first compressed to
    `tsne_pca_components` dimensions, a standard internal step that removes noise and speeds things up."""
    X_red = X_in
    if CONFIG["tsne_pca_components"]:
        k = min(CONFIG["tsne_pca_components"], X_in.shape[1], X_in.shape[0] - 1)
        X_red = PCA(n_components=k, random_state=RS).fit_transform(X_in)
    perplexity = min(perplexity, (X_in.shape[0] - 1) / 3)       # t-SNE requires perplexity < n/3
    return TSNE(n_components=n_components, perplexity=perplexity, init="pca", learning_rate="auto",
                random_state=RS).fit_transform(X_red)

---
# Part 1 · Data analysis and exploration

**Question:** what is in the table, and can the BGC classes be told apart by their Pfam domains?

| Section | What it does | Main outputs |
|---|---|---|
| 1.1 | Loads the CSV | — |
| 1.2 | Merges the GenBank/RefSeq copies of the same genome | — |
| 1.3 | Counts BGCs per class, excludes rare classes and detects overlapping BGCs | `1.3_bgcs_per_class`, `1.3_overlapping_bgcs` |
| 1.4 | Builds the domain matrix and describes how many domains each BGC has | `1.4_domains_per_bgc` |
| 1.5 | t-SNE map by class (overview, class by class and perplexity sensitivity) | `1.5a/b/c` |
| 1.6 | The same t-SNE in its four orientations | `1.6_tsne_four_views` |
| 1.7 | 3D t-SNE | `1.7_tsne_3d` |
| 1.8 | Correlation circle before the model (25 % of the data) | `1.8_correlation_circle_before` |

## 1.1 Loading the CSV

Pfam domain columns (numeric, named `PF…:…`) are identified and the first rows of the metadata are shown.

In [ ]:
raw = pd.read_csv(CONFIG["input_csv"], encoding="utf-8-sig", low_memory=False)    # utf-8-sig removes the leading BOM
TARGET = CONFIG["target_column"]
assert TARGET in raw.columns, f"Missing column {TARGET}"

pfam_columns = [c for c in raw.columns
                if c.startswith(CONFIG["feature_prefix"]) and pd.api.types.is_numeric_dtype(raw[c])]
print(f"{len(raw):,} BGC regions · {len(pfam_columns):,} Pfam-domain columns · "
      f"{raw.shape[1] - len(pfam_columns)} metadata columns")
if "organism" in raw.columns:
    genera = raw["organism"].astype(str).str.split().str[0].value_counts()
    print(f"Organisms: {raw['organism'].nunique()} distinct names · genus/genera: "
          + ", ".join(f"{g} ({n} rows)" for g, n in genera.items()))

preview = [c for c in ("organism", CONFIG["assembly_column"], TARGET, CONFIG["region_column"], "length_kb", "num_cds_total")
           if c in raw.columns]
raw[preview].head()

## 1.2 GenBank / RefSeq copies of the same genome

NCBI distributes many genomes twice: `GCA_…` (GenBank) and `GCF_…` (RefSeq). Both copies yield the same BGCs with identical profiles. If both were kept, one copy could land in training and its twin in the test set, inflating accuracy. One row per genome, class, region and location is kept.

In [ ]:
ACC, REG, LOC = CONFIG["assembly_column"], CONFIG["region_column"], CONFIG["location_column"]
data = raw.copy()
data["genome"] = data[ACC].astype(str).str.replace(r"^GC[AF]_", "", regex=True)    # GCA_x.1 and GCF_x.1 → x.1

if CONFIG["collapse_assembly_redundancy"]:
    n_twins = int((data.groupby("genome")[ACC].nunique() > 1).sum())
    keys = ["genome", TARGET] + [c for c in (REG, LOC) if c in data.columns]
    data = data.drop_duplicates(subset=keys).reset_index(drop=True)
    print(f"{n_twins} genomes appear twice (GenBank and RefSeq) → "
          f"removed {len(raw) - len(data)} duplicate rows · remaining {len(data)} "
          f"BGCs from {data['genome'].nunique()} genomes")

## 1.3 BGCs per class

Classes with fewer than `min_class_size` BGCs are excluded: they are too small to be split across training, validation and test, and their metrics would not be interpretable. The class order set here (largest to smallest) and its colours are kept throughout the notebook.

In [ ]:
MIN_N = CONFIG["min_class_size"]
counts_all = data[TARGET].value_counts()
class_order = counts_all[counts_all >= MIN_N].index.tolist()      # fixed order for every figure
CLASS_STYLE = make_style(class_order)

inventory = save_table(pd.DataFrame({
    "class": counts_all.index, "n_BGC": counts_all.values,
    "percent": (100 * counts_all.values / counts_all.sum()).round(1),
    "status": np.where(counts_all.values >= MIN_N, "retained", f"excluded (< {MIN_N})")}),
    "1.3", "bgcs_per_class")
data = data[data[TARGET].isin(class_order)].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8.5, 0.36 * len(counts_all) + 1.8))
kept = counts_all.index.isin(class_order)
ax.barh(np.arange(len(counts_all)), counts_all.values, height=0.62, color=np.where(kept, SERIES[0], CONTEXT))
for i, v in enumerate(counts_all.values):
    ax.text(v + counts_all.max() * 0.01, i, str(v), va="center", fontsize=9.5, color=INK2)
ax.axvline(MIN_N, color=MUTED, lw=1)
ax.text(MIN_N + counts_all.max() * 0.01, len(counts_all) - 0.6, f"minimum = {MIN_N}", color=MUTED, fontsize=9)
ax.set_yticks(np.arange(len(counts_all)))
ax.set_yticklabels(counts_all.index)
ax.set_ylim(len(counts_all) - 0.4, -0.6)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Number of BGCs (after merging GenBank/RefSeq)")
titles(ax, "BGCs per product class",
       f"{len(data)} BGCs in {len(class_order)} retained classes (blue) · grey: classes with fewer than {MIN_N} BGCs, excluded")
show_and_save(fig, "1.3", "bgcs_per_class")
inventory

### Overlapping BGCs of the same genome

Some CSV rows describe genome stretches that overlap each other with different classes (for example, a BGC inside the coordinates of another BGC of the same genome). They are not removed, but it is worth knowing: if two rows cover almost the same stretch, their domain profiles are almost identical even though their classes differ, and no classifier can tell them apart. Table `1.3_overlapping_bgcs` lists every pair and section 3.4 shows which model errors fall in these cases.

Two BGCs are considered to overlap if they belong to the same genome and their coordinates (`location`) intersect.

In [ ]:
overlap_rows, overlap_classes = [], {}                          # overlap_classes[row] = classes of the BGCs overlapping it
if LOC in data.columns:
    coords = data[LOC].astype(str).str.extract(r"(\d+)\D+(\d+)").astype(float).to_numpy()
    for _, idx in data.groupby("genome").indices.items():
        for a in range(len(idx)):
            for b in range(a + 1, len(idx)):
                i, j = idx[a], idx[b]
                ov = min(coords[i, 1], coords[j, 1]) - max(coords[i, 0], coords[j, 0])
                if ov > 0:
                    shorter = min(coords[i, 1] - coords[i, 0], coords[j, 1] - coords[j, 0])
                    pair = sorted([(data.at[i, TARGET], i), (data.at[j, TARGET], j)])
                    overlap_rows.append({ACC: data.at[i, ACC], "class_a": pair[0][0], "row_a": pair[0][1],
                                         "class_b": pair[1][0], "row_b": pair[1][1], "overlap_kb": round(ov / 1000, 1),
                                         "pct_of_shorter": round(100 * ov / shorter, 1)})
                    overlap_classes.setdefault(i, set()).add(data.at[j, TARGET])
                    overlap_classes.setdefault(j, set()).add(data.at[i, TARGET])
overlaps = save_table(pd.DataFrame(overlap_rows), "1.3", "overlapping_bgcs")

if overlaps.empty:
    print("No pair of BGCs from the same genome overlaps.")
else:
    overlaps["pair"] = overlaps["class_a"] + " + " + overlaps["class_b"]
    overlap_summary = (overlaps.groupby("pair").agg(pairs=("overlap_kb", "size"),
                                                         pct_of_shorter_median=("pct_of_shorter", "median"))
                       .sort_values("pairs", ascending=False).reset_index())
    print(f"{len(overlaps)} pairs of BGCs from the same genome overlap ({len(overlap_classes)} BGCs involved)")
    display(overlap_summary)

## 1.4 Pfam domain matrix

`X` holds the score of each domain in each BGC (0 = absent) and `y` the class. Domains that never vary are removed because they carry no information. The figure shows how many distinct Pfam domains each BGC has, by class.

**How to read it:** each dot is a BGC; the box spans the central 50 % and the thick line marks the median.

In [ ]:
X_frame = data[pfam_columns].astype(float).fillna(0.0)
n_before = X_frame.shape[1]
if CONFIG["drop_zero_variance"]:
    X_frame = X_frame.loc[:, X_frame.std(axis=0) > 0]           # constant domains cannot discriminate
feature_names = X_frame.columns.tolist()
domain_names = [f.split(":", 1)[-1] for f in feature_names]       # "PF00005.34:ABC_tran" → "ABC_tran"
X, y = X_frame.to_numpy(), data[TARGET].to_numpy()
P = X > 0                                                         # presence / absence of each domain
n_domains = P.sum(axis=1)
n_profiles = len({row.tobytes() for row in P})

print(f"Matrix X: {X.shape[0]} BGCs × {X.shape[1]} domains ({n_before - X.shape[1]} without variation, removed)")
print(f"{100 * P.mean():.1f} % of the cells contain a domain · non-zero values between "
      f"{X[P].min():.2f} and {X[P].max():.2f}")
print(f"Distinct presence/absence profiles: {n_profiles} out of {len(y)} BGCs")

dom_table = (pd.DataFrame({"cls": y, "n": n_domains}).groupby("cls")["n"].agg(["size", "min", "median", "max"])
             .reindex(class_order).reset_index())
dom_table.columns = ["class", "n_BGC", "minimum", "median", "maximum"]
save_table(dom_table, "1.4", "domains_per_bgc")

fig, ax = plt.subplots(figsize=(8.5, 0.5 * len(class_order) + 1.8))
hbox(ax, [n_domains[y == c] for c in class_order], class_order)
ax.set_xlabel("Distinct Pfam domains in the BGC")
titles(ax, "Pfam domains per BGC, by class",
       "Each dot is a BGC · box = central 50 % · thick line = median")
show_and_save(fig, "1.4", "domains_per_bgc")
dom_table

## 1.5 t-SNE map of the BGCs

t-SNE places BGCs with similar domain profiles close together. It shows whether the classes form their own groups **before** any model is trained. Three figures are produced:

- **1.5a · Overview:** all classes at once. Each ellipse encloses a compact group of BGCs of the same class (95 %); a class can form several groups.
- **1.5b · Class by class:** the same map once per class, highlighting that class (the rest in grey). It is the clearest way to see where each class falls and whether it overlaps another.
- **1.5c · Perplexity sensitivity:** the same analysis with three values of t-SNE's main parameter. If the groups persist, they are not an artefact of the parameter.

**Caution:** in t-SNE the distances *between* groups and their sizes have no meaning; only which BGCs end up together matters.

In [ ]:
perp_main = CONFIG["tsne_perplexity_main"]
X_scaled = StandardScaler().fit_transform(X)                     # mean 0 and variance 1 per domain
tsne_runs = {p: run_tsne(X_scaled, p) for p in sorted(set(CONFIG["tsne_perplexities"]) | {perp_main})}
tsne_all = tsne_runs[perp_main]
counts = pd.Series(y).value_counts()
save_table(pd.DataFrame({"class": y, "tSNE_1": tsne_all[:, 0].round(3), "tSNE_2": tsne_all[:, 1].round(3)}),
           "1.5", "tsne_coordinates")

# ---- 1.5a Overview -----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 7.4))
cat_scatter(ax, tsne_all, y, class_order, CLASS_STYLE)
tsne_axes(ax)
cat_legend(ax, class_order, CLASS_STYLE, counts, title="Class (n BGCs)")
titles(ax, "t-SNE map of the BGCs, coloured by class",
       f"{len(y)} BGCs · {X.shape[1]} Pfam domains · perplexity {perp_main}\n"
       "Each ellipse encloses a compact group of BGCs of the same class (95 %)")
show_and_save(fig, "1.5", "a_tsne_by_class")

# ---- 1.5b Class by class (the rest in grey) --------------------------------
ncol = 3
nrow = int(np.ceil(len(class_order) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.3 * ncol, 3.8 * nrow), squeeze=False)
for ax, c in zip(axes.ravel(), class_order):
    highlight_panel(ax, tsne_all, y, c, CLASS_STYLE)
for ax in axes.ravel()[len(class_order):]:
    ax.axis("off")
fig.suptitle("Where each class falls on the t-SNE map (grey: all other BGCs)",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
show_and_save(fig, "1.5", "b_tsne_class_by_class")
for c in class_order:                                            # individual panels
    fig, ax = plt.subplots(figsize=(6.5, 5.4))
    highlight_panel(ax, tsne_all, y, c, CLASS_STYLE, s_fg=26)
    show_and_save(fig, "1.5", f"b_tsne_class_{slug(c)}", individual=True)

# ---- 1.5c Perplexity sensitivity ------------------------------------------------
perps = sorted(CONFIG["tsne_perplexities"])
fig, axes = plt.subplots(1, len(perps), figsize=(5.3 * len(perps), 5.0))
for ax, p in zip(np.atleast_1d(axes), perps):
    cat_scatter(ax, tsne_runs[p], y, class_order, CLASS_STYLE, direct_labels=False, s=14)
    tsne_axes(ax)
    ax.set_title(f"Perplexity {p}", loc="left", fontsize=11)
fig.suptitle("The same t-SNE map with three perplexities", x=0.01, ha="left",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig_legend(fig, class_order, CLASS_STYLE, counts)
show_and_save(fig, "1.5", "c_tsne_perplexities")

## 1.6 The same t-SNE in its four orientations

t-SNE axes have neither sign nor units: flipping an axis gives an equivalent map. The four orientations (±t-SNE 1 × ±t-SNE 2) are shown so the most readable one, or the one that matches another figure, can be chosen. The data are exactly the same in all four. Each view is also saved as an individual figure.

In [ ]:
views = [(+1, +1, "t-SNE 1 vs t-SNE 2", "1"), (+1, -1, "t-SNE 1 vs −t-SNE 2", "2"),
         (-1, +1, "−t-SNE 1 vs t-SNE 2", "3"), (-1, -1, "−t-SNE 1 vs −t-SNE 2", "4")]

fig, axes = plt.subplots(2, 2, figsize=(12.5, 10.6))
for ax, (sx, sy, name, k) in zip(axes.ravel(), views):
    cat_scatter(ax, tsne_all, y, class_order, CLASS_STYLE, flip=(sx, sy), s=16, label_size=8.5)
    tsne_axes(ax, sx, sy)
    ax.set_title(f"View {k}: {name}", loc="left", fontsize=11)
fig.suptitle("t-SNE map in its four orientations (same data)", x=0.01,
             ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
fig_legend(fig, class_order, CLASS_STYLE, counts)
show_and_save(fig, "1.6", "tsne_four_views")

for sx, sy, name, k in views:                                    # one figure per view
    fig, ax = plt.subplots(figsize=(10, 7.4))
    cat_scatter(ax, tsne_all, y, class_order, CLASS_STYLE, flip=(sx, sy))
    tsne_axes(ax, sx, sy)
    cat_legend(ax, class_order, CLASS_STYLE, counts, title="Class (n BGCs)")
    titles(ax, f"t-SNE map, view {k}: {name}")
    show_and_save(fig, "1.6", f"tsne_view{k}", individual=True)

## 1.7 3D t-SNE

t-SNE with three output dimensions, seen from three angles. The third dimension is not a "source of variance": the algorithm spreads the neighbourhood structure across the three axes. It helps check whether two classes that overlap in 2D separate in 3D.

In [ ]:
tsne_3d = run_tsne(X_scaled, perp_main, n_components=3)
angles = [(25, 45), (25, 135), (60, -60)]

fig = plt.figure(figsize=(18, 6.2))
for i, (elev, azim) in enumerate(angles, start=1):
    ax = fig.add_subplot(1, 3, i, projection="3d")
    for c in class_order:
        m, st = y == c, CLASS_STYLE[c]
        ax.scatter(*tsne_3d[m].T, s=16 * (1.5 if st["hollow"] else 1), marker=st["marker"], depthshade=False,
                   facecolors="none" if st["hollow"] else st["color"],
                   edgecolors=st["color"] if st["hollow"] else SURFACE, linewidths=1.1 if st["hollow"] else 0.4)
    ax.view_init(elev=elev, azim=azim)
    ax.set_box_aspect(None, zoom=0.88)
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    ax.set_xlabel("t-SNE 1", labelpad=-8); ax.set_ylabel("t-SNE 2", labelpad=-8); ax.set_zlabel("t-SNE 3", labelpad=-8)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color(to_rgba(SURFACE, 1.0))
    ax.set_title(f"Angle {i}: elevation {elev}°, azimuth {azim}°", loc="left", fontsize=11)
fig.suptitle("3D t-SNE map, coloured by class", x=0.01, ha="left", fontsize=13,
             fontweight="bold")
fig_legend(fig, class_order, CLASS_STYLE, counts, y=0.04)
show_and_save(fig, "1.7", "tsne_3d")

## 1.8 Correlation circle — before the model (25 % of the data)

It shows **which domains organise the map**. For each Pfam domain its correlation with axis 1 and axis 2 of a t-SNE is computed and drawn as an arrow:

- a **stratified 25 % subsample** of the BGCs is used (the same one is reused in section 3.5, after the model, for comparison);
- "before the model" = with **all** domains; the `corr_circle_top_features` most correlated domains are drawn;
- a long arrow, close to the outer circle, means the domain changes a lot along that direction of the map: the BGCs the arrow points to carry that domain. The dotted circle marks |r| = 0.5.

**Caution:** t-SNE is non-linear, so these correlations are descriptive: they say which domains vary along each axis, they are not the loadings of a projection.

In [ ]:
_, corr_idx = train_test_split(np.arange(len(y)), test_size=CONFIG["corr_circle_fraction"], stratify=y, random_state=RS)
corr_idx = np.sort(corr_idx)                                    # same subsample before and after the model


def correlation_circle(X_sub, y_sub, names, section, name, title):
    """t-SNE of the subsample + correlation of each domain with both map axes."""
    Xs = StandardScaler().fit_transform(X_sub)
    emb = run_tsne(Xs, perp_main)
    E = (emb - emb.mean(axis=0)) / emb.std(axis=0)
    corr = Xs.T @ E / len(y_sub)                                # Pearson r (both variables standardised)
    norm = np.linalg.norm(corr, axis=1)
    top = np.argsort(norm)[::-1][:CONFIG["corr_circle_top_features"]]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7), gridspec_kw={"width_ratios": [1.1, 1]})
    cat_scatter(ax1, emb, y_sub, class_order, CLASS_STYLE, s=24, label_size=8.5)
    tsne_axes(ax1)
    ax1.set_title(f"Subsample BGCs (n = {len(y_sub)})", loc="left", fontsize=11)

    # Domains with the same correlation (found together in the same BGCs) share one arrow
    bundles = []
    for i in top:
        for b in bundles:
            if np.linalg.norm(corr[i] - corr[b[0]]) < 0.03:
                b.append(i)
                break
        else:
            bundles.append([i])
    labels = ["\n".join(names[i].split(":", 1)[-1] for i in b[:3]) + (f"\n(+{len(b) - 3})" if len(b) > 3 else "") for b in bundles]
    sizes = [(0.03 * max(len(t) for t in lab.split("\n")) + 0.06, 0.085 * (lab.count("\n") + 1)) for lab in labels]

    theta = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(theta), np.sin(theta), color=AXIS, lw=1)
    ax2.plot(0.5 * np.cos(theta), 0.5 * np.sin(theta), color=AXIS, lw=0.8, ls=":")
    ax2.axhline(0, color=GRID, lw=0.8)
    ax2.axvline(0, color=GRID, lw=0.8)
    heads = np.array([corr[b[0]] for b in bundles])
    label_pos = repel(heads * 1.18, sizes)
    for (hx, hy), (lx, ly), lab in zip(heads, label_pos, labels):
        ax2.annotate("", xy=(hx, hy), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1.1))
        ax2.plot([hx, lx], [hy, ly], color=CONTEXT, lw=0.6)
        ax2.text(lx, ly, lab, fontsize=8.5, ha="center", va="center", color=INK, path_effects=HALO, linespacing=1.1)
    ax2.set_xlim(-1.5, 1.5)
    ax2.set_ylim(-1.5, 1.5)
    ax2.set_aspect("equal")
    ax2.grid(False)
    ax2.set_xlabel("correlation with t-SNE 1")
    ax2.set_ylabel("correlation with t-SNE 2")
    ax2.set_title(f"The {len(top)} domains most correlated with the map\n"
                  "(domains found together in the same BGCs share an arrow)",
                  loc="left", fontsize=11)
    fig.suptitle(title, x=0.01, ha="left", fontsize=13, fontweight="bold")
    fig.tight_layout()
    show_and_save(fig, section, name)
    return pd.DataFrame({"domain": [names[i] for i in top], "r_tSNE_1": corr[top, 0].round(3),
                         "r_tSNE_2": corr[top, 1].round(3), "length": norm[top].round(3)})


corr_before = save_table(
    correlation_circle(X[corr_idx], y[corr_idx], feature_names, "1.8", "correlation_circle_before",
                       f"Correlation circle before the model: all domains ({X.shape[1]})"),
    "1.8", "correlation_circle_before")
corr_before

---
# Part 2 · Model training

**Question:** how many domains does the classifier need, and is it overfitting?

| Section | What it does | Main outputs |
|---|---|---|
| 2.1 | Sets 30 % of the BGCs aside as the test set (untouched until Part 3) | `2.1_partition_balance` |
| 2.2 | Ranks the domains with RFE and chooses how many to use | `2.2_feature_selection_curve` |
| 2.3 | Chooses the SVM regularisation `C` | `2.3_validation_curve_C` |
| 2.4 | Learning curve | `2.4_learning_curve` |
| 2.5 | Permutation test | `2.5_permutation_test` |
| 2.6 | Fits the final model | `2.6_confusion_matrix_training` |
| 2.7 | Ranking of the model's domains (loadings) | `2.7_domain_ranking` |

**Golden rule:** everything in this part uses only the 70 % training set. The held-out test set is opened once, in Part 3.

## 2.1 Stratified 70 / 30 split

30 % of the BGCs are set aside as the **test set** and used for no decision at all. The split is stratified: every class keeps its proportion in training and test (the table checks it).

In [ ]:
idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=CONFIG["test_size"], stratify=y, random_state=RS)
X_train, y_train = X[idx_train], y[idx_train]
SEALED_TEST = {"idx": idx_test}           # opened only in Part 3

balance = pd.DataFrame({"class": class_order,
                        "total": [int((y == c).sum()) for c in class_order],
                        "training": [int((y_train == c).sum()) for c in class_order],
                        "test": [int((y[idx_test] == c).sum()) for c in class_order]})
for col in ("total", "training", "test"):
    balance[f"% {col}"] = (100 * balance[col] / balance[col].sum()).round(1)
pct = balance[[c for c in balance.columns if c.startswith("%")]]
max_dev = float((pct.max(axis=1) - pct.min(axis=1)).max())
save_table(balance, "2.1", "partition_balance")
print(f"Training: {len(idx_train)} BGCs · held-out test: {len(idx_test)} BGCs · "
      f"largest proportion difference between partitions: {max_dev:.1f} percentage points")
balance

## 2.2 How many domains does the model need? (RFE)

1. The training set is split again (80 / 20) into **inner training** and **validation**.
2. **RFE** (recursive feature elimination) with a linear SVM ranks the domains from most to least useful, using only the inner training set.
3. For each number of domains *k*, accuracy is measured by cross-validation (inner training) and on the independent validation set.
4. **One-standard-error rule:** the smallest *k* whose validation accuracy is within 1 standard error of the maximum is chosen. The model thus uses few domains without losing accuracy.

**How to read the figure:** the blue line is optimistic because RFE already saw those BGCs; the decision is taken with the orange line (independent validation). The coloured band = ± 1 standard error.

In [ ]:
X_inner, X_valid, y_inner, y_valid = train_test_split(X_train, y_train, test_size=CONFIG["validation_size"],
                                                      stratify=y_train, random_state=RS)

# ---- RFE ranking (inner training only) ---------------------------------------
rfe = RFE(SVC(kernel="linear", C=CONFIG["svm_c"], random_state=RS), n_features_to_select=1, step=CONFIG["rfe_step"])
rfe.fit(StandardScaler().fit_transform(X_inner), y_inner)         # down to 1 domain → full ranking
ranked_idx = np.argsort(rfe.ranking_, kind="stable")             # most to least useful
save_table(pd.DataFrame({"rank_rfe": np.arange(1, len(ranked_idx) + 1),
                         "domain": [feature_names[i] for i in ranked_idx]}), "2.2", "rfe_ranking")

# ---- Accuracy versus number of domains ------------------------------------------
rows = []
for k in [k for k in CONFIG["feature_grid"] if k <= X_inner.shape[1]]:
    cols = ranked_idx[:k]
    cv_mean, cv_se = cv_summary(cross_val_score(make_estimator(), X_inner[:, cols], y_inner, cv=make_cv(),
                                                scoring="accuracy", n_jobs=CONFIG["n_jobs"]))
    acc = accuracy_score(y_valid, make_estimator().fit(X_inner[:, cols], y_inner).predict(X_valid[:, cols]))
    lo, hi = wilson_interval(round(acc * len(y_valid)), len(y_valid), z=1.0)
    rows.append({"k": k, "cv_accuracy": cv_mean, "cv_se": cv_se, "validation_accuracy": acc,
                 "validation_low_1se": lo, "validation_high_1se": hi})
curve = pd.DataFrame(rows)
save_table(curve.round(4), "2.2", "feature_selection_curve")

# ---- One-standard-error rule -----------------------------------------------------------
best = curve.loc[curve["validation_accuracy"].idxmax()]
se_best = np.sqrt(best["validation_accuracy"] * (1 - best["validation_accuracy"]) / len(y_valid))
threshold = best["validation_accuracy"] - se_best
k_optimal = int(curve.loc[curve["validation_accuracy"] >= threshold, "k"].min())
sel_idx = ranked_idx[:k_optimal]
selected_features = [feature_names[i] for i in sel_idx]

fig, ax = plt.subplots(figsize=(9.5, 5.6))
ax.fill_between(curve["k"], curve["cv_accuracy"] - curve["cv_se"], curve["cv_accuracy"] + curve["cv_se"],
                color=SERIES[0], alpha=0.12, lw=0)
ax.plot(curve["k"], curve["cv_accuracy"], color=SERIES[0], marker="o", ms=4,
        label="Cross-validation on inner training (optimistic)")
ax.fill_between(curve["k"], curve["validation_low_1se"], curve["validation_high_1se"], color=SERIES[1], alpha=0.12, lw=0)
ax.plot(curve["k"], curve["validation_accuracy"], color=SERIES[1], marker="o", ms=4,
        label="Independent validation (decides k)")
ax.axhline(threshold, color=MUTED, lw=1)
ax.text(curve["k"].min(), threshold + 0.005, "validation maximum − 1 standard error", color=MUTED, fontsize=9,
        ha="left", va="bottom")
ax.axvline(k_optimal, color=INK2, lw=1)
ax.annotate(f"k = {k_optimal} domains", xy=(k_optimal, threshold), xytext=(8, -32), textcoords="offset points",
            fontsize=10.5, fontweight="bold", color=INK, path_effects=HALO)
ax.set_xscale("log")
ticks = [t for t in (1, 2, 5, 10, 20, 50, 100, 200) if t <= curve["k"].max()]
ax.set_xticks(ticks)
ax.set_xticklabels([str(t) for t in ticks])
ax.set_xlabel("Number of Pfam domains used (log scale)")
ax.set_ylabel("Accuracy")
ax.legend(loc="lower right")
titles(ax, "How many domains does the model need?",
       f"The smallest number of domains within 1 standard error of the validation maximum is chosen → k = {k_optimal}")
show_and_save(fig, "2.2", "feature_selection_curve")
print(f"Validation maximum: {best['validation_accuracy']:.3f} at k = {int(best['k'])} · threshold = {threshold:.3f} "
      f"→ using k = {k_optimal} domains")

## 2.3 Overfitting control: regularisation `C`

`C` controls how closely the SVM fits the training data: a large `C` gives a narrow margin and a risk of overfitting. With the *k* chosen domains, training and cross-validated accuracy are compared for several `C` values, and the **smallest** (most regularised) `C` whose cross-validated accuracy is within 1 standard error of the maximum is chosen.

**How to read the figure:** if the training line rises while the cross-validation line does not, the model memorises instead of generalising.

In [ ]:
Xtr_sel = X_train[:, sel_idx]
rows = []
for C in CONFIG["c_grid"]:
    m, se = cv_summary(cross_val_score(make_estimator(C), Xtr_sel, y_train, cv=make_cv(), scoring="accuracy",
                                       n_jobs=CONFIG["n_jobs"]))
    train_acc = make_estimator(C).fit(Xtr_sel, y_train).score(Xtr_sel, y_train)     # resubstitution
    rows.append({"C": C, "train_accuracy": train_acc, "cv_accuracy": m, "cv_se": se})
c_curve = pd.DataFrame(rows)
save_table(c_curve.round(4), "2.3", "validation_curve_C")

best_c = c_curve.loc[c_curve["cv_accuracy"].idxmax()]
eligible = c_curve.loc[c_curve["cv_accuracy"] >= best_c["cv_accuracy"] - best_c["cv_se"], "C"]
C_final = float(eligible.min()) if CONFIG["tune_c"] else CONFIG["svm_c"]

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.plot(c_curve["C"], c_curve["train_accuracy"], color=SERIES[0], marker="o", ms=4, label="Training (resubstitution)")
ax.fill_between(c_curve["C"], c_curve["cv_accuracy"] - c_curve["cv_se"], c_curve["cv_accuracy"] + c_curve["cv_se"],
                color=SERIES[1], alpha=0.12, lw=0)
ax.plot(c_curve["C"], c_curve["cv_accuracy"], color=SERIES[1], marker="o", ms=4, label="Cross-validation")
ax.axvline(C_final, color=INK2, lw=1)
ax.annotate(f"chosen C = {C_final:g}", xy=(C_final, ax.get_ylim()[0]), xytext=(6, 12), textcoords="offset points",
            fontsize=10.5, fontweight="bold", color=INK, path_effects=HALO)
ax.set_xscale("log")
ax.set_xlabel("C (inverse regularisation, log scale)")
ax.set_ylabel("Accuracy")
ax.legend(loc="lower right")
titles(ax, f"SVM regularisation with k = {k_optimal} domains",
       "The smallest C within 1 standard error of the cross-validation maximum is chosen")
show_and_save(fig, "2.3", "validation_curve_C")
print(f"C with the highest cross-validated accuracy: {best_c['C']:g} ({best_c['cv_accuracy']:.3f}) · chosen C: {C_final:g}")

## 2.4 Overfitting control: learning curve

Training and cross-validated accuracy (5 folds) as more BGCs are used for training.

**How to read it:** if both curves end together and high, the model generalises; if the training curve stays far above the validation curve, it memorises.

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
sizes, tr_scores, va_scores = learning_curve(make_estimator(C_final), Xtr_sel, y_train, cv=cv5, scoring="accuracy",
                                             train_sizes=CONFIG["learning_curve_sizes"], n_jobs=CONFIG["n_jobs"],
                                             shuffle=True, random_state=RS)
lc = pd.DataFrame({"n_train": sizes, "train_mean": tr_scores.mean(axis=1), "train_sd": tr_scores.std(axis=1),
                   "cv_mean": va_scores.mean(axis=1), "cv_sd": va_scores.std(axis=1)})
save_table(lc.round(4), "2.4", "learning_curve")

fig, ax = plt.subplots(figsize=(9, 5.2))
for col, sd, color, label in (("train_mean", "train_sd", SERIES[0], "Training"),
                              ("cv_mean", "cv_sd", SERIES[1], "Cross-validation (5 folds)")):
    ax.fill_between(lc["n_train"], lc[col] - lc[sd], lc[col] + lc[sd], color=color, alpha=0.12, lw=0)
    ax.plot(lc["n_train"], lc[col], color=color, marker="o", ms=4, label=label)
ax.set_xlabel("BGCs used for training")
ax.set_ylabel("Accuracy")
ax.legend(loc="lower right")
gap_lc = lc["train_mean"].iloc[-1] - lc["cv_mean"].iloc[-1]
titles(ax, "Learning curve",
       f"Final training − validation gap = {gap_lc:+.3f} · band = ± 1 SD across folds")
show_and_save(fig, "2.4", "learning_curve")

## 2.5 Overfitting control: permutation test

The classes are shuffled at random and the model is retrained `n_permutations` times. If the real model does not clearly beat the models trained on shuffled classes, its accuracy would not be real.

**How to read it:** the grey bars are the accuracy with random classes; the blue line is the real model. The *p* value is the share of permutations that match or beat the real model (the smallest possible value is 1 / (permutations + 1)).

In [ ]:
score_real, perm_scores, p_perm = permutation_test_score(
    make_estimator(C_final), Xtr_sel, y_train, cv=cv5, scoring="accuracy",
    n_permutations=CONFIG["n_permutations"], n_jobs=CONFIG["n_jobs"], random_state=RS)
save_table(pd.DataFrame({"accuracy_with_permuted_classes": perm_scores.round(4)}),
           "2.5", "permutation_test")

fig, ax = plt.subplots(figsize=(9, 5.0))
ax.hist(perm_scores, bins=np.arange(0, 1.0001, 0.01), color=AXIS, edgecolor=SURFACE, linewidth=0.6,
        label="Randomly permuted classes")             # 0.01-wide bars
ax.axvline(score_real, color=SERIES[0], lw=2.5, label=f"Real model = {score_real:.3f}")
ax.axvline(1 / len(class_order), color=MUTED, lw=1, label=f"Chance (1 / number of classes) = {1 / len(class_order):.3f}")
ax.set_xlim(0, 1.02)
ax.set_xlabel("Cross-validated accuracy")
ax.set_ylabel("Permutations")
ax.legend(loc="upper center")
titles(ax, "Permutation test",
       f"{CONFIG['n_permutations']} permutations · p = {p_perm:.4f}")
show_and_save(fig, "2.5", "permutation_test")

## 2.6 Final model

The model is refitted on the whole training set (70 %) with the *k* chosen domains and the chosen `C`. `probability=True` lets the exported model output a probability for each class.

The training confusion matrix is nearly perfect by construction (the model has already seen those BGCs); the honest accuracy is the cross-validated one printed below and, above all, the held-out test accuracy (Part 3).

**How to read the matrix:** rows = true class, columns = predicted class. The number is the count of BGCs and the colour the percentage of the true class; the diagonal holds the correct predictions.

In [ ]:
def confusion_figure(y_true, y_pred, section, name, title):
    """Confusion matrix: number = BGCs; colour = % of the true class (row)."""
    cm = confusion_matrix(y_true, y_pred, labels=class_order)
    share = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(8.6, 7.2))
    heatmap(ax, share, class_order, class_order, vmin=0, vmax=1, annot=np.where(cm > 0, cm.astype(str), ""),
            cbar_label="Fraction of the true class", annot_size=10)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    rotate_xlabels(ax)
    ok = int(np.trace(cm))
    titles(ax, title, f"{ok} of {cm.sum()} BGCs on the diagonal (correct) · empty cells = 0")
    show_and_save(fig, section, name)
    return pd.DataFrame(cm, index=class_order, columns=class_order).rename_axis("true_class").reset_index()


final_model = make_estimator(C_final, probability=True).fit(Xtr_sel, y_train)
y_train_pred = final_model.predict(Xtr_sel)
metrics_train, per_class_train = compute_metrics(y_train, y_train_pred, class_order, "training (resubstitution)")
cv_final, cv_final_se = cv_summary(cross_val_score(make_estimator(C_final), Xtr_sel, y_train, cv=make_cv(),
                                                   scoring="accuracy", n_jobs=CONFIG["n_jobs"]))
metrics_train["cv_accuracy"], metrics_train["cv_se"] = round(cv_final, 4), round(cv_final_se, 4)

save_table(confusion_figure(y_train, y_train_pred, "2.6", "confusion_matrix_training",
                            "Confusion matrix — training"),
           "2.6", "confusion_matrix_training")
print(f"Final model: {len(y_train)} BGCs · {k_optimal} domains · C = {C_final:g} · "
      f"{int(final_model.named_steps['svm'].n_support_.sum())} support vectors")
print(f"Training accuracy (resubstitution): {metrics_train.at[0, 'accuracy']:.3f} · "
      f"cross-validated: {cv_final:.3f} ± {cv_final_se:.3f} (standard error)")

## 2.7 Ranking of the model's domains (loadings)

For each of the model's *k* domains:

| Column | What it measures |
|---|---|
| `rank_svm` / `svm_weight` | Weight of the domain in the final SVM: norm of its coefficients over all one-vs-one classifiers. This is the model's loading: how much the domain shapes the decision boundaries |
| `rank_rfe` | Position in the RFE ranking (1 = the last to be eliminated) |
| `train_prevalence` | Fraction of training BGCs that carry the domain |
| `dominant_class` | Class with the highest mean score for the domain: aids biological reading |

**How to read the figure:** longer bars = domains that weigh most in the model's decision; the text at the end of each bar says in which class the domain is abundant.

In [ ]:
W = final_model.named_steps["svm"].coef_                     # (one-vs-one pairs × k), standardised space
class_means = pd.DataFrame(Xtr_sel, columns=selected_features).groupby(y_train).mean()
loadings = pd.DataFrame({
    "domain": [f.split(":", 1)[-1] for f in selected_features],
    "pfam_column": selected_features,
    "rank_rfe": np.arange(1, k_optimal + 1),
    "svm_weight": np.sqrt((W ** 2).sum(axis=0)).round(4),
    "train_prevalence": (Xtr_sel > 0).mean(axis=0).round(3),
    "dominant_class": class_means.idxmax(axis=0).to_numpy()})
loadings = loadings.sort_values("svm_weight", ascending=False).reset_index(drop=True)
loadings.insert(0, "rank_svm", np.arange(1, len(loadings) + 1))
save_table(loadings, "2.7", "domain_ranking")

top = loadings.head(20)
fig, ax = plt.subplots(figsize=(9.5, 7.6))
ax.barh(np.arange(len(top)), top["svm_weight"], height=0.62, color=SERIES[0])
xmax = top["svm_weight"].max()
for i, (v, c) in enumerate(zip(top["svm_weight"], top["dominant_class"])):
    ax.text(v + xmax * 0.015, i, c, va="center", fontsize=9, color=INK2)
ax.set_yticks(np.arange(len(top)))
ax.set_yticklabels(top["domain"])
ax.set_ylim(len(top) - 0.4, -0.6)
ax.set_xlim(0, xmax * 1.32)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Domain weight in the SVM (norm of its coefficients)")
titles(ax, f"The 20 domains with the largest model weight (out of {k_optimal})",
       "Text at the end of the bar = class in which the domain is most abundant")
show_and_save(fig, "2.7", "domain_ranking")
loadings.head(20)

---
# Part 3 · Analysis of the model results

**Question:** how well does the model classify BGCs it never saw, and where does it go wrong?

| Section | What it does | Main outputs |
|---|---|---|
| 3.1 | Opens the held-out test set and computes the metrics | `3.1_metrics` |
| 3.2 | Test confusion matrix | `3.2_confusion_matrix_test` |
| 3.3 | Error per class | `3.3_error_per_class` |
| 3.4 | t-SNE map with the model's domains and the errors marked | `3.4a/b` |
| 3.5 | Correlation circle after the model (same 25 % as in 1.8) | `3.5_correlation_circle_after` |

## 3.1 Metrics on the held-out test set

First and only opening of the test set. The model, the domains and `C` are already fixed: nothing computed here changes them.

- **Accuracy** with its 95 % confidence interval (Wilson); **balanced accuracy** and **macro F1**, which weigh every class equally however small; **Cohen's kappa** and **MCC**, which correct for chance agreement.
- **Generalisation gap** = training accuracy − test accuracy. Together with the permutation test (2.5) it gives the overfitting verdict.
- **BGCs with a "twin" in training:** test BGCs whose domain profile (presence/absence) is identical to some training BGC. Accuracy is reported for those and for the rest, because the former are easier to get right.

In [ ]:
idx_test = SEALED_TEST["idx"]
X_test, y_test = X[idx_test], y[idx_test]
y_test_pred = final_model.predict(X_test[:, sel_idx])

metrics_test, per_class_test = compute_metrics(y_test, y_test_pred, class_order, "held-out test")
n_correct = int((y_test == y_test_pred).sum())
ci_low, ci_high = wilson_interval(n_correct, len(y_test))
metrics_test["ci95_low"], metrics_test["ci95_high"] = round(ci_low, 4), round(ci_high, 4)
gap = float(metrics_train.at[0, "accuracy"] - metrics_test.at[0, "accuracy"])

# How many test BGCs have an identical profile in training?
train_profiles = {P[i].tobytes() for i in idx_train}
has_twin = np.array([P[i].tobytes() in train_profiles for i in idx_test])
acc_twin = accuracy_score(y_test[has_twin], y_test_pred[has_twin]) if has_twin.any() else np.nan
acc_new = accuracy_score(y_test[~has_twin], y_test_pred[~has_twin]) if (~has_twin).any() else np.nan

metrics = pd.concat([metrics_train, metrics_test], ignore_index=True)
save_table(metrics, "3.1", "metrics")
save_table(pd.concat([per_class_train, per_class_test], ignore_index=True), "3.1", "per_class_metrics")

no_overfit = gap <= CONFIG["max_generalization_gap"] and p_perm < 0.05
print(f"Test accuracy: {metrics_test.at[0, 'accuracy']:.3f} ({n_correct}/{len(y_test)} BGCs) · "
      f"95 % CI: {ci_low:.3f}–{ci_high:.3f}")
print(f"Balanced accuracy: {metrics_test.at[0, 'balanced_accuracy']:.3f} · F1 macro: {metrics_test.at[0, 'f1_macro']:.3f} · "
      f"kappa: {metrics_test.at[0, 'cohen_kappa']:.3f} · MCC: {metrics_test.at[0, 'mcc']:.3f}")
print(f"Training − test gap: {gap:+.3f} (tolerable ≤ {CONFIG['max_generalization_gap']}) · "
      f"permutation p = {p_perm:.4f} → "
      + ("no evidence of overfitting" if no_overfit else "check: possible overfitting"))
print(f"Test BGCs with an identical profile in training: {has_twin.sum()} of {len(y_test)} "
      f"({100 * has_twin.mean():.0f} %) → accuracy {acc_twin:.3f} · rest (n = {(~has_twin).sum()}) → accuracy {acc_new:.3f}")
metrics[["partition", "n", "accuracy", "balanced_accuracy", "f1_macro", "cohen_kappa", "mcc", "ci95_low", "ci95_high"]]

## 3.2 Test confusion matrix

**How to read it:** rows = true class, columns = predicted class. The number is the count of BGCs and the colour the percentage of the true class. Everything off the diagonal is an error, and its position tells which class it was confused with.

In [ ]:
cm_test = save_table(confusion_figure(y_test, y_test_pred, "3.2", "confusion_matrix_test",
                                      "Confusion matrix — held-out test"),
                     "3.2", "confusion_matrix_test")

## 3.3 Error per class

Percentage of each class's test BGCs the model got wrong. Small classes have few test BGCs, so a single error moves their percentage a lot; that is why the number of errors is also shown.

In [ ]:
err = per_class_test.sort_values(["error_rate", "n"], ascending=[False, False]).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8.8, 0.5 * len(err) + 1.8))
ax.barh(np.arange(len(err)), 100 * err["error_rate"], height=0.62, color=SERIES[0])
xmax = max(100 * err["error_rate"].max(), 5)
for i, (e, n, r) in enumerate(zip(err["errors"], err["n"], err["error_rate"])):
    ax.text(100 * r + xmax * 0.02, i, f"{e} of {n}", va="center", fontsize=9.5, color=INK2)
ax.set_yticks(np.arange(len(err)))
ax.set_yticklabels(err["class"])
ax.set_ylim(len(err) - 0.4, -0.6)
ax.set_xlim(0, xmax * 1.25)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Misclassified test BGCs (%)")
titles(ax, "Error per class on the held-out test set",
       "Text = errors out of that class's test BGCs")
show_and_save(fig, "3.3", "error_per_class")

## 3.4 Where does the model go wrong? t-SNE with the model's domains

t-SNE cannot project new BGCs onto an existing map, so a map of **all** BGCs is computed using only the model's *k* domains, and the test BGCs are then marked. It is descriptive: the model is already fixed.

- **3.4a:** the map coloured by class. If the classes stay as separate as in 1.5a, the *k* domains keep the information that tells the classes apart.
- **3.4b:** only the test BGCs, with their true class; training BGCs are shown in grey for context. **Errors** have a black ring and the label `true → predicted`. If errors fall where two classes meet, the model fails where the data are ambiguous.
- **Error table:** for each error, the probability the model gave its prediction, whether its domain profile is identical to some training BGC, and which classes it overlaps in its genome (section 1.3).

In [ ]:
X_sig_scaled = StandardScaler().fit_transform(X[:, sel_idx])
tsne_sig = run_tsne(X_sig_scaled, perp_main)
save_table(pd.DataFrame({"class": y, "partition": np.where(np.isin(np.arange(len(y)), idx_test), "test", "training"),
                         "tSNE_1": tsne_sig[:, 0].round(3), "tSNE_2": tsne_sig[:, 1].round(3)}),
           "3.4", "signature_tsne_coordinates")

# ---- 3.4a Map with the model's k domains --------------------------------------------
fig, ax = plt.subplots(figsize=(10, 7.4))
cat_scatter(ax, tsne_sig, y, class_order, CLASS_STYLE)
tsne_axes(ax)
cat_legend(ax, class_order, CLASS_STYLE, counts, title="Class (n BGCs)")
titles(ax, f"t-SNE map with the {k_optimal} model domains",
       f"All BGCs (n = {len(y)}) · compare with 1.5a, which uses all {X.shape[1]} domains")
show_and_save(fig, "3.4", "a_tsne_model_domains")

# ---- 3.4b Test errors -----------------------------------------------------------------------
T_test = tsne_sig[idx_test]
wrong = y_test != y_test_pred
fig, ax = plt.subplots(figsize=(10, 7.4))
ax.scatter(*tsne_sig[idx_train].T, s=9, c=CONTEXT, linewidths=0, zorder=1)
cat_scatter(ax, T_test, y_test, class_order, CLASS_STYLE, ellipses=False, direct_labels=False, s=30)
ax.scatter(*T_test[wrong].T, s=230, facecolors="none", edgecolors=INK, linewidths=1.6, zorder=6)
if wrong.any():
    rx, ry = np.ptp(tsne_sig, axis=0)
    label_pos = repel(T_test[wrong] + [0, 0.06 * ry], (0.3 * rx, 0.07 * ry))
    for (x0, y0), (x1, y1), t, p in zip(T_test[wrong], label_pos, y_test[wrong], y_test_pred[wrong]):
        ax.plot([x0, x1], [y0, y1], color=INK2, lw=0.6, zorder=6)
        ax.text(x1, y1, f"{t} → {p}", fontsize=8.5, ha="center", va="bottom", color=INK, path_effects=HALO, zorder=7)
tsne_axes(ax)
handles = cat_handles(class_order, CLASS_STYLE) + [
    Line2D([], [], ls="", marker="o", ms=11, mfc="none", mec=INK, mew=1.6),
    Line2D([], [], ls="", marker="o", ms=6, mfc=CONTEXT, mec=CONTEXT)]
ax.legend(handles, class_order + [f"error ({wrong.sum()})", "training BGC"], loc="center left",
          bbox_to_anchor=(1.01, 0.5), title="Test BGCs (true class)", title_fontsize=10, labelcolor=INK2)
titles(ax, f"Model errors on the held-out test set: {wrong.sum()} of {len(y_test)} BGCs",
       "Black ring = misclassified BGC · label = true class → predicted class")
show_and_save(fig, "3.4", "b_test_errors")

errors_table = pd.DataFrame({
    "data_row": idx_test[wrong], ACC: data.loc[idx_test[wrong], ACC].to_numpy(),
    REG: data.loc[idx_test[wrong], REG].to_numpy() if REG in data.columns else np.nan,
    "true_class": y_test[wrong], "predicted_class": y_test_pred[wrong],
    "predicted_probability": (final_model.predict_proba(X_test[wrong][:, sel_idx]).max(axis=1).round(3)
                                                      if wrong.any() else []),
    "identical_profile_in_training": has_twin[wrong],
    "overlaps_with": [", ".join(sorted(overlap_classes.get(i, ()))) or "—" for i in idx_test[wrong]]})
save_table(errors_table, "3.4", "test_errors")
n_err_overlap = int(sum(bool(overlap_classes.get(i)) for i in idx_test[wrong]))
print(f"Test errors: {wrong.sum()} · of which in BGCs overlapping another BGC of the same genome (1.3): {n_err_overlap}")
errors_table

## 3.5 Correlation circle — after the model (same 25 %)

The same stratified 25 % subsample as in 1.8, now using only the model's *k* domains. Comparing it with 1.8 shows whether the domains that organise the map are part of the model's signature.

In [ ]:
corr_after = save_table(
    correlation_circle(X[corr_idx][:, sel_idx], y[corr_idx], selected_features, "3.5",
                       "correlation_circle_after",
                       f"Correlation circle after the model: the {k_optimal} model domains"),
    "3.5", "correlation_circle_after")
shared = set(corr_before["domain"]) & set(selected_features)
print(f"Domains from the 'before' circle (1.8) that are in the model: {len(shared)} of {len(corr_before)}")
corr_after

---
# Part 4 · Results exploration: groups and architecture

**Question:** clustering the BGCs yields *k* groups (3 by default). Which Pfam domains are responsible for those groups, and how does BGC architecture change from one group to another?

**Methodological reference.** Kralova *et al.* (2026) describe a siderophore cluster at two levels: the cluster architecture (genes coloured by functional category, their Fig. 4A) and the domains and modules of the biosynthetic genes (their Fig. 4B). This part applies the same reading to the groups of this dataset.

| Section | What it does | Main outputs |
|---|---|---|
| 4.1 | Forms the groups and shows which classes each one contains | `4.1a/b/c` |
| 4.2 | Finds the domains responsible for each group (enrichment and decision-tree rules) | `4.2a/b/c` |
| 4.3 | Draws the gene-by-gene architecture of a representative BGC of each group | `4.3_group_architecture` |
| 4.4 | Compares assembly domains, size and protein architectures across groups | `4.4a/b/c` |
| 4.5 | Dedicated analysis of the siderophore BGCs | `4.5a/b/c` |
| 4.6 | Compares BGCs by protein sequence similarity (alignments) | `4.6a/b/c` |

## 4.1 Forming the groups

Groups are formed by **Ward hierarchical clustering** on the coordinates of the t-SNE map from section 1.5a (`CONFIG["group_on"] = "signature"` uses the map built with the model's domains, 3.4a). They are named G1, G2, … from largest to smallest.

- **4.1a · Group map:** each dot is a BGC coloured by its class; the dashed line encloses each group.
- **4.1b · Composition:** how many BGCs of each class fall in each group.
- **4.1c · How many groups are there?** The silhouette measures how compact and separated the groups are for k = 2…8. If it keeps rising beyond the k used, the map has more natural groups and the *k* groups are a coarse summary.

In [ ]:
from scipy.spatial import ConvexHull

if CONFIG["group_on"] == "signature":
    group_space, space_label = tsne_sig, f"t-SNE map with the {k_optimal} model domains (3.4a)"
else:
    group_space, space_label = tsne_all, "t-SNE map with all domains (1.5a)"

# ---- Silhouette for k = 2…8 (higher = more compact, better separated groups) ----
sil = pd.DataFrame([{"k": k, "silhouette": silhouette_score(
    group_space, AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(group_space))} for k in range(2, 9)])
save_table(sil.round(4), "4.1", "silhouette")

# ---- Ward with k = n_groups; G1 = the largest group --------------------------------
k_groups = CONFIG["n_groups"]
ward = AgglomerativeClustering(n_clusters=k_groups, linkage="ward").fit_predict(group_space)
by_size = pd.Series(ward).value_counts().index.tolist()
groups = np.array([f"G{by_size.index(g) + 1}" for g in ward])
group_names = [f"G{i + 1}" for i in range(k_groups)]
group_sizes = pd.Series(groups).value_counts().reindex(group_names)

composition = (pd.crosstab(pd.Series(groups, name="group"), pd.Series(y, name="class"))
               .reindex(index=group_names, columns=class_order, fill_value=0))
save_table(composition.reset_index(), "4.1", "group_composition")
save_table(pd.DataFrame({"data_row": np.arange(len(y)), ACC: data[ACC].to_numpy(),
                         REG: data[REG].to_numpy() if REG in data.columns else np.nan,
                         "class": y, "group": groups}), "4.1", "group_of_each_bgc")


def group_outlines(ax, E, labels, names, sizes=None, fontsize=12):
    """Dashed outline (padded convex hull) and name of each group."""
    span = float(np.ptp(E, axis=0).max())
    tops = []
    for g in names:
        pts = E[labels == g]
        if len(pts) < 3:
            continue
        hull = pts[ConvexHull(pts).vertices]
        d = hull - pts.mean(axis=0)
        hull = hull + d / np.linalg.norm(d, axis=1, keepdims=True) * 0.03 * span      # margin around the points
        ax.add_patch(Polygon(hull, closed=True, fill=False, ls=(0, (5, 3)), lw=1.4, edgecolor=INK2, zorder=4))
        top = hull[hull[:, 1].argmax()]
        tops.append(top[1])
        text = g if sizes is None else f"{g} · {sizes[g]} BGCs"
        ax.text(top[0], top[1] + 0.012 * span, text, ha="center", va="bottom", fontsize=fontsize, fontweight="bold",
                color=INK, path_effects=HALO, zorder=6)
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, max(hi, max(tops) + 0.09 * span))


# ---- 4.1a Group map -------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 7.4))
cat_scatter(ax, group_space, y, class_order, CLASS_STYLE, ellipses=False, direct_labels=False, s=20, alpha=0.8)
group_outlines(ax, group_space, groups, group_names, group_sizes)
tsne_axes(ax)
cat_legend(ax, class_order, CLASS_STYLE, counts, title="Class (n BGCs)")
titles(ax, f"{k_groups} groups of BGCs on the t-SNE map",
       f"Ward on the {space_label}\nDashed line = outline of each group · colour = BGC class")
show_and_save(fig, "4.1", "a_groups_tsne")

# ---- 4.1b Group composition ------------------------------------------------------------
share = composition / composition.sum(axis=0).replace(0, 1)
fig, ax = plt.subplots(figsize=(0.95 * len(class_order) + 3, 0.7 * k_groups + 2.4))
heatmap(ax, share, [f"{g}  (n = {group_sizes[g]})" for g in group_names], class_order, vmin=0, vmax=1,
        annot=np.where(composition > 0, composition.astype(str), ""), cbar_label="Fraction of the class",
        annot_size=10)
rotate_xlabels(ax)
ax.tick_params(axis="y", rotation=0)
titles(ax, "Which classes make up each group",
       "Number = BGCs · colour = fraction of the class's BGCs that falls in the group · empty = 0")
show_and_save(fig, "4.1", "b_group_composition")

# ---- 4.1c Silhouette -------------------------------------------------------------------------------------
best_k = int(sil.loc[sil["silhouette"].idxmax(), "k"])
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot(sil["k"], sil["silhouette"], color=SERIES[0], marker="o", ms=5)
ax.axvline(k_groups, color=INK2, lw=1)
ax.annotate(f"k used = {k_groups}", xy=(k_groups, sil.set_index("k").at[k_groups, "silhouette"]), xytext=(8, 10),
            textcoords="offset points", fontsize=10.5, fontweight="bold", color=INK, path_effects=HALO)
ax.set_xlabel("Number of groups (k)")
ax.set_ylabel("Silhouette")
titles(ax, "Clustering quality by number of groups",
       "Silhouette: higher = more compact, better separated groups"
       + (f"\nIt keeps rising after k = {k_groups}: the map has more natural groups; the {k_groups} "
          "groups are a coarse summary" if best_k > k_groups else ""))
show_and_save(fig, "4.1", "c_silhouette")

for g in group_names:
    comp = composition.loc[g]
    print(f"{g} ({group_sizes[g]} BGCs): " + ", ".join(f"{c} {n}" for c, n in comp[comp > 0].sort_values(ascending=False).items()))
print(f"Silhouette at k = {k_groups}: {sil.set_index('k').at[k_groups, 'silhouette']:.3f} · k with the highest silhouette: {best_k}"
      + (f" → the map has more natural groups than {k_groups}; the groups are a coarse summary of the map"
         if best_k > k_groups else ""))

## 4.2 Domains responsible for the groups

For each Pfam domain and each group (the group against all other BGCs):

| Column | What it measures |
|---|---|
| `prevalence_group` / `prevalence_rest` | Fraction of BGCs carrying the domain inside and outside the group |
| `log2_OR` | log2 odds ratio (Haldane correction). Positive = more frequent in the group (↑); negative = less frequent (↓), i.e. the domain is *missing* from the group |
| `p_fisher` / `q_fdr` | Fisher's exact test and its multiple-testing correction (Benjamini–Hochberg) |
| `AUC` / `discrimination` | Ability of the domain on its own to separate the group from the rest: discrimination = \|2·AUC − 1\|, from 0 (no separation) to 1 (perfect separation) |

The **marker domains** of each group are those with the highest discrimination and `q_fdr < 0.05`. In addition, a shallow **decision tree**, trained only on the presence or absence of common domains, gives the minimal combination of domains that reproduces the groups.

Each domain also gets a **functional category** from its Pfam family (editable lists at the top of the cell; used in section 4.3):

- **Core biosynthesis** depends on the BGC's class: it is the Pfam domains equivalent to the profiles in antiSMASH 7's detection rule for that class (`CORE_RULES`; e.g. betalactone = `AMP-binding` + `HMGL-like`, ni-siderophore = `IucA_IucC`, nrps = `Condensation` + `AMP-binding`, plus T and TE of the NRPS line). Thus `HMGL-like` is core biosynthesis in a betalactone BGC but not in an ectoine BGC. In addition, any protein with two or more distinct NRPS/PKS assembly domains (e.g. C-A-T) is core biosynthesis.
- **Precursor supply, tailoring, transport and regulation** are assigned per Pfam family (`DOMAIN_CATEGORIES`); everything else is "other domain".

The cell checks that every list entry matches real domains in the CSV. In the enrichment table, the `antismash_rule_of` column shows which classes use the domain in their rule.

- **4.2a:** percentage of BGCs in each group that carry each marker domain.
- **4.2b:** the decision tree drawn as yes/no rules.
- **4.2c:** where the BGCs carrying the two best markers of each group lie on the map.

In [ ]:
from functools import lru_cache

# ---- Functional categories (editable) -----------------------------------------------------------
CAT_CORE, CAT_PRECURSOR = "core biosynthesis", "precursor supply"
CAT_TAILORING, CAT_TRANSPORT = "tailoring", "transport and iron uptake"
CAT_REGULATION, CAT_OTHER, CAT_NONE = "regulation and signalling", "other domain", "no Pfam domain"

# (1) NRPS/PKS assembly-line domains. A gene with two or more of them (e.g. C-A-T) is "core biosynthesis" in any BGC.
ASSEMBLY_LINE = ["Condensation", "AMP-binding", "PP-binding", "Thioesterase", "ketoacyl-synt", "Ketoacyl-synt_C",
                 "KAsynt_C_assoc", "Acyl_transf_1", "PS-DH"]
# (2) Pfam families equivalent to the profiles in antiSMASH 7's detection rule for each class:
#     "core biosynthesis" only in BGCs of that class (e.g. HMGL-like is core in betalactone, not in ectoine).
CORE_RULES = {"betalactone": ["AMP-binding", "HMGL-like"], "redox-cofactor": ["Radical_SAM", "SPASM", "PqqD"],
              "naggn": ["Asn_synthase", "GATase_7", "Peptidase_M42"], "ectoine": ["Ectoine_synth"],
              "terpene": ["SQS_PSY", "Lycopene_cycl"], "arylpolyene": ["ketoacyl-synt", "Ketoacyl-synt_C"],
              "ni-siderophore": ["IucA_IucC"],
              "nrps": ["Condensation", "AMP-binding", "PP-binding", "Thioesterase"],          # + T and TE of the NRPS line
              "nrp-metallophore": ["Condensation", "AMP-binding", "PP-binding", "Thioesterase"]}
# (3) Other categories by Pfam family: exact name or start of the name (case-insensitive).
DOMAIN_CATEGORIES = {
    CAT_PRECURSOR: ["Chorismate_bind", "Isochorismatase", "Lys_Orn_oxgnase", "polyprenyl_synt"],
    CAT_TAILORING: ["Methyltransf_", "adh_short", "KR", "Epimerase", "Glycos_transf_", "Glyco_tran", "UDPGT",
                    "Acetyltransf_", "Aldedh", "Aldo_ket_red", "FAD_binding_", "Trp_halogenase", "Abhydrolase_",
                    "NAD_binding_", "Oxidored_", "Aminotran_", "TauD", "PhyH"],
    CAT_TRANSPORT: ["ABC_tran", "ABC_membrane", "ABC2_membrane", "BPD_transp_", "MFS_", "Peripla_BP_", "SBP_bac_",
                    "FecCD", "TonB_", "Plug", "OMP_b-brl", "DctP", "DctM", "DctQ", "ACR_tran", "OEP", "EamA",
                    "Sugar_tr", "Na_H_Exchanger", "FhuF"],
    CAT_REGULATION: ["HTH_", "Response_reg", "HisKA", "HATPase_c", "LysR_substrate", "TetR_", "GntR", "MarR", "MerR",
                     "AraC_", "Sigma70_", "Sigma54_", "Trans_reg_C", "AsnC_trans_reg", "GerE", "GGDEF", "EAL",
                     "PAS", "HAMP"],
}
CATEGORY_COLORS = {CAT_CORE: "#e34948", CAT_PRECURSOR: "#eda100", CAT_TAILORING: "#4a3aa7", CAT_TRANSPORT: "#2a78d6",
                   CAT_REGULATION: "#1baf7a", CAT_OTHER: "#a9a8a0", CAT_NONE: "#dcdbd4"}
RULE_CLASSES = {}                                                # domain → classes whose rule uses it
for _cls, _doms in CORE_RULES.items():
    for _d in _doms:
        RULE_CLASSES.setdefault(_d, []).append(_cls)


@lru_cache(maxsize=None)
def domain_category(name, cls=None):
    """Category of a domain. With `cls`, "core biosynthesis" follows that class's rule;
    without `cls`, any domain used by some rule is core."""
    if (name in CORE_RULES.get(cls, ())) if cls is not None else (name in ASSEMBLY_LINE or name in RULE_CLASSES):
        return CAT_CORE
    n = name.lower()
    for cat, entries in DOMAIN_CATEGORIES.items():
        if any(n.startswith(e.lower()) for e in entries):
            return cat
    return CAT_OTHER


all_pfam = [c.split(":", 1)[-1] for c in pfam_columns]          # every domain in the CSV
entries = ASSEMBLY_LINE + [d for v in CORE_RULES.values() for d in v] + [e for v in DOMAIN_CATEGORIES.values() for e in v]
unmatched = sorted({e for e in entries if not any(n.lower().startswith(e.lower()) for n in all_pfam)})
cat_count = pd.Series([domain_category(n) for n in domain_names]).value_counts()
print("Domains per category: " + " · ".join(f"{c} {cat_count.get(c, 0)}" for c in CATEGORY_COLORS if c != CAT_NONE))
print("List entries with no domain in the CSV:", ", ".join(unmatched) if unmatched else "none")


def bh_fdr(p):
    """Benjamini–Hochberg correction (false discovery rate)."""
    p = np.asarray(p, float)
    order = np.argsort(p)
    q = np.empty(len(p))
    q[order] = np.minimum.accumulate((p[order] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    return np.minimum(q, 1)


# ---- Enrichment: each group against the rest --------------------------------
rows = []
for g in group_names:
    m = groups == g
    for j, name in enumerate(domain_names):
        a, c = int(P[m, j].sum()), int(P[~m, j].sum())            # BGCs with the domain inside / outside
        if a + c == 0:
            continue
        b, d = int(m.sum()) - a, int((~m).sum()) - c
        rows.append({"group": g, "domain": name, "category": domain_category(name),
                     "antismash_rule_of": ", ".join(RULE_CLASSES.get(name, [])),
                     "prevalence_group": a / m.sum(), "prevalence_rest": c / (~m).sum(),
                     "log2_OR": np.log2(((a + .5) * (d + .5)) / ((b + .5) * (c + .5))),     # Haldane correction
                     "p_fisher": fisher_exact([[a, b], [c, d]])[1], "AUC": roc_auc_score(m, X[:, j])})
enrich = pd.DataFrame(rows)
enrich["q_fdr"] = enrich.groupby("group")["p_fisher"].transform(bh_fdr)
enrich["direction"] = np.where(enrich["log2_OR"] > 0, "↑", "↓")
enrich["discrimination"] = (2 * enrich["AUC"] - 1).abs()
save_table(enrich.round(4), "4.2", "domain_enrichment")

markers = (enrich[enrich["q_fdr"] < 0.05]
           .sort_values(["group", "discrimination"], ascending=[True, False])
           .groupby("group").head(CONFIG["marker_domains_per_group"]).reset_index(drop=True))
save_table(markers.round(4), "4.2", "marker_domains")
markers_by_group = {g: set(markers.loc[markers["group"] == g, "domain"]) for g in group_names}

# ---- Decision tree on the presence of common domains --------------------
common = np.where(P.mean(axis=0) >= CONFIG["tree_min_prevalence"])[0]       # avoids rules on rare domains
tree_names = [domain_names[j] for j in common]
tree = DecisionTreeClassifier(max_depth=CONFIG["tree_max_depth"], min_samples_leaf=5, random_state=RS)
tree_cv = cross_val_score(tree, P[:, common], groups, scoring="balanced_accuracy",
                          cv=StratifiedKFold(5, shuffle=True, random_state=RS))
tree.fit(P[:, common], groups)

for g in group_names:
    top3 = markers[markers["group"] == g].head(3)
    print(f"{g}: " + "; ".join(f"{r['domain']} {r['direction']} ({100 * r['prevalence_group']:.0f} % in the group vs "
                               f"{100 * r['prevalence_rest']:.0f} % in the rest)" for _, r in top3.iterrows()))
print(f"Decision tree (depth ≤ {CONFIG['tree_max_depth']}, {len(common)} common domains): cross-validated balanced accuracy = "
      f"{tree_cv.mean():.3f} ± {tree_cv.std():.3f}")
markers[["group", "domain", "direction", "prevalence_group", "prevalence_rest",
         "discrimination", "q_fdr", "category"]].round(3)

**Figures 4.2a – 4.2c.** The previous cell computes the enrichment, the marker domains and the decision tree; this one draws the three figures described above.

In [ ]:
# ---- 4.2a Prevalence of the marker domains in each group ----------------
rows_lab, block_of, M = [], [], []
for g in group_names:
    for _, r in markers[markers["group"] == g].iterrows():
        j = domain_names.index(r["domain"])
        rows_lab.append(f"{r['domain']} {r['direction']}")
        block_of.append(g)
        M.append([100 * P[groups == h, j].mean() for h in group_names])
M = np.array(M)
fig, ax = plt.subplots(figsize=(1.25 * k_groups + 4.6, 0.36 * len(rows_lab) + 2.2))
heatmap(ax, M, rows_lab, [f"{g}\n(n = {group_sizes[g]})" for g in group_names], vmin=0, vmax=100,
        annot=np.vectorize(lambda v: f"{v:.0f}")(M), cbar_label="% of the group's BGCs with the domain")
ax.tick_params(axis="y", rotation=0)
ax.tick_params(axis="x", rotation=0)
for i in range(1, len(block_of)):                                # separates each group's block
    if block_of[i] != block_of[i - 1]:
        ax.axhline(i, color=INK, lw=1.4)
for g in group_names:                                            # block label on the left
    idx = [i for i, b in enumerate(block_of) if b == g]
    if idx:
        ax.text(-0.02, np.mean(idx) + 0.5, f"markers of {g}", transform=ax.get_yaxis_transform(), ha="right", va="center",
                fontsize=10, fontweight="bold", color=INK, rotation=90)
ax.set_yticklabels(rows_lab)
ax.tick_params(axis="y", pad=30)
titles(ax, "Marker domains of each group",
       "Number = % of the group's BGCs with the domain · ↑ more frequent in the group · ↓ less frequent")
show_and_save(fig, "4.2", "a_marker_domains")


# ---- 4.2b Decision tree drawn as yes / no rules ------------------------------------
def draw_tree(ax, tree, names, X_tree, labels):
    t = tree.tree_
    leaf_of = tree.apply(X_tree)
    pos, leaves = {}, []

    def walk(node, depth):                                       # leaves from left to right
        if t.children_left[node] == -1:
            pos[node] = (len(leaves), -depth)
            leaves.append(node)
            return
        walk(t.children_left[node], depth + 1)
        walk(t.children_right[node], depth + 1)
        pos[node] = ((pos[t.children_left[node]][0] + pos[t.children_right[node]][0]) / 2, -depth)

    walk(0, 0)
    for node, (x, yv) in pos.items():
        left, right = t.children_left[node], t.children_right[node]
        if left == -1:                                           # leaf: majority group and purity
            members = pd.Series(labels[leaf_of == node]).value_counts()
            ax.text(x, yv, f"{members.index[0]}\n{members.iloc[0]} of {members.sum()} BGCs", ha="center", va="center",
                    fontsize=10.5, fontweight="bold", color=INK, linespacing=1.4,
                    bbox=dict(boxstyle="round,pad=0.55", fc=SURFACE, ec=INK, lw=1.3), zorder=3)
            continue
        for child, word in ((left, "no"), (right, "yes")):        # left = absent, right = present
            cx, cy = pos[child]
            ax.plot([x, cx], [yv, cy], color=AXIS, lw=1.3, zorder=1)
            ax.text((x + cx) / 2, (yv + cy) / 2, word, ha="center", va="center", fontsize=9.5, color=INK2,
                    bbox=dict(boxstyle="round,pad=0.2", fc=SURFACE, ec="none"), zorder=2)
        ax.text(x, yv, f"Has {names[t.feature[node]]}?", ha="center", va="center", fontsize=10, color=INK,
                bbox=dict(boxstyle="round,pad=0.5", fc="#f0efec", ec=AXIS, lw=1), zorder=3)
    ax.set_xlim(-0.7, len(leaves) - 0.3)
    ax.set_ylim(-tree.get_depth() - 0.5, 0.5)
    ax.axis("off")


n_leaves = tree.get_n_leaves()
fig, ax = plt.subplots(figsize=(max(9, 2.4 * n_leaves), 1.7 * (tree.get_depth() + 1) + 0.8))
draw_tree(ax, tree, tree_names, P[:, common], groups)
titles(ax, "Domain-presence rules that reproduce the groups",
       f"Decision tree of depth ≤ {CONFIG['tree_max_depth']} · cross-validated balanced accuracy = {tree_cv.mean():.2f}\n"
       "Each leaf shows the majority group and how many of its BGCs belong to it")
show_and_save(fig, "4.2", "b_decision_tree")


# ---- 4.2c Where the BGCs carrying each marker are --------------------------------------
def presence_panel(ax, name, g):
    j = domain_names.index(name)
    ax.scatter(*group_space[~P[:, j]].T, s=7, c=CONTEXT, linewidths=0, zorder=1)
    ax.scatter(*group_space[P[:, j]].T, s=14, c=SERIES[0], linewidths=0, zorder=2)
    group_outlines(ax, group_space, groups, group_names, fontsize=10)
    inside, rest = 100 * P[groups == g, j].mean(), 100 * P[groups != g, j].mean()
    ax.set_title(f"{name}\n{g}: {inside:.0f} %  ·  rest: {rest:.0f} %", loc="left", fontsize=10.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)


top_markers, shown = [], set()                                  # 2 per group: ↑ first, no repeated domains
for g in group_names:
    cand = markers[markers["group"] == g].sort_values(["direction", "discrimination"], ascending=[True, False],
                                                          key=lambda c: c.map({"↑": 0, "↓": 1}) if c.name == "direction" else c)
    picks = [n for n in cand["domain"] if n not in shown][:2]
    shown.update(picks)
    top_markers += [(g, n) for n in picks]
ncol = 3
nrow = int(np.ceil(len(top_markers) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.0 * ncol, 4.6 * nrow), squeeze=False)
for ax, (g, name) in zip(axes.ravel(), top_markers):
    presence_panel(ax, name, g)
for ax in axes.ravel()[len(top_markers):]:
    ax.axis("off")
fig.legend([Line2D([], [], ls="", marker="o", ms=7, mfc=SERIES[0], mec=SERIES[0]), Line2D([], [], ls="", marker="o", ms=6, mfc=CONTEXT, mec=CONTEXT)],
           ["BGC with the domain", "BGC without the domain"], loc="upper center",
           bbox_to_anchor=(0.5, -0.0), ncol=2, labelcolor=INK2)
fig.suptitle("Where the BGCs with the two best markers of each group lie",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
show_and_save(fig, "4.2", "c_markers_on_map")
for g, name in top_markers:                                      # individual panels
    fig, ax = plt.subplots(figsize=(6.5, 5.6))
    presence_panel(ax, name, g)
    show_and_save(fig, "4.2", f"c_marker_{g}_{slug(name)}", individual=True)

## 4.3 Architecture of the representative BGC of each group

The `protein_seq_pfam_map` column holds each BGC's proteins (separated by `|`) with every domain replaced by `[Domain(value)]`; comparing it with `protein_sequences` recovers which domains each gene carries and in which order. From this, for the **representative** of each group (its *medoid*: the BGC most similar on average to the rest of the group, by Jaccard distance between domain profiles), a figure in the style of Fig. 4 of Kralova *et al.* (2026) is drawn, without chemical structures:

- **A · Cluster architecture.** Each gene is an arrow whose length is proportional to its protein (aa × 3 bp), coloured by the functional category of its domains (4.2; if it has several, the highest-priority one wins). The three groups share the same scale. Genes in panel B and genes carrying a marker domain of the group (4.2) are numbered; the latter have a **black border** and a bold number, showing where in the cluster the group's responsible domains lie.
- **B · Domains of the core biosynthetic genes**, in N→C order. Assembly-line abbreviations: C = condensation (`Condensation`), A = adenylation (`AMP-binding`), T = thiolation (`PP-binding`), TE = thioesterase, KS = ketosynthase (`ketoacyl-synt`), AT = acyltransferase (`Acyl_transf_1`), DH (`PS-DH`), KR, MT = methyltransferase; all other domains keep their Pfam name. **Modules** (M1, M2…) are detected automatically: they start at C or KS (or at A when no C precedes it) and only count if they contain A + T (NRPS) or KS + T (PKS). Domains with a **black border** are markers of the group.
- **About `C/E`:** Pfam does not distinguish the epimerisation (E) domain from the condensation domain: both belong to the `Condensation` family. When a C domain follows a T and closes the gene or is followed by another C (the typical position of an E), it is labelled `C/E`.

Table `4.3_representative_genes` lists every gene of each representative (number, length, category and domains) to read the figure gene by gene.

**The map does not contain every domain in the table.** The cell measures how much of the table's domain presences also appear in `protein_seq_pfam_map` and saves the missing domains (`4.3_domains_missing_from_map`). Everything that depends on gene order (this section, the modules in 4.4 and 4.5, and Part 5) only sees the domains present in the map; whether a domain is present in a BGC is always taken from the table columns.

In [ ]:
from scipy.spatial.distance import pdist, squareform

TAG_RE = re.compile(r"\[([^\[\]()]+)\(([0-9.]+)\)\]")              # tag [Domain(0.94)]
AA_OK = set("ACDEFGHIKLMNPQRSTVWYBZX")
# Assembly-domain abbreviations (exact Pfam name)
DOMAIN_ABBREV = {"Condensation": "C", "AMP-binding": "A", "PP-binding": "T", "Thioesterase": "TE", "ketoacyl-synt": "KS",
                 "Acyl_transf_1": "AT", "PS-DH": "DH", "KR": "KR"}
# C-terminal parts: merged into the previous box of the same domain; if the map lacks the N-terminal part they form their own box
C_TERMINAL = {"AMP-binding_C": "A", "Ketoacyl-synt_C": "KS", "KAsynt_C_assoc": "KS"}
SHORT_LABELS = {"C", "A", "T", "TE", "KS", "AT", "DH", "KR", "MT", "C/E"}
MODULE_START, MODULE_BODY = {"C", "KS"}, {"A", "T", "C/E", "AT", "DH", "KR", "MT"}


def parse_architecture(map_cell, seq_cell):
    """Genes and domains (with aa position) from "...SEQ[Domain(0.94)]SEQ...|..."."""
    if pd.isna(map_cell):
        return []
    maps = str(map_cell).split("|")
    seqs = str(seq_cell).split("|") if not pd.isna(seq_cell) else []
    seqs = seqs + [None] * (len(maps) - len(seqs))
    genes = []
    for mp, seq in zip(maps, seqs):
        parts = TAG_RE.split(mp)                                 # [text, name, value, text, …]
        pos, pending, doms = 0, [], []
        for k in range(0, len(parts), 3):
            chunk = parts[k]
            if chunk:
                if pending:                                      # the domain ends where the sequence resumes
                    end = seq.find(chunk, pos) if seq else -1
                    end = end if end > pos else pos + 100
                    doms += [(n, v, pos, end) for n, v in pending]
                    pending, pos = [], end
                pos += len(chunk)
            if k + 1 < len(parts):
                pending.append((parts[k + 1], float(parts[k + 2])))
        length = len(seq) if seq else pos
        doms += [(n, v, pos, max(length, pos + 1)) for n, v in pending]
        genes.append({"length_aa": max(length, 1), "domains": doms})
    return genes


def abbrev(name):
    if name in DOMAIN_ABBREV:
        return DOMAIN_ABBREV[name]
    if name.startswith("Methyltransf"):
        return "MT"
    return name if len(name) <= 13 else name[:12] + "…"


def gene_category(gene, cls):
    """Gene category = the highest-priority one among
    its domains (order of CATEGORY_COLORS). A protein with two or more distinct assembly domains (e.g. C-A-T, KS-AT) is always core."""
    names = {d[0] for d in gene["domains"]}
    if len({DOMAIN_ABBREV.get(n) or C_TERMINAL.get(n) for n in names} & {"C", "A", "T", "TE", "KS", "AT", "DH"}) >= 2:
        return CAT_CORE
    cats = {domain_category(n, cls) for n in names}
    return next((c for c in CATEGORY_COLORS if c in cats), CAT_NONE)


def gene_boxes(gene, marker_set=frozenset()):
    """Gene domains in N→C order: (abbreviation, Pfam name, marker?)."""
    out = []
    for n, *_ in sorted(gene["domains"], key=lambda d: d[2]):
        if n in C_TERMINAL:                                      # C-terminal part of A or KS
            if not (out and out[-1][0] == C_TERMINAL[n]):
                out.append([C_TERMINAL[n], n, n in marker_set])
            continue
        ab = abbrev(n)
        if not (out and out[-1][1] == n and ab not in ("C", "A", "T")):
            out.append([ab, n, n in marker_set])                 # merges overlapping repeats (except C, A, T)
    for i in range(1, len(out)):                                 # C in an E position → "C/E"
        nxt = out[i + 1][0] if i + 1 < len(out) else None
        if out[i][0] == "C" and out[i - 1][0] == "T" and nxt in (None, "C"):
            out[i][0] = "C/E"
    return [tuple(b) for b in out]


def assign_modules(core_boxes):
    """NRPS/PKS modules per gene: (start, end, "M1"), numbered along the BGC."""
    counter, result = 0, []
    for boxes in core_boxes:
        mods, cur = [], None
        for i, (ab, *_rest) in enumerate(boxes):
            if ab in MODULE_START or (ab == "A" and (cur is None or "A" in cur["types"])):
                cur = {"start": i, "end": i, "types": {ab}}
                mods.append(cur)
            elif ab in MODULE_BODY and cur is not None:
                cur["end"], cur["types"] = i, cur["types"] | {ab}
            else:
                cur = None                                       # TE and stray domains stay outside modules
        labelled = []
        for m in mods:
            if {"A", "T"} <= m["types"] or {"KS", "T"} <= m["types"]:     # NRPS: A + T · PKS: KS + T
                counter += 1
                labelled.append((m["start"], m["end"], f"M{counter}"))
        result.append(labelled)
    return result


def core_genes(genes, cls, marker_set=frozenset()):
    """Core biosynthetic genes with their domain boxes: [(gene number, boxes)]."""
    out = [(k, gene_boxes(g, marker_set)) for k, g in enumerate(genes, start=1) if gene_category(g, cls) == CAT_CORE]
    return [(k, b) for k, b in out if b]


def n_modules(genes, cls):
    return sum(len(m) for m in assign_modules([b for _, b in core_genes(genes, cls)]))


# ---- Drawing: panel A (cluster) and panel B (core-gene domains) --------
GAP_KB, BOX_FILL, ROW_H = 0.12, "#f6dcdc", 2.35


def draw_panel_a(ax, genes, kb_max, cls, numbered, marker_set):
    total = sum(g["length_aa"] * 3e-3 for g in genes) + GAP_KB * (len(genes) - 1)
    ax.plot([0, total], [0, 0], color=MUTED, lw=1, zorder=1)        # cluster backbone
    x, last_label, H = 0.0, -1e9, 0.42
    for k, gene in enumerate(genes, start=1):
        L = gene["length_aa"] * 3e-3                                 # kb = aa × 3 bp
        head = min(0.45, 0.45 * L)
        marker = any(d[0] in marker_set for d in gene["domains"])
        ax.add_patch(Polygon([(x, -H / 2), (x + L - head, -H / 2), (x + L - head, -H * 0.85), (x + L, 0),
                              (x + L - head, H * 0.85), (x + L - head, H / 2), (x, H / 2)], closed=True,
                             facecolor=CATEGORY_COLORS[gene_category(gene, cls)], edgecolor=INK if marker else SURFACE,
                             lw=1.8 if marker else 0.8, zorder=3 if marker else 2))
        if k in numbered:
            lift = 0.28 if x + L / 2 - last_label < 0.6 else 0.0     # staggers crowded numbers
            ax.text(x + L / 2, H * 0.85 + 0.08 + lift, str(k), ha="center", va="bottom", fontsize=9, color=INK,
                    fontweight="bold" if marker else "normal")
            last_label = x + L / 2 if lift == 0 else last_label
        x += L + GAP_KB
    ax.plot([kb_max - 2.5, kb_max], [-0.72, -0.72], color=INK, lw=1.2)   # scale bar (bottom right)
    ax.text(kb_max - 2.62, -0.72, "2.5 kb", ha="right", va="center", fontsize=8.5, color=INK2)
    ax.text(-0.35, 0, "A", fontsize=13, fontweight="bold", color=INK, ha="right", va="center")
    ax.set_xlim(-0.75, kb_max + 0.3)
    ax.set_ylim(-0.95, 1.05)
    ax.axis("off")


def layout_panel_b(core, max_units=34):
    """Places the domain boxes of core genes in rows (each whole gene on one row)."""
    items, row, x = [], 0, 0.0
    for (k, boxes), mods in zip(core, assign_modules([b for _, b in core])):
        widths = [max(1.0, (0.24 if ab in SHORT_LABELS else 0.19) * len(ab) + 0.5) for ab, *_ in boxes]
        w_boxes = sum(widths) + 0.14 * (len(widths) - 1)
        w_gene = max(w_boxes, 2.6)                               # the arrow must fit "Gene N"
        if x > 0 and x + w_gene > max_units:
            row, x = row + 1, 0.0
        xs, cx = [], x + (w_gene - w_boxes) / 2
        for w in widths:
            xs.append((cx, w))
            cx += w + 0.14
        items.append({"gene": k, "boxes": boxes, "xs": xs, "row": row, "x0": x, "x1": x + w_gene, "mods": mods})
        x += w_gene + 0.9
    return items, row + 1


def draw_panel_b(ax, items, n_rows, max_units=34):
    for it in items:
        yb, x0, x1 = -it["row"] * ROW_H, it["x0"], it["x1"]
        ay, ah = yb + 1.4, 0.5                                   # gene arrow
        head = min(0.6, 0.3 * (x1 - x0))
        ax.add_patch(Polygon([(x0, ay), (x1 - head, ay), (x1 - head, ay - 0.1), (x1, ay + ah / 2),
                              (x1 - head, ay + ah + 0.1), (x1 - head, ay + ah), (x0, ay + ah)],
                             closed=True, facecolor=CATEGORY_COLORS[CAT_CORE], edgecolor="none"))
        ax.text((x0 + x1 - head) / 2, ay + ah / 2, f"Gene {it['gene']}", ha="center", va="center", color="white",
                fontsize=9.5, fontweight="bold")
        for s, e, lab in it["mods"]:                             # modules M1, M2…
            mx0, mx1 = it["xs"][s][0], it["xs"][e][0] + it["xs"][e][1]
            ax.plot([mx0 + 0.05, mx1 - 0.05], [yb + 0.98] * 2, color=INK, lw=1)
            ax.text((mx0 + mx1) / 2, yb + 1.02, lab, ha="center", va="bottom", fontsize=8.5, fontweight="bold", color=INK)
        for (ab, name, mk), (bx, w) in zip(it["boxes"], it["xs"]):
            ax.add_patch(FancyBboxPatch((bx, yb), w, 0.8, boxstyle="round,pad=0,rounding_size=0.18", facecolor=BOX_FILL,
                                        edgecolor=INK if mk else "none", lw=2.0 if mk else 0))
            ax.text(bx + w / 2, yb + 0.4, ab, ha="center", va="center", color=INK, fontweight="bold",
                    fontsize=10 if ab in SHORT_LABELS else 8)
    ax.text(-0.35, 0.4, "B", fontsize=13, fontweight="bold", color=INK, ha="right", va="center")
    ax.set_xlim(-0.75, max_units + 0.3)
    ax.set_ylim(-(n_rows - 1) * ROW_H - 0.25, 2.0)
    ax.axis("off")


def panel_b_rows(genes, cls):
    return layout_panel_b(core_genes(genes, cls))[1]


def draw_architecture(fig, genes, title, kb_max, cls, marker_set=frozenset()):
    core = core_genes(genes, cls, marker_set)
    items, n_rows = layout_panel_b(core)
    numbered = {k for k, _ in core} | {k for k, g in enumerate(genes, start=1) if any(d[0] in marker_set for d in g["domains"])}
    gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 0.25 + 0.95 * max(n_rows, 1)], hspace=0.05)
    axA, axB = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    draw_panel_a(axA, genes, kb_max, cls, numbered, marker_set)
    axA.set_title(title, loc="left", fontsize=12, fontweight="bold", color=INK, pad=4)
    if items:
        draw_panel_b(axB, items, n_rows)
    else:
        axB.text(0.0, 0.5, "B · No recognised core biosynthetic genes", ha="left",
                 va="center", fontsize=10, color=MUTED, transform=axB.transAxes)
        axB.axis("off")


def block_height(genes, cls):
    return 1.0 + 0.25 + 0.95 * max(panel_b_rows(genes, cls), 1)


def architecture_legend(fig, y=0.0):
    handles = [plt.Rectangle((0, 0), 1, 1, color=CATEGORY_COLORS[c]) for c in CATEGORY_COLORS]
    handles.append(plt.Rectangle((0, 0), 1, 1, facecolor="white", edgecolor=INK, lw=2))
    fig.legend(handles, list(CATEGORY_COLORS) + ["black border = marker gene or domain of the group (4.2)"],
               ncol=4, loc="upper center", bbox_to_anchor=(0.5, y), fontsize=9.5, handlelength=1.4, labelcolor=INK2)


def medoid(idx):
    """BGCs sorted by mean Jaccard distance to the rest (the first one is the medoid)."""
    D = np.nan_to_num(squareform(pdist(P[idx], metric="jaccard")), nan=1.0)
    return idx[np.argsort(D.mean(axis=1))]


def bgc_label(i):
    return f"{data.iloc[i][ACC]} region {data.iloc[i][REG] if REG in data.columns else i}"


def plot_architectures(blocks, kb, section, name, individual_name):
    """blocks = [(title, row, class, markers, suffix)]: combined figure + one figure per block."""
    heights = [block_height(all_arch[i], c) for _, i, c, _, _ in blocks]
    fig = plt.figure(figsize=(16, 1.1 * sum(heights) + 0.8))
    for sf, (t, i, c, mk, _) in zip(np.atleast_1d(fig.subfigures(len(blocks), 1, height_ratios=heights, hspace=0.04)), blocks):
        draw_architecture(sf, all_arch[i], t, kb, c, mk)
    architecture_legend(fig, y=0.0)
    show_and_save(fig, section, name)
    for (t, i, c, mk, suffix), h in zip(blocks, heights):
        fig = plt.figure(figsize=(16, 1.1 * h + 0.8))
        draw_architecture(fig, all_arch[i], t, kb, c, mk)
        architecture_legend(fig, y=0.0)
        show_and_save(fig, section, f"{individual_name}_{suffix}", individual=True)


map_col, seq_col = CONFIG["pfam_map_column"], CONFIG["sequence_column"]
ARCH_OK = map_col in data.columns
if not ARCH_OK:
    print(f"Column not found '{map_col}': sections 4.3 – 4.5 are skipped.")
else:
    seq_src = data[seq_col] if seq_col in data.columns else pd.Series([np.nan] * len(data))
    all_arch = [parse_architecture(m, s) for m, s in zip(data[map_col], seq_src)]
    has_arch = np.array([len(a) > 0 for a in all_arch])
    print(f"Architectures rebuilt: {has_arch.sum()} of {len(all_arch)} BGCs · genes per BGC (median): "
          f"{int(np.median([len(a) for a in all_arch if a]))}")

    # ---- How much of the table's domains does the map contain? --------------------
    col_of = {name: j for j, name in enumerate(domain_names)}
    in_map = np.zeros(P.shape, bool)                             # in_map[BGC, domain] = the domain appears in the BGC's map
    for i, genes in enumerate(all_arch):
        for gn in genes:
            for d in gn["domains"]:
                if d[0] in col_of:
                    in_map[i, col_of[d[0]]] = True
    map_gaps = pd.DataFrame({"domain": domain_names, "BGCs_in_table": P.sum(axis=0),
                             "BGCs_in_map": (P & in_map).sum(axis=0)})
    map_gaps["missing_from_map"] = map_gaps["BGCs_in_table"] - map_gaps["BGCs_in_map"]
    map_gaps = save_table(map_gaps[map_gaps["missing_from_map"] > 0]
                          .sort_values("missing_from_map", ascending=False).reset_index(drop=True),
                          "4.3", "domains_missing_from_map")
    map_coverage = float((P & in_map).sum() / P.sum())
    print(f"The map holds {int((P & in_map).sum()):,} of the {int(P.sum()):,} domain presences in the table "
          f"({100 * map_coverage:.1f} %). Most often missing: "
          + ", ".join(f"{r['domain']} ({r['missing_from_map']} BGCs)" for _, r in map_gaps.head(5).iterrows()))

    reps = [(g, i) for g in group_names for i in medoid(np.where((groups == g) & has_arch)[0])[:CONFIG["arch_representatives"]]]
    kb_max = max(sum(gn["length_aa"] * 3e-3 for gn in all_arch[i]) + GAP_KB * len(all_arch[i]) for _, i in reps)
    plot_architectures([(f"{g} · representative: {y[i]} · {bgc_label(i)} · {len(all_arch[i])} genes", i, y[i],
                         markers_by_group[g], g) for g, i in reps], kb_max, "4.3", "group_architecture",
                       "architecture")

    rep_genes = save_table(pd.DataFrame([{
        "group": g, "BGC": bgc_label(i), "class": y[i], "gene": k, "length_aa": gn["length_aa"],
        "category": gene_category(gn, y[i]),
        "domains_N_to_C": " · ".join(d[0] for d in sorted(gn["domains"], key=lambda d: d[2])),
        "marker_domains": ", ".join(sorted({d[0] for d in gn["domains"]} & markers_by_group[g])),
    } for g, i in reps for k, gn in enumerate(all_arch[i], start=1)]), "4.3", "representative_genes")
    for g, i in reps:
        print(f"{g}: representative = {y[i]} · {bgc_label(i)} · genes with a marker domain: "
              + (", ".join(str(k) for k, gn in enumerate(all_arch[i], start=1) if any(d[0] in markers_by_group[g] for d in gn["domains"])) or "—"))

## 4.4 Modules, size and protein architectures per group

In the reference article, the decisive architectural difference between related clusters was the presence or absence of a complete NRPS module. Here groups are compared on:

- **4.4a:** what percentage of each group's BGCs carries each assembly domain (C, A, T, TE, KS, AT; from the table) and what percentage has at least one NRPS/PKS module (from the map);
- **4.4b:** the number of genes (`num_cds_total`) and BGC length (`length_kb`), as given in the table;
- **4.4c:** the **protein architectures** (ordered sequence of domains in one protein, e.g. `Condensation · AMP-binding · AMP-binding_C · PP-binding`) whose frequency changes most between groups.

In [ ]:
from collections import Counter

if ARCH_OK:
    ASSEMBLY = {"C": "Condensation", "A": "AMP-binding", "T": "PP-binding", "TE": "Thioesterase", "KS": "ketoacyl-synt",
                "AT": "Acyl_transf_1"}
    per_bgc = pd.DataFrame({"group": groups, "class": y})
    per_bgc["modules"] = [n_modules(a, c) for a, c in zip(all_arch, y)]     # NRPS/PKS modules (A + T or KS + T), from the map
    per_bgc["genes"] = data["num_cds_total"].to_numpy() if "num_cds_total" in data.columns else [len(a) for a in all_arch]
    per_bgc["kb"] = data["length_kb"].to_numpy() if "length_kb" in data.columns else [
        sum(gn["length_aa"] for gn in a) * 3e-3 for a in all_arch]
    save_table(per_bgc, "4.4", "architecture_per_bgc")

    # Presence of each assembly domain from the TABLE (complete); modules from the MAP
    MODULE_COL = "≥ 1 NRPS/PKS\nmodule"
    has = pd.DataFrame({ab: (P[:, col_of[pf]] if pf in col_of else np.zeros(len(y), bool)) for ab, pf in ASSEMBLY.items()})
    has[MODULE_COL] = per_bgc["modules"].to_numpy() > 0
    pct_with = 100 * has.groupby(groups).mean().reindex(group_names)
    summary = pd.concat([group_sizes.rename("n_BGC"),
                         pct_with.round(0).rename(columns=lambda c: "pct_with_" + c.replace("\n", " ")),
                         per_bgc.groupby("group")["modules"].mean().round(2).rename("modules_mean"),
                         per_bgc.groupby("group")[["genes", "kb"]].median().add_prefix("median_")], axis=1)
    save_table(summary.rename_axis("group").reset_index(), "4.4", "summary_per_group")

    # ---- 4.4a Assembly domains and modules per group -------------------------------
    fig, ax = plt.subplots(figsize=(1.15 * pct_with.shape[1] + 3.2, 0.8 * k_groups + 2.4))
    heatmap(ax, pct_with, [f"{g} (n = {group_sizes[g]})" for g in group_names],
            [f"{c}\n({ASSEMBLY[c]})" if c in ASSEMBLY else c for c in pct_with.columns], vmin=0, vmax=100,
            annot=np.vectorize(lambda v: f"{v:.0f}")(pct_with.to_numpy()), cbar_label="% of the group's BGCs",
            annot_size=10)
    ax.tick_params(axis="y", rotation=0)
    ax.tick_params(axis="x", rotation=0, labelsize=9)
    titles(ax, "Assembly domains and modules per group",
           "Number = % of the group's BGCs with the domain (from the table) · last column = % with at least one module")
    show_and_save(fig, "4.4", "a_assembly_domains")

    # ---- 4.4b Genes and length per group ------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 0.6 * k_groups + 2.4), sharey=True)
    labels_g = [f"{g} (n = {group_sizes[g]})" for g in group_names]
    hbox(ax1, [per_bgc.loc[groups == g, "genes"] for g in group_names], labels_g)
    hbox(ax2, [per_bgc.loc[groups == g, "kb"] for g in group_names], labels_g)
    ax1.set_xlabel("Genes per BGC")
    ax2.set_xlabel("BGC length (kb)")
    ax1.set_title("Genes per BGC", loc="left", fontsize=11)
    ax2.set_title("BGC length", loc="left", fontsize=11)
    fig.suptitle("BGC size per group (box = central 50 %, thick line = median)",
                 x=0.01, ha="left", fontsize=13, fontweight="bold")
    fig.tight_layout()
    show_and_save(fig, "4.4", "b_group_size")

    # ---- 4.4c Protein architectures that change most between groups --------
    freq = {g: Counter() for g in group_names}
    for a, g in zip(all_arch, groups):
        freq[g].update({" · ".join(d[0] for d in sorted(gn["domains"], key=lambda d: d[2])) for gn in a if gn["domains"]})
    arch_df = pd.DataFrame({g: {k: v / group_sizes[g] for k, v in freq[g].items()} for g in group_names}).fillna(0)
    arch_df["spread"] = arch_df.max(axis=1) - arch_df.min(axis=1)
    arch_df = arch_df.sort_values("spread", ascending=False)
    save_table(arch_df.round(4).rename_axis("protein_architecture").reset_index(), "4.4",
               "protein_architectures")
    top_arch = arch_df.head(15)
    short = [a if len(a) <= 64 else a[:63] + "…" for a in top_arch.index]
    fig, ax = plt.subplots(figsize=(1.3 * k_groups + 7.5, 0.4 * len(top_arch) + 2.0))
    heatmap(ax, 100 * top_arch[group_names].to_numpy(), short, group_names, vmin=0, vmax=100,
            annot=np.vectorize(lambda v: f"{v:.0f}")(100 * top_arch[group_names].to_numpy()),
            cbar_label="% of the group's BGCs")
    ax.tick_params(axis="y", rotation=0, labelsize=9)
    ax.tick_params(axis="x", rotation=0)
    titles(ax, "Protein architectures that change most between groups",
           "Domains of one protein in N→C order · number = % of the group's BGCs carrying that protein")
    show_and_save(fig, "4.4", "c_protein_architectures")
    display(summary)

## 4.5 Siderophore BGC analysis

This section isolates the BGCs of the siderophore classes (`CONFIG["siderophore_classes"]`: `nrp-metallophore`, siderophores assembled by NRPS, and `ni-siderophore`, NRPS-independent siderophores made by IucA/IucC-family enzymes). The classes in `CONFIG["siderophore_compare_classes"]` (`nrps`) are included for comparison, because section 1.3 shows that many `nrps` BGCs lie within the coordinates of an `nrp-metallophore` BGC of the same genome.

1. **Table:** which group from 4.1 each class falls into.
2. **4.5a:** percentage of each class's BGCs carrying each domain from a list of siderophore biosynthesis and uptake domains (`SID_DOMAINS`, editable; the description is the Pfam family name).
3. **4.5b:** number of NRPS modules per BGC.
4. **4.5c:** architecture (panels A and B, as in 4.3) of the representative of each siderophore class in each group with at least `sid_min_bgcs` BGCs.

In [ ]:
# Pfam domains of siderophore biosynthesis and uptake (exact name → Pfam family name)
SID_DOMAINS = {
    "IucA_IucC": "IucA/IucC family (NIS synthetase)",
    "Lys_Orn_oxgnase": "L-lysine 6-monooxygenase",
    "Acetyltransf_8": "acetyltransferase (GNAT)",
    "FhuF": "FhuF-like ferric iron reductase",
    "Condensation": "condensation domain (C)",
    "AMP-binding": "AMP-binding enzyme (A)",
    "PP-binding": "phosphopantetheine attachment site (T)",
    "Thioesterase": "thioesterase domain (TE)",
    "MbtH": "MbtH-like protein",
    "Chorismate_bind": "chorismate binding enzyme",
    "Isochorismatase": "isochorismatase family",
    "TauD": "TauD/TfdA dioxygenase",
    "TonB_dep_Rec_b-barrel": "TonB-dependent receptor (β-barrel)",
    "Plug": "TonB-dependent receptor plug domain",
    "FecCD": "FecCD transport family",
    "Peripla_BP_2": "periplasmic binding protein",
}
sid_classes = [c for c in class_order if c in CONFIG["siderophore_classes"]]
cmp_classes = [c for c in class_order if c in CONFIG["siderophore_compare_classes"]]
show_classes = sid_classes + cmp_classes
sid_mask = np.isin(y, sid_classes)

if not ARCH_OK or not sid_mask.any():
    print("No BGCs of the siderophore classes or the Pfam map column is missing: 4.5 is skipped.")
else:
    print(f"Siderophore BGCs: {sid_mask.sum()} of {len(y)} (" + ", ".join(f"{c} {int((y == c).sum())}" for c in sid_classes) + ")")
    for c in cmp_classes:                                        # overlap from section 1.3
        inside = {i for i in np.where(y == c)[0] if any(o in sid_classes for o in overlap_classes.get(i, ()))}
        print(f"BGCs {c} overlapping a siderophore BGC of the same genome: {len(inside)} of {int((y == c).sum())}")

    # ---- Table: classes × groups -------------------------------------------------------------
    sid_groups = (pd.crosstab(pd.Series(y[np.isin(y, show_classes)], name="class"),
                              pd.Series(groups[np.isin(y, show_classes)], name="group"))
                  .reindex(index=show_classes, columns=group_names, fill_value=0))
    save_table(sid_groups.reset_index(), "4.5", "siderophores_by_group")
    display(sid_groups)

    # ---- 4.5a Siderophore domains by class ------------------------------------------------
    present = [d for d in SID_DOMAINS if d in domain_names]
    prev_sid = pd.DataFrame({c: [100 * P[y == c, domain_names.index(d)].mean() for d in present] for c in show_classes},
                            index=present)
    prev_sid = prev_sid.loc[prev_sid.max(axis=1) > 0]
    save_table(prev_sid.round(1).rename_axis("domain").reset_index().assign(
        **{"pfam_family": [SID_DOMAINS[d] for d in prev_sid.index]}), "4.5", "siderophore_domains")
    col_lab = [f"{c}\n(n = {int((y == c).sum())})" + ("\ncomparison" if c in cmp_classes else "") for c in show_classes]
    fig, ax = plt.subplots(figsize=(1.6 * len(show_classes) + 7.5, 0.42 * len(prev_sid) + 2.4))
    heatmap(ax, prev_sid.to_numpy(), [f"{d} — {SID_DOMAINS[d]}" for d in prev_sid.index], col_lab, vmin=0, vmax=100,
            annot=np.where(prev_sid.to_numpy() > 0, np.vectorize(lambda v: f"{v:.0f}")(prev_sid.to_numpy()), ""),
            cbar_label="% of the class's BGCs")
    ax.tick_params(axis="y", rotation=0, labelsize=9.5)
    ax.tick_params(axis="x", rotation=0)
    titles(ax, "Siderophore biosynthesis and uptake domains",
           "Number = % of the class's BGCs with the domain · empty = 0")
    show_and_save(fig, "4.5", "a_siderophore_domains")

    # ---- 4.5b NRPS modules per BGC --------------------------------------------------------------
    rng = np.random.default_rng(RS)
    fig, ax = plt.subplots(figsize=(9, 0.75 * len(show_classes) + 1.9))
    for r, c in enumerate(show_classes):
        v = per_bgc.loc[y == c, "modules"].to_numpy()
        st = CLASS_STYLE[c]
        ax.scatter(v + rng.uniform(-0.12, 0.12, len(v)), r + rng.uniform(-0.22, 0.22, len(v)), s=28 * (1.5 if st["hollow"] else 1),
                   marker=st["marker"], facecolors="none" if st["hollow"] else st["color"],
                   edgecolors=st["color"] if st["hollow"] else SURFACE, linewidths=1.2 if st["hollow"] else 0.5, alpha=0.85)
        ax.plot([np.median(v)] * 2, [r - 0.32, r + 0.32], color=INK, lw=2.2)
        ax.text(v.max() + 0.45, r, f"median = {np.median(v):g}", va="center", fontsize=9.5, color=INK2)
    ax.set_yticks(range(len(show_classes)))
    ax.set_yticklabels([c + (" (comparison)" if c in cmp_classes else "") for c in show_classes])
    ax.set_ylim(len(show_classes) - 0.5, -0.5)
    ax.set_xlim(-0.6, per_bgc.loc[np.isin(y, show_classes), "modules"].max() + 2.2)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("NRPS/PKS modules per BGC (A + T or KS + T)")
    titles(ax, "Assembly modules in siderophore BGCs",
           "Each dot is a BGC (slightly jittered so they do not overlap) · black line = median")
    show_and_save(fig, "4.5", "b_siderophore_modules")

    sid_summary = (per_bgc[np.isin(y, show_classes)].groupby(["class", "group"])
                   .agg(n_BGC=("kb", "size"), modules_median=("modules", "median"),
                        modules_max=("modules", "max"), genes_median=("genes", "median"),
                        kb_median=("kb", "median")).reset_index())
    save_table(sid_summary, "4.5", "siderophore_summary")

    # ---- 4.5c Architecture of the representative of each siderophore class in each group --
    reps_sid = []
    for c in sid_classes:
        for g in group_names:
            idx = np.where((y == c) & (groups == g) & has_arch)[0]
            if len(idx) >= CONFIG["sid_min_bgcs"]:
                reps_sid.append((c, g, medoid(idx)[0], len(idx)))
    if reps_sid:
        kb_sid = max(sum(gn["length_aa"] * 3e-3 for gn in all_arch[i]) + GAP_KB * len(all_arch[i]) for _, _, i, _ in reps_sid)
        plot_architectures([(f"{c} · {g} (n = {n}) · representative: {bgc_label(i)} · {len(all_arch[i])} genes",
                             i, c, markers_by_group[g], f"{slug(c)}_{g}") for c, g, i, n in reps_sid], kb_sid, "4.5",
                           "c_siderophore_architecture", "c_architecture")
    sid_summary

## 4.6 Sequence alignment distances

So far BGCs have been compared by their Pfam domains. This section compares them by **protein sequence**, to see whether the classes also separate at the sequence level.

1. A stratified subsample of `seq_per_class` BGCs per class is taken (the number of alignments grows with n²) and, from each BGC, its `seq_max_proteins_per_bgc` longest proteins.
2. Every pair of proteins is aligned with **local Smith–Waterman** (BLOSUM62, gap open −11, extension −1; Biopython). The score is normalised: s(a, b) = S(a, b) / √(S(a, a) · S(b, b)), between 0 and 1.
3. Similarity between two BGCs = mean of the best protein matches in both directions; **distance = 1 − similarity**.

- **4.6a:** distance matrix ordered by class. Dark blocks on the diagonal = BGCs of the same class with similar proteins.
- **4.6b:** mean distance between and within classes.
- **4.6c:** relationship between sequence distance and model-domain distance (Jaccard on the *k* domains). The **Mantel test** assesses whether the two matrices are correlated.

This is the slowest section of the notebook (a few minutes).

In [ ]:
from joblib import Parallel, delayed

BIO_OK = importlib.util.find_spec("Bio") is not None
SEQ_OK = BIO_OK and seq_col in data.columns
AA_VALID = set("ACDEFGHIKLMNPQRSTVWYBZX")


def parse_proteins(cell):
    """BGC proteins separated by "|" (also accepts FASTA or ; , spaces); U, O, J → X."""
    if pd.isna(cell):
        return []
    text = str(cell)
    chunks = (["".join(r.splitlines()[1:]) for r in text.split(">") if r.strip()] if ">" in text
              else re.split(r"[|;,\s]+", text))
    prots = []
    for ch in chunks:
        ch = "".join(c if c in AA_VALID else "X" for c in re.split(r"[:=]", ch)[-1].upper() if c.isalpha())
        if len(ch) >= CONFIG["seq_min_length"]:
            prots.append(ch)
    return prots


def make_aligner():
    """Local Smith–Waterman with BLOSUM62 (Biopython's C implementation)."""
    from Bio.Align import PairwiseAligner, substitution_matrices
    al = PairwiseAligner()
    al.mode = "local"
    al.substitution_matrix = substitution_matrices.load("BLOSUM62")
    al.open_gap_score, al.extend_gap_score = -11, -1
    return al


def row_similarities(i, prots, self_sc):
    """Similarity of BGC i with every BGC j > i (rows are spread across cores)."""
    al = make_aligner()
    out = []
    for j in range(i + 1, len(prots)):
        S = np.array([[al.score(a, b) for b in prots[j]] for a in prots[i]])
        N = np.clip(S / np.sqrt(np.outer(self_sc[i], self_sc[j])), 0, 1)
        out.append((j, 0.5 * (N.max(axis=1).mean() + N.max(axis=0).mean())))
    return i, out


if not SEQ_OK:
    print("Biopython or the sequence column is missing: 4.6 is skipped (run cell 0.1).")
else:
    all_prots = data[seq_col].map(parse_proteins)
    n_prot = all_prots.map(len).to_numpy()
    rng = np.random.default_rng(RS)
    seq_rows = np.concatenate([np.sort(rng.choice(np.where((y == c) & (n_prot > 0))[0],
                                                  min(CONFIG["seq_per_class"], int(((y == c) & (n_prot > 0)).sum())),
                                                  replace=False)) for c in class_order])
    seq_y = y[seq_rows]
    prots = [sorted(all_prots.iloc[r], key=len, reverse=True)[:CONFIG["seq_max_proteins_per_bgc"]] for r in seq_rows]
    al0 = make_aligner()
    self_sc = [np.array([al0.score(p, p) for p in pp]) for pp in prots]     # self-scores for normalising
    n_seq = len(seq_rows)
    n_aln = sum(len(prots[i]) * len(prots[j]) for i in range(n_seq) for j in range(i + 1, n_seq))

    t0 = time.time()
    sim = np.eye(n_seq)
    for i, row in Parallel(n_jobs=CONFIG["n_jobs"])(delayed(row_similarities)(i, prots, self_sc) for i in range(n_seq)):
        for j, s in row:
            sim[i, j] = sim[j, i] = s
    D_seq = 1 - sim
    seq_ids = [f"{bgc_label(r)} ({c})" for r, c in zip(seq_rows, seq_y)]
    save_table(pd.DataFrame(D_seq.round(4), columns=seq_ids).assign(BGC=seq_ids).set_index("BGC").reset_index(),
               "4.6", "alignment_distance_matrix")
    print(f"{n_seq} BGCs ({CONFIG['seq_per_class']} per class) · {n_aln:,} alignments in {time.time() - t0:.0f} s")

**Figures 4.6a – 4.6c.** The previous cell computes the alignment distance matrix; this one draws the three figures described above and runs the Mantel test.

In [ ]:
if SEQ_OK:
    # ---- Model-domain distance (Jaccard on presence of the k domains) --
    D_pfam = np.nan_to_num(squareform(pdist(P[seq_rows][:, sel_idx], metric="jaccard")))
    present = [c for c in class_order if c in set(seq_y)]
    bounds = np.cumsum([0] + [int((seq_y == c).sum()) for c in present])

    # ---- 4.6a Matrix ordered by class -------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9.5, 8.3))
    im = ax.imshow(D_seq, cmap=SEQ_CMAP.reversed(), vmin=0, vmax=1, interpolation="nearest")
    for b in bounds[1:-1]:                                       # lines between classes
        ax.axhline(b - 0.5, color=SURFACE, lw=2)
        ax.axvline(b - 0.5, color=SURFACE, lw=2)
    centres = (bounds[:-1] + bounds[1:]) / 2 - 0.5
    ax.set_xticks(centres)
    ax.set_xticklabels(present)
    rotate_xlabels(ax)
    ax.set_yticks(centres)
    ax.set_yticklabels(present)
    ax.tick_params(length=0)
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.75, label="Alignment distance (1 − similarity)")
    titles(ax, "Alignment distances between BGCs, ordered by class",
           f"{n_seq} BGCs · dark = more similar proteins · each block is a class")
    show_and_save(fig, "4.6", "a_alignment_matrix")

    # ---- 4.6b Mean distance between and within classes -----------------------------------
    class_mean = pd.DataFrame(index=present, columns=present, dtype=float)
    for a in present:
        for b in present:
            block = D_seq[np.ix_(seq_y == a, seq_y == b)]
            if a == b:
                block = block[~np.eye(len(block), dtype=bool)]       # without the diagonal (zero self-distance)
            class_mean.loc[a, b] = block.mean() if block.size else np.nan
    save_table(class_mean.round(4).rename_axis("class").reset_index(), "4.6", "class_mean_distance")
    fig, ax = plt.subplots(figsize=(9, 7.6))
    heatmap(ax, class_mean.to_numpy(float), present, present, cmap=SEQ_CMAP.reversed(), vmin=0, vmax=1,
            cbar_label="Mean distance", annot_size=9)
    rotate_xlabels(ax)
    ax.tick_params(axis="y", rotation=0)
    intra = np.nanmean(np.diag(class_mean.to_numpy(float)))
    inter = np.nanmean(class_mean.to_numpy(float)[~np.eye(len(present), dtype=bool)])
    titles(ax, "Mean alignment distance between and within classes",
           f"Diagonal = within the class · mean within = {intra:.2f} · mean between classes = {inter:.2f}")
    show_and_save(fig, "4.6", "b_class_mean_distance")

    # ---- 4.6c Sequence versus domains: Mantel test ------------------------------------
    iu = np.triu_indices(n_seq, 1)
    r_obs = spearmanr(D_seq[iu], D_pfam[iu])[0]
    perm_r = []
    for _ in range(CONFIG["mantel_permutations"]):               # permutes rows and columns together
        p = rng.permutation(n_seq)
        perm_r.append(spearmanr(D_seq[iu], D_pfam[np.ix_(p, p)][iu])[0])
    p_mantel = (1 + np.sum(np.array(perm_r) >= r_obs)) / (1 + len(perm_r))
    same = (seq_y[iu[0]] == seq_y[iu[1]])
    save_table(pd.DataFrame({"pfam_distance": D_pfam[iu].round(4), "sequence_distance": D_seq[iu].round(4),
                             "same_class": same}), "4.6", "sequence_vs_domains")
    fig, ax = plt.subplots(figsize=(8.5, 6.4))
    ax.scatter(D_pfam[iu][~same], D_seq[iu][~same], s=10, c=CONTEXT, linewidths=0, label="pairs from different classes")
    ax.scatter(D_pfam[iu][same], D_seq[iu][same], s=14, c=SERIES[0], linewidths=0, label="pairs from the same class")
    ax.set_xlabel(f"Model-domain distance (Jaccard on {k_optimal} domains)")
    ax.set_ylabel("Alignment distance")
    ax.set_xlim(-0.03, 1.03)
    ax.set_ylim(-0.03, 1.03)
    ax.legend(loc="lower right")
    titles(ax, "Do BGCs with similar domains also have similar proteins?",
           f"Mantel test (Spearman): r = {r_obs:.2f} · p = {p_mantel:.3f} ({CONFIG['mantel_permutations']} permutations)")
    show_and_save(fig, "4.6", "c_sequence_vs_domains")
    print(f"Mean within-class distance = {intra:.3f} · between classes = {inter:.3f} · "
          f"Mantel r = {r_obs:.3f}, p = {p_mantel:.4f}")

---
# Part 5 · Cluster comparison: shared domains, repeats and synteny

**Question:** which domains and genes recur across clusters and within them, and how conserved is gene order (synteny)?

| Section | What it does | Main figures |
|---|---|---|
| 5.1 | Heatmap of all clusters against the characteristic domains of each class | `5.1_cluster_domain_heatmap` |
| 5.2 | Domains that recur across clusters: in how many BGCs and classes each domain appears | `5.2_domains_recurring_across_clusters` |
| 5.3 | Genes repeated within the same cluster (5.3a) and domains repeated within the same gene (5.3b) | `5.3_a_repeated_genes`, `5.3_b_domains_repeated_in_a_gene` |
| 5.4 | Relationship between syntenies: how similar gene order is between clusters | `5.4_a_synteny_matrix`, `5.4_b_synteny_per_class` |
| 5.5 | All clusters aligned in order, locus by locus | `5.5_alignment_<class>_locus<n>` |

**Where each result comes from.** Sections 5.1 and 5.2 use the table's `PF…` columns, which record every domain of each BGC. Sections 5.3 to 5.5 work gene by gene: they use `protein_sequences` to decide when two genes are the same gene and `protein_seq_pfam_map` to know which domains each gene has. As section 4.3 shows, that map does not contain every domain in the table, so gene names only include the domains that are in the map.

**Assumptions.** The table provides neither the coordinates nor the strand of each gene. Sections 5.4 and 5.5 assume that the genes of each BGC are listed in their order in the genome. In addition, the gene list of a BGC can extend beyond the `location` coordinates (section 5.4 quantifies this).

## 5.1 Heatmap of clusters against domains

Each row is a BGC and each column a Pfam domain. Showing every domain would make the figure unreadable, so the **most characteristic domains of each class** are shown (up to `heatmap_domains_per_class` per class): those present in at least half of the class's BGCs whose frequency in the class most exceeds their frequency in the rest. Each domain is assigned to a single class, the one where that difference is largest.

**How to read it:** rows are grouped by class and columns by the class they characterise, so a well-defined class shows up as a dark block on the diagonal. A dark block off the diagonal means two classes share those domains. Colour is the domain's score in the table (0 = absent); because the scores of the domains that are present are all high, the map reads in practice as presence or absence.

In [ ]:
from collections import Counter
from itertools import combinations
from scipy.cluster.hierarchy import fcluster, leaves_list, linkage
from scipy.spatial.distance import pdist, squareform

N_CHAR = CONFIG["heatmap_domains_per_class"]
prev_class = np.array([P[y == c].mean(axis=0) for c in class_order])                    # classes × domains: fraction of BGCs with the domain
diff_class = prev_class - np.array([P[y != c].mean(axis=0) for c in class_order])       # frequency in the class − in the rest
owner = diff_class.argmax(axis=0)                                # class each domain characterises most
blocks = []
for k in range(len(class_order)):
    mine = np.where((owner == k) & (prev_class[k] >= 0.5))[0]    # the class's own domains, present in at least half of its BGCs
    blocks.append(mine[np.argsort(-diff_class[k, mine], kind="stable")][:N_CHAR])
heat_cols = np.concatenate(blocks)

heat_rows = []                                                   # BGCs by class; within each class, similar profiles together
for c in class_order:
    idx = np.where(y == c)[0]
    if len(idx) > 2:
        idx = idx[leaves_list(linkage(np.nan_to_num(pdist(P[idx][:, heat_cols], "jaccard"), nan=1.0), "average"))]
    heat_rows += list(idx)
heat_rows = np.array(heat_rows)

save_table(pd.DataFrame({
    "class": np.repeat(class_order, [len(b) for b in blocks]),
    "domain": [domain_names[j] for j in heat_cols],
    "pct_in_class": [round(100 * prev_class[k, j], 1) for k, b in enumerate(blocks) for j in b],
    "pct_in_rest": [round(100 * (prev_class[k, j] - diff_class[k, j]), 1) for k, b in enumerate(blocks) for j in b]}),
    "5.1", "characteristic_domains")

row_bounds = np.cumsum([0] + [int((y == c).sum()) for c in class_order])
col_bounds = np.cumsum([0] + [len(b) for b in blocks])
fig, ax = plt.subplots(figsize=(0.21 * len(heat_cols) + 4.4, 9.8))
im = ax.imshow(X[np.ix_(heat_rows, heat_cols)], cmap=SEQ_CMAP, vmin=0, vmax=1, aspect="auto", interpolation="nearest")
for b in row_bounds[1:-1]:
    ax.axhline(b - 0.5, color=INK, lw=0.8)
for b in col_bounds[1:-1]:
    ax.axvline(b - 0.5, color=INK, lw=0.8)
ax.set_yticks((row_bounds[:-1] + row_bounds[1:]) / 2 - 0.5)
ax.set_yticklabels([f"{c} ({counts[c]})" for c in class_order])
ax.set_xticks(range(len(heat_cols)))
ax.set_xticklabels([domain_names[j] for j in heat_cols], rotation=90, fontsize=8.5)
ax.tick_params(length=0)
ax.grid(False)
for k, c in enumerate(class_order):                              # class each column block characterises (on two lines if the block is narrow)
    if len(blocks[k]):
        ax.text((col_bounds[k] + col_bounds[k + 1]) / 2 - 0.5, 1.006, c if len(blocks[k]) >= 6 else c.replace("-", "-\n"),
                transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=9, fontweight="bold", color=INK, linespacing=1.0)
fig.colorbar(im, ax=ax, shrink=0.5, pad=0.015, label="Domain score in the table (0 = absent)")
ax.set_title("Heatmap: clusters × characteristic domains of each class", loc="left", pad=58)
ax.text(0, 1.058, f"Each row is a BGC ({len(heat_rows)}) · above, the class characterised by each block of domains · "
        f"score of the domains that are present: {X[X > 0].min():.2f}–{X.max():.2f}",
        transform=ax.transAxes, ha="left", va="bottom", fontsize=10, color=INK2)
show_and_save(fig, "5.1", "cluster_domain_heatmap")

## 5.2 Domains that recur across clusters

For each Pfam domain, the number of BGCs carrying it and its spread across classes are counted:

| Column | What it measures |
|---|---|
| `n_BGC` / `pct_BGC` | BGCs carrying the domain |
| `classes_with_the_domain` | Classes in which at least one BGC carries the domain |
| `classes_where_frequent` | Classes in which the domain is in at least half of the BGCs |
| `pct_<class>` | Percentage of that class's BGCs carrying the domain |

**How to read the figure:** each row is one of the most recurrent domains; the first column is the percentage over all BGCs and the others the percentage within each class. A row that is dark in several columns is a domain shared by several classes; a row that is dark in a single column is a very recurrent domain that is specific to that class. Several domains occur almost always in the same BGCs, so their rows are almost identical.

In [ ]:
recurrence = pd.DataFrame({"domain": domain_names, "n_BGC": P.sum(axis=0), "pct_BGC": (100 * P.mean(axis=0)).round(1),
                           "classes_with_the_domain": (prev_class > 0).sum(axis=0),
                           "classes_where_frequent": (prev_class >= 0.5).sum(axis=0)})
for k, c in enumerate(class_order):
    recurrence[f"pct_{c}"] = (100 * prev_class[k]).round(1)
recurrence = save_table(recurrence.sort_values("n_BGC", ascending=False, kind="stable").reset_index(drop=True),
                        "5.2", "domains_recurring_across_clusters")

top = recurrence.head(CONFIG["recurrent_top_domains"])
M = top[["pct_BGC"] + [f"pct_{c}" for c in class_order]].to_numpy()
fig, ax = plt.subplots(figsize=(0.95 * (len(class_order) + 1) + 3.4, 0.33 * len(top) + 2.6))
heatmap(ax, M, top["domain"].tolist(), [f"All\n({len(y)})"] + [f"{c}\n({counts[c]})" for c in class_order],
        vmin=0, vmax=100, annot=np.where(M >= 0.5, np.vectorize(lambda v: f"{v:.0f}")(M), ""), cbar_label="% of BGCs with the domain",
        annot_size=8.5)
ax.axvline(1, color=INK, lw=1.6)                                 # separates "All" from the classes
ax.tick_params(axis="y", rotation=0, labelsize=9.5)
ax.tick_params(axis="x", labelsize=8.5)
rotate_xlabels(ax)
titles(ax, f"The {len(top)} domains that recur most across clusters",
       "Number = % of BGCs with the domain (first column: all BGCs; others: per class) · empty = below 0.5 %")
show_and_save(fig, "5.2", "domains_recurring_across_clusters")

FREQ = "classes_where_frequent"
in_all = recurrence[recurrence["classes_with_the_domain"] == len(class_order)]
shared = recurrence[recurrence[FREQ] >= 2]
widest = recurrence[recurrence[FREQ] == recurrence[FREQ].max()]
print(f"Domains frequent (in ≥ 50 % of the BGCs) in 2 or more classes: {len(shared)} of {len(recurrence)}")
print(f"No domain is frequent in more than {recurrence[FREQ].max()} classes ({', '.join(widest['domain'].head(6))})")
print(f"Domains present in at least one BGC of each of the {len(class_order)} classes: {len(in_all)}"
      + (" (" + ", ".join(in_all["domain"].head(8)) + ("…" if len(in_all) > 8 else "") + ")" if len(in_all) else ""))
recurrence.iloc[:, :5].head(10)                                   # the per-class % are in the figure and in the CSV

## 5.3 Genes and domains repeated within a cluster

**When are two genes the same gene?** The table has no gene names, so genes are grouped into **families** by the similarity of their proteins: from longest to shortest, each protein joins the family whose representative it resembles with a similarity of at least `gene_family_min_similarity`; if it resembles no representative, it founds a new family. The **similarity** between two proteins is the score of their local alignment (BLOSUM62) divided by the larger of the scores each protein obtains when aligned with itself (normally that of the longer one): it is 1 if they are identical and drops if the sequences differ or if one is much shorter than the other. Each family is named after its most frequent **domain architecture** (the ordered list of the gene's Pfam domains according to the map). Two genes with the same domains but different sequences therefore end up in different families.

With that definition, two kinds of repetition within a single BGC are measured:

- **5.3a · Genes repeated within the same cluster:** a family that appears twice or more in the BGC. The right-hand panel of the figure shows the sequence similarity between the copies.
- **5.3b · Domains repeated within the same gene:** a domain that appears in two or more separate regions of the same protein (for example the `Condensation` domain of a multi-module NRPS).

**Limitations**

- The similarity threshold is a parameter: a lower value joins more distant genes and increases the repeated genes; a higher one decreases them. For reference, the cell also prints how many BGCs have two or more genes with the same domain architecture without requiring sequence similarity.
- To avoid aligning each protein with every representative, only the (up to three) representatives that share most four-amino-acid fragments with it are tried. A pair whose similarity is only just above the threshold can therefore end up in different families.
- In 5.3b, two identical consecutive tags (with no residues between them) count once, because the table cannot tell whether they are two tandem copies or one domain detected in two parts. The count is therefore a minimum; the summary table also gives the count when every tag is counted (`n_BGC_counting_consecutive_tags`).

The first cell groups the proteins into families (it takes about half a minute).

In [ ]:
def gene_instances(gene):
    """Gene domains in
    N→C order, one entry per region (two identical consecutive tags count once)."""
    seen, out = set(), []
    for name, _, start, _ in sorted(gene["domains"], key=lambda d: d[2]):
        if (name, start) not in seen:
            seen.add((name, start))
            out.append(name)
    return out


def gene_architecture(gene):
    """Domain
    architecture: ordered list of the gene's Pfam domains ("" if it has none in the map)."""
    return " · ".join(gene_instances(gene))


def short_arch(text, n=44):
    """Long architectures: first and last domain and how many there are."""
    parts = text.split(" · ")
    return text if len(text) <= n or len(parts) < 3 else f"{parts[0]} · … · {parts[-1]} ({len(parts)} domains)"


def gene_label(name):
    """Short label: first domain of the gene and how many more it has."""
    parts = name.split(" · ")
    label = parts[0] if len(parts) == 1 else f"{parts[0]} +{len(parts) - 1}"
    return label if len(label) <= 20 else label[:19] + "…"


COMPARE_OK = bool(globals().get("ARCH_OK")) and bool(globals().get("SEQ_OK"))
if not COMPARE_OK:
    print("The Pfam map, the sequences or Biopython are missing: sections 5.3 – 5.5 are skipped (run cell 0.1).")
else:
    aligner = make_aligner()
    DIAGONAL = {a: float(aligner.substitution_matrix[a, a]) for a in aligner.substitution_matrix.alphabet}

    def self_score(seq):
        """Score of a
        protein aligned with itself: sum of the BLOSUM62 diagonal (best contiguous stretch)."""
        total = np.concatenate([[0.0], np.cumsum([DIAGONAL[ch] for ch in seq])])
        return float((total - np.minimum.accumulate(total)).max())

    def similarity(a, b):
        """Similarity between two proteins: S(a, b) / max(S(a, a), S(b, b)). It is 1
        if they are identical and low if the sequences differ or if one is much shorter than the other."""
        return aligner.score(a, b) / max(self_score(a), self_score(b))

    def protein_families(proteins, min_sim, k=4, n_candidates=3, min_shared=0.10):
        """Groups
        proteins into families (greedy clustering, longest first). Each protein is compared with the representative of the families
        it shares most k-mers with (up to `n_candidates`, and at least `min_shared` of its own) and joins the first one whose
        similarity is ≥ min_sim; if there is none, it founds a family and becomes its representative."""
        family = np.full(len(proteins), -1)
        seen, reps, rep_self, index = {}, [], [], {}             # index: k-mer → families whose representative contains it
        for i in sorted(range(len(proteins)), key=lambda j: (-len(proteins[j]), j)):
            seq = proteins[i]
            if seq and seq in seen:                              # protein identical to one already assigned
                family[i] = seen[seq]
                continue
            kmers = {seq[j:j + k] for j in range(len(seq) - k + 1)}
            shared = Counter(f for km in kmers for f in index.get(km, ()))
            own = self_score(seq) if seq else 0.0
            for f, n in sorted(shared.items(), key=lambda kv: (-kv[1], kv[0]))[:n_candidates]:       # fixed order: same result on every run
                if n < min_shared * len(kmers) or own <= 0:
                    break
                if aligner.score(seq, reps[f]) / max(own, rep_self[f]) >= min_sim:
                    family[i] = f
                    break
            if family[i] < 0:                                    # new family
                family[i] = len(reps)
                reps.append(seq)
                rep_self.append(own)
                for km in kmers:
                    index.setdefault(km, []).append(family[i])
            seen[seq] = family[i]
        return family

    t0 = time.time()
    architectures = [[gene_architecture(g) for g in genes] for genes in all_arch]         # architectures[BGC][gene]
    seq_lists = [str(s).split("|") if not pd.isna(s) else [] for s in seq_src]
    proteins = [["".join(ch if ch in AA_OK else "X" for ch in seq_lists[i][g].upper()) if g < len(seq_lists[i]) else ""
                 for g in range(len(genes))] for i, genes in enumerate(all_arch)]         # proteins[BGC][gene]
    gene_index = [(i, g) for i, genes in enumerate(all_arch) for g in range(len(genes))]
    raw_family = protein_families([proteins[i][g] for i, g in gene_index], CONFIG["gene_family_min_similarity"])

    # Families numbered from largest to smallest (1 = the one with most genes) and named after their most frequent architecture
    by_size = sorted(Counter(raw_family.tolist()).items(), key=lambda kv: (-kv[1], kv[0]))
    number = {f: r for r, (f, _) in enumerate(by_size, start=1)}
    gene_fam = [[] for _ in all_arch]                            # gene_fam[BGC][gene] = family number
    for (i, g), f in zip(gene_index, raw_family.tolist()):
        gene_fam[i].append(number[f])
    members = {}
    for i, fams in enumerate(gene_fam):
        for g, f in enumerate(fams):
            members.setdefault(f, []).append((i, g))
    NO_DOMAIN = "no Pfam domain"
    fam_name = {f: Counter(architectures[i][g] for i, g in genes).most_common(1)[0][0] or NO_DOMAIN for f, genes in members.items()}

    gene_families = save_table(pd.DataFrame([{
        "family": f, "name": fam_name[f], "genes": len(genes), "BGCs": len({i for i, _ in genes}),
        "classes": len({y[i] for i, _ in genes}), "median_length_aa": int(np.median([all_arch[i][g]["length_aa"] for i, g in genes])),
        "distinct_architectures": len({architectures[i][g] for i, g in genes})} for f, genes in sorted(members.items())]),
        "5.3", "gene_families")
    save_table(pd.DataFrame([{"data_row": i, "BGC": bgc_label(i), "class": y[i], "gene": g + 1,
                              "length_aa": all_arch[i][g]["length_aa"], "family": gene_fam[i][g],
                              "architecture": architectures[i][g]} for i, g in gene_index]), "5.3", "genes")
    print(f"{len(gene_index):,} genes ({len({proteins[i][g] for i, g in gene_index}):,} distinct proteins) → {len(members):,} gene families "
          f"(similarity ≥ {CONFIG['gene_family_min_similarity']}) · {int((gene_families['BGCs'] >= 2).sum()):,} present in 2 or more BGCs · "
          f"{sum(a == '' for archs in architectures for a in archs):,} genes with no domain in the map · {time.time() - t0:.0f} s")

**Tables for 5.3a and 5.3b.** With the families defined, this cell looks for families with two or more genes in the same BGC (5.3a) and for domains that appear in two or more separate regions of the same gene (5.3b), saves the tables and summarises the results per class.

In [ ]:
if COMPARE_OK:
    # ---- 5.3a Genes repeated within the same cluster: a family with 2 or more genes in the BGC ----
    SIM = "similarity_between_copies"
    rows = []
    for i, fams in enumerate(gene_fam):
        for f, k in Counter(fams).items():
            if k >= 2:
                where = [g for g, x in enumerate(fams) if x == f]
                sims = [similarity(proteins[i][a], proteins[i][b]) for a, b in combinations(where, 2)]
                rows.append({"data_row": i, "BGC": bgc_label(i), "class": y[i], "family": f, "gene": fam_name[f],
                             "copies": k, "positions": ", ".join(str(g + 1) for g in where), SIM: round(float(np.median(sims)), 3)})
    repeated_genes = save_table(pd.DataFrame(rows), "5.3", "repeated_genes")
    gene_summary = (repeated_genes.groupby(["family", "gene"])
                    .agg(n_BGC=("data_row", "nunique"), max_copies=("copies", "max"),
                         median_similarity=(SIM, "median"))
                    .sort_values("n_BGC", ascending=False, kind="stable").reset_index())
    save_table(gene_summary.round(3), "5.3", "repeated_genes_summary")

    # ---- 5.3b Domains repeated within the same gene ----------------------------------------
    rows, with_runs = [], {}                                     # with_runs[domain] = BGCs with the domain repeated when every tag is counted
    for i, genes in enumerate(all_arch):
        for g, gene in enumerate(genes, start=1):
            for name, k in Counter(gene_instances(gene)).items():
                if k >= 2:
                    rows.append({"data_row": i, "BGC": bgc_label(i), "class": y[i], "gene": g, "domain": name,
                                 "copies_in_gene": k, "length_aa": gene["length_aa"],
                                 "gene_architecture": architectures[i][g - 1]})
            for name, k in Counter(d[0] for d in gene["domains"]).items():
                if k >= 2:
                    with_runs.setdefault(name, set()).add(i)
    repeated_domains = save_table(pd.DataFrame(rows), "5.3", "domains_repeated_in_a_gene")
    domain_summary = (repeated_domains.groupby("domain")
                      .agg(genes=("gene", "size"), n_BGC=("data_row", "nunique"),
                           max_copies=("copies_in_gene", "max"))
                      .sort_values("n_BGC", ascending=False, kind="stable").reset_index())
    domain_summary["n_BGC_counting_consecutive_tags"] = [len(with_runs[d]) for d in domain_summary["domain"]]
    save_table(domain_summary, "5.3", "domains_repeated_in_a_gene_summary")

    # ---- Summary ------------------------------------------------------------------------------------------------------
    n_bgc_rep = repeated_genes["data_row"].nunique()
    n_bgc_dom = repeated_domains["data_row"].nunique()
    n_bgc_dom_runs = len(set().union(*with_runs.values()))
    n_same_arch = sum(max(Counter(a for a in archs if a).values(), default=0) >= 2 for archs in architectures)     # same architecture, regardless of sequence
    near = repeated_genes[repeated_genes[SIM] >= 0.9]             # near-identical copies
    repeat_by_class = pd.DataFrame({
        "BGCs": [counts[c] for c in class_order],
        "pct_BGC_with_repeated_gene": [
            round(100 * repeated_genes[repeated_genes["class"] == c]["data_row"].nunique() / counts[c], 1) for c in class_order],
        "pct_BGC_with_domain_repeated_in_a_gene": [
            round(100 * repeated_domains[repeated_domains["class"] == c]["data_row"].nunique() / counts[c], 1) for c in class_order],
    }, index=pd.Index(class_order, name="class"))
    save_table(repeat_by_class.reset_index(), "5.3", "repeats_by_class")

    print(f"5.3a · BGCs with a repeated gene (same family): {n_bgc_rep} of {len(y)} · cases (BGC, family): {len(repeated_genes):,} · "
          f"distinct families: {repeated_genes['family'].nunique()}")
    print(f"       similarity between the copies: median {repeated_genes[SIM].median():.2f} · "
          f"cases with near-identical copies (≥ 0.9): {len(near)}"
          + (" (" + ", ".join(f"{g}: {k}" for g, k in near["gene"].value_counts().head(4).items()) + ")" if len(near) else ""))
    print(f"       for reference, BGCs with 2 or more genes sharing a domain architecture, regardless of sequence: {n_same_arch}")
    print(f"5.3b · BGCs with a domain repeated within one gene: {n_bgc_dom} of {len(y)} "
          f"({n_bgc_dom_runs} if every tag is counted)")
    display(repeat_by_class)

**Figures 5.3a and 5.3b.** The `repeat_top_items` families and domains repeated in the most BGCs are shown.

- **5.3a:** each row is a repeated gene family. The number is the percentage of the class's BGCs that carry it twice or more; on the right, each dot is the similarity between the copies in one BGC.
- **5.3b:** each row is a domain. The number is the percentage of the class's BGCs with a gene that carries it in two or more separate regions.

In [ ]:
if COMPARE_OK:
    K = CONFIG["repeat_top_items"]

    def pct_by_class(table, key, items):
        """% of each class's BGCs that appear in `table` for each item."""
        return np.array([[100 * table[(table[key] == item) & (table["class"] == c)]["data_row"].nunique() / counts[c]
                          for c in class_order] for item in items])

    # ---- 5.3a Figure: repeated families (left) and similarity between their copies (right) ----
    show_g = gene_summary.head(K)
    names = [short_arch(n) for n in show_g["gene"]]
    labels = [f"{n}  (family {f})" if names.count(n) > 1 else n for n, f in zip(names, show_g["family"])]     # the number tells apart families with the same name
    M = pct_by_class(repeated_genes, "family", show_g["family"])
    fig, (ax, ax_sim) = plt.subplots(1, 2, figsize=(0.9 * len(class_order) + 9.2, 0.42 * len(show_g) + 2.9),
                                     gridspec_kw={"width_ratios": [0.9 * len(class_order), 3.4], "wspace": 0.05})
    heatmap(ax, M, labels, class_order, vmin=0, vmax=100, annot=np.where(M >= 0.5, np.vectorize(lambda v: f"{v:.0f}")(M), ""), annot_size=9, cbar=False)
    ax.tick_params(axis="y", rotation=0, labelsize=9.5)
    rotate_xlabels(ax)
    hbox(ax_sim, [repeated_genes[repeated_genes["family"] == f][SIM].to_numpy() for f in show_g["family"]], [""] * len(show_g))
    ax_sim.set_xlim(0, 1)
    ax_sim.set_xticks([0, 0.25, 0.5, 0.75, 1])
    ax_sim.set_xticklabels(["0", "0.25", "0.5", "0.75", "1"])
    ax_sim.tick_params(axis="y", length=0)
    ax_sim.set_xlabel("Sequence similarity between the copies\n(1 = identical)")
    titles(ax, "Genes repeated within the same cluster",
           "Each row is a gene family · number = % of the class's BGCs with 2 or more copies · right: each dot is a BGC")
    show_and_save(fig, "5.3", "a_repeated_genes")

    # ---- 5.3b Figure: domains repeated within the same gene ----------------------------
    show_d = domain_summary.head(K)
    M = pct_by_class(repeated_domains, "domain", show_d["domain"])
    fig, ax = plt.subplots(figsize=(0.95 * len(class_order) + 6.2, 0.4 * len(show_d) + 2.8))
    heatmap(ax, M, [f"{d}  (up to {k} per gene)" for d, k in zip(show_d["domain"], show_d["max_copies"])],
            class_order, vmin=0, vmax=100, annot=np.where(M >= 0.5, np.vectorize(lambda v: f"{v:.0f}")(M), ""),
            cbar_label="% of the class's BGCs", annot_size=9)
    ax.tick_params(axis="y", rotation=0, labelsize=9.5)
    rotate_xlabels(ax)
    titles(ax, "Domains repeated within the same gene",
           "Number = % of the class's BGCs with a gene carrying the domain in 2 or more separate regions · empty = below 0.5 %")
    show_and_save(fig, "5.3", "b_domains_repeated_in_a_gene")
    display(gene_summary.head(10).round(3))

## 5.4 Relationship between syntenies

Synteny is the conservation of gene order. To measure it between two BGCs a **synteny index** is used: each BGC is walked in table order, every **pair of neighbouring genes** is recorded (each gene identified by its family, 5.3), and

> index = shared neighbour pairs / neighbour pairs present in either BGC

It equals 1 if both BGCs have exactly the same neighbouring genes and 0 if they share no pair. This is the formula of the BiG-SCAPE adjacency index (Navarro-Muñoz *et al.*, 2020), which is applied there to pairs of neighbouring domains and here to pairs of neighbouring genes. Pairs have no direction, so the index does not change if a cluster is inverted.

The index defines two levels within each class:

- **Locus:** a group of BGCs connected by shared neighbour pairs (two BGCs are in the same locus if they share a pair, directly or through other BGCs). BGCs of different loci share no pair of neighbouring genes. Locus 1 is the one with most BGCs.
- **Synteny type:** variants of gene order within a locus. They come from average-linkage hierarchical clustering: two groups of BGCs are joined while the mean index between them is at least `synteny_type_threshold`, so the number of types depends on that threshold. Type 1 is the most frequent in the class.

**Figures**

- **5.4a · Synteny matrix** between all BGCs, ordered by class. A dark block on the diagonal is a group of BGCs with the same gene order; a class with several separate blocks has several loci. A dark block off the diagonal is two classes that share order.
- **5.4b · Conservation within each class:** for each BGC, its mean index with the other BGCs of its class. A class with several loci has a low mean index even if each locus is well conserved; the table in the cell also gives the mean index within the same locus.

The cell also prints the index of the BGC pairs that overlap in the genome (section 1.3) and in how many BGCs the summed gene lengths exceed the length of the region.

In [ ]:
from scipy import sparse
from scipy.sparse.csgraph import connected_components

if COMPARE_OK:
    # ---- Synteny index between all BGCs -----------------------------------------------------
    pair_sets = [{tuple(sorted(p)) for p in zip(f[:-1], f[1:])} for f in gene_fam]       # pairs of neighbouring families, undirected
    vocab = {p: j for j, p in enumerate(sorted(set().union(*pair_sets)))}
    B = sparse.csr_matrix((np.ones(sum(len(ps) for ps in pair_sets)),
                           ([i for i, ps in enumerate(pair_sets) for _ in ps], [vocab[p] for ps in pair_sets for p in ps])),
                          shape=(len(pair_sets), len(vocab)))                 # BGCs × pairs: 1 if the BGC has that pair
    shared_pairs = (B @ B.T).toarray()
    n_pairs = shared_pairs.diagonal()
    union_pairs = n_pairs[:, None] + n_pairs[None, :] - shared_pairs
    SYN = np.divide(shared_pairs, union_pairs, out=np.zeros_like(shared_pairs), where=union_pairs > 0)

    # ---- Loci and synteny types within each class ---------------------------------------
    THR = CONFIG["synteny_type_threshold"]
    locus, syn_type, syn_pos, type_rows = np.zeros(len(y), int), np.zeros(len(y), int), np.zeros(len(y), int), []
    for c in class_order:
        idx = np.where(y == c)[0]
        sub = SYN[np.ix_(idx, idx)]
        comp = connected_components(sparse.csr_matrix(sub > 0), directed=False)[1]     # locus = BGCs connected by shared pairs
        order = {k: r for r, (k, _) in enumerate(sorted(Counter(comp.tolist()).items(), key=lambda kv: (-kv[1], kv[0])), start=1)}
        locus[idx] = [order[k] for k in comp.tolist()]
        D = 1 - sub
        np.fill_diagonal(D, 0)
        Z = linkage(squareform(D, checks=False), "average")
        lab = fcluster(Z, 1 - THR, "distance")                   # groups are joined while their mean index is ≥ THR
        by_size = sorted(Counter(lab.tolist()).items(), key=lambda kv: (-kv[1], kv[0]))  # type 1 = the most frequent
        rank = {t: r for r, (t, _) in enumerate(by_size, start=1)}
        syn_type[idx] = [rank[t] for t in lab.tolist()]
        syn_pos[idx[leaves_list(Z)]] = np.arange(len(idx))                             # tree order: similar neighbours
        for t, n in by_size:
            in_type = idx[lab == t]
            rep = in_type[np.argmax(SYN[np.ix_(in_type, in_type)].mean(axis=1))]        # BGC most similar to the rest of its type
            type_rows.append({"class": c, "locus": locus[rep], "type": rank[t], "n_BGC": n, "pct_of_class": round(100 * n / len(idx), 1),
                              "representative": bgc_label(rep), "representative_row": rep})
    synteny_types = save_table(pd.DataFrame(type_rows), "5.4", "synteny_types")

    same_class = y[:, None] == y[None, :]
    same_locus = same_class & (locus[:, None] == locus[None, :])
    not_self = ~np.eye(len(y), dtype=bool)
    with np.errstate(invalid="ignore", divide="ignore"):         # a BGC alone in its locus has no within-locus mean
        mean_in_class = (SYN * (same_class & not_self)).sum(axis=1) / (same_class & not_self).sum(axis=1)
        mean_in_locus = (SYN * (same_locus & not_self)).sum(axis=1) / (same_locus & not_self).sum(axis=1)
    save_table(pd.DataFrame({"data_row": np.arange(len(y)), "BGC": [bgc_label(i) for i in range(len(y))], "class": y, "locus": locus,
                             "synteny_type": syn_type, "genes": [len(f) for f in gene_fam],
                             "mean_index_with_its_class": mean_in_class.round(3),
                             "mean_index_with_its_locus": mean_in_locus.round(3)}), "5.4", "b_locus_and_type_of_each_bgc")

    class_mean_syn = pd.DataFrame(index=class_order, columns=class_order, dtype=float)
    for a in class_order:
        for b in class_order:
            block = SYN[np.ix_(y == a, y == b)]
            class_mean_syn.loc[a, b] = block[~np.eye(len(block), dtype=bool)].mean() if a == b else block.mean()
    save_table(class_mean_syn.round(3).rename_axis("class").reset_index(), "5.4", "mean_synteny_between_classes")

    def pair_mean(mask):
        """Mean index of the marked BGC pairs (NaN if there is none)."""
        return round(float(SYN[mask].mean()), 3) if mask.any() else np.nan

    n_loci = pd.Series({c: int(locus[y == c].max()) for c in class_order})
    n_types = pd.Series({c: int(syn_type[y == c].max()) for c in class_order})
    class_summary = pd.DataFrame({
        "BGCs": [counts[c] for c in class_order],
        "loci": n_loci.to_numpy(),
        "BGCs_in_locus_1": [int(((y == c) & (locus == 1)).sum()) for c in class_order],
        "synteny_types": n_types.to_numpy(),
        "BGCs_in_type_1": [int(((y == c) & (syn_type == 1)).sum()) for c in class_order],
        "mean_index_in_class": [pair_mean(same_class & not_self & (y == c)[:, None]) for c in class_order],
        "mean_index_in_same_locus": [pair_mean(same_locus & not_self & (y == c)[:, None]) for c in class_order],
    }, index=pd.Index(class_order, name="class"))
    save_table(class_summary.reset_index(), "5.4", "summary_by_class")

    # ---- 5.4a Synteny matrix, ordered by class → locus → type → tree ---
    class_rank = np.array([class_order.index(c) for c in y])
    syn_rows = np.lexsort((syn_pos, syn_type, locus, class_rank))
    matrix_table = pd.DataFrame(SYN[np.ix_(syn_rows, syn_rows)].round(3), columns=[f"row_{i}" for i in syn_rows])
    matrix_table.insert(0, "class", y[syn_rows])
    matrix_table.insert(0, "data_row", syn_rows)
    save_table(matrix_table, "5.4", "a_synteny_matrix")
    bounds = np.cumsum([0] + [int((y == c).sum()) for c in class_order])
    centres = (bounds[:-1] + bounds[1:]) / 2 - 0.5
    fig, ax = plt.subplots(figsize=(9.8, 9.2))
    im = ax.imshow(SYN[np.ix_(syn_rows, syn_rows)], cmap=SEQ_CMAP, vmin=0, vmax=1, interpolation="nearest")
    for b in bounds[1:-1]:
        ax.axhline(b - 0.5, color=INK, lw=0.7)
        ax.axvline(b - 0.5, color=INK, lw=0.7)
    ax.set_xticks(centres)
    ax.set_xticklabels(class_order, rotation=90)
    ax.set_yticks(centres)
    ax.set_yticklabels([f"{c} ({counts[c]})" for c in class_order])
    ax.tick_params(length=0)
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.7, label="Synteny index (1 = same neighbouring genes)")
    titles(ax, "Synteny between all BGCs",
           f"{len(y)} BGCs ordered by class · dark = similar gene order · each diagonal block is a class")
    show_and_save(fig, "5.4", "a_synteny_matrix")

    # ---- 5.4b Synteny conservation within each class -----------------------------------
    fig, ax = plt.subplots(figsize=(10.5, 0.5 * len(class_order) + 2.0))
    hbox(ax, [mean_in_class[y == c] for c in class_order], [f"{c} ({counts[c]})" for c in class_order])
    for r, c in enumerate(class_order):
        ax.text(1.03, r, f"{n_loci[c]} {'locus' if n_loci[c] == 1 else 'loci'} · {n_types[c]} types", va="center", ha="left", fontsize=9.5, color=INK2)
    ax.set_xlim(0, 1.0)
    ax.set_xlabel("Mean synteny index with the other BGCs of its class")
    titles(ax, "How similar gene order is within each class",
           f"Each dot is a BGC · right: loci and synteny types of the class (types: mean index ≥ {THR})")
    show_and_save(fig, "5.4", "b_synteny_per_class")

    syn_within, syn_between = pair_mean(same_class & not_self), pair_mean(~same_class)
    off = class_mean_syn.where(~np.eye(len(class_order), dtype=bool)).stack()
    print(f"Mean synteny index between BGCs of the same class: {syn_within:.3f} · of different classes: {syn_between:.3f}")
    print(f"Pair of different classes with the most synteny: {off.idxmax()[0]} – {off.idxmax()[1]} ({off.max():.2f})")
    if len(overlaps):                                             # pairs of rows describing overlapping stretches of the same genome (1.3)
        ov_syn = overlaps.assign(syn=[SYN[a, b] for a, b in zip(overlaps["row_a"], overlaps["row_b"])])
        print("Overlapping BGC pairs (1.3), median synteny index · "
              + " · ".join(f"{pair}: {g['syn'].median():.2f} (n = {len(g)})" for pair, g in ov_syn.groupby("pair")))
    if "length_kb" in data.columns:                               # does the gene list fit in the region?
        ratio = np.array([sum(g["length_aa"] for g in genes) * 3e-3 for genes in all_arch]) / data["length_kb"].to_numpy(float)
        print(f"The summed gene lengths exceed the length of the region (length_kb) in "
              f"{int((ratio > 1).sum())} of {len(ratio)} BGCs (median ratio: {np.median(ratio):.2f})")
    display(class_summary)

## 5.5 All clusters aligned in order

BGCs of different loci have no neighbouring genes in common, so they cannot be aligned with one another. Each **locus** is therefore aligned separately, with a progressive alignment of gene order:

1. Each BGC is written as the list of the families of its genes (5.3), in table order.
2. The BGC most similar to the rest of its locus (index of 5.4) is placed first, and at each step the BGC most similar to any of those already aligned is added. Its genes are placed in the columns that already hold a gene of the same family, preserving order. Where the BGC lacks a gene a gap is left; where it has extra genes, new columns are opened.
3. Each BGC is tried in both orientations and the one giving more matches with the BGCs already aligned is kept, because the table does not store the strand.
4. Rows are ordered by synteny type (5.4) and, within each type, by similarity.

One figure is drawn per locus. Loci with fewer than `align_min_bgcs_shown` BGCs are saved in the results folder but not displayed here, and a locus with a single BGC has nothing to align.

**How to read each figure**

- **A · Representatives.** The representative BGC of each main synteny type of the locus. Each gene is a box proportional to the length of its protein, in the same order as in B; the spacing between genes is fixed because the table has no per-gene coordinates. Each representative is shifted so that its first gene in common with the first one sits at the same position.
- **B · All BGCs of the locus.** Each row is a BGC and each cell a gene. Genes of the same family end up in the same column and an empty cell is a gap. A gene without a match is placed in the column after its neighbour's, so a column can also contain genes of another family (shown in a light tone). The labels on top name the most frequent family of the columns shared by at least `align_label_min_fraction` of the BGCs (the gene's first domain and, if it has more, how many); when they do not all fit, the most conserved ones are named.
- **Colour (A and B):** fraction of the locus's BGCs that carry a gene of that same family in that same column (darker = more conserved position).

**Limitations:** the result depends on the order in which BGCs are added (two BGCs with the same gene order can end up in slightly different columns) and on the similarity threshold that defines the families.

Tables `5.5_alignment_genes` (column of every gene), `5.5_alignment_bgcs` (orientation of every BGC) and `5.5_columns` (most frequent family of each column) store the full alignment. A per-locus summary is shown at the end:

| Summary column | What it measures |
|---|---|
| `columns_50pct`, `columns_90pct` | Columns in which at least 50 % (or 90 %) of the locus's BGCs carry a gene of the same family |
| `pct_genes_in_their_column` | Percentage of the genes that sit in a column whose most frequent family is their own; the higher, the cleaner the alignment |

In [ ]:
ALIGN_GAP_KB = 0.12                                              # gap drawn between genes (kb)
# Same blue ramp without its white end: a poorly conserved gene is still visible as a cell
CONS_CMAP = LinearSegmentedColormap.from_list("cons_blue", [SEQ_CMAP(v) for v in np.linspace(0.14, 1, 64)])


def match_to_profile(seq, profile):
    """Matches the genes of a BGC to the columns that already hold a gene of their family, preserving order
    (heaviest common subsequence; a column's weight is the number of BGCs that already have that family there). Returns (score,
    [(gene, column), …])."""
    m, w = len(seq), len(profile)
    weight = np.zeros((m, w))
    positions = {}
    for g, f in enumerate(seq):
        positions.setdefault(f, []).append(g)
    for col, genes in enumerate(profile):
        for f, k in genes.items():
            for g in positions.get(f, ()):
                weight[g, col] = k
    best = np.zeros((m + 1, w + 1))                               # best[g, col]: best score using the first g genes and col columns
    for g in range(m):
        best[g + 1, 1:] = np.maximum.accumulate(np.maximum(best[g, 1:], best[g, :-1] + weight[g]))
    pairs, g, col = [], m, w
    while g > 0 and col > 0:                                      # walk back to recover the matches
        if weight[g - 1, col - 1] > 0 and best[g, col] == best[g - 1, col - 1] + weight[g - 1, col - 1]:
            pairs.append((g - 1, col - 1))
            g, col = g - 1, col - 1
        elif best[g, col] == best[g - 1, col]:
            g -= 1
        else:
            col -= 1
    return best[m, w], pairs[::-1]


def align_locus(idx):
    """Progressive alignment of the BGCs of a locus. Returns {BGC:
    column of each gene, in drawing order}, {BGC: flipped} and the profile [Counter(families) of each column]."""
    sub = SYN[np.ix_(idx, idx)]
    first = int(np.argmax(sub.mean(axis=1)))                     # start with the one most similar to the rest
    order, nearest, left = [first], sub[first].copy(), set(range(len(idx))) - {first}
    while left:                                                  # then always the one most similar to any already aligned
        nxt = max(left, key=lambda j: (nearest[j], -j))
        order.append(nxt)
        left.discard(nxt)
        nearest = np.maximum(nearest, sub[nxt])
    profile, columns, flipped = [], {}, {}

    def open_columns(at, count):
        """Opens `count` empty
        columns at position `at`: the BGCs already placed get a gap there."""
        if count > 0:
            profile[at:at] = [Counter() for _ in range(count)]
            for k, cols in columns.items():
                columns[k] = [col + count if col >= at else col for col in cols]

    for i in idx[order]:
        options = [(*match_to_profile(gene_fam[i][::o], profile), o) for o in (1, -1)]      # both orientations
        score, pairs, o = max(options, key=lambda t: (t[0], t[2]))                       # ties: not flipped
        seq = gene_fam[i][::o]
        if not pairs:                                                    # first BGC (or nothing in common with the previous ones): new columns at the end
            cols = list(range(len(profile), len(profile) + len(seq)))
            open_columns(len(profile), len(seq))
        else:
            cols = [0] * len(seq)
            g0, c0 = pairs[0]
            shift = max(0, g0 - c0)                                      # columns opened so far for this BGC
            open_columns(0, shift)
            cols[:g0 + 1] = range(c0 + shift - g0, c0 + shift + 1)       # genes before the first match, next to it
            for (ga, ca), (gb, cb) in zip(pairs[:-1], pairs[1:]):
                extra = (gb - ga) - (cb - ca)                            # extra genes between two matches
                open_columns(cb + shift, extra)
                shift += max(0, extra)
                cols[ga + 1:gb] = range(cols[ga] + 1, cols[ga] + gb - ga)
                cols[gb] = cb + shift
            g_last = pairs[-1][0]
            cols[g_last + 1:] = range(cols[g_last] + 1, cols[g_last] + len(seq) - g_last)
            open_columns(len(profile), cols[-1] + 1 - len(profile))
        columns[i], flipped[i] = cols, o == -1
        for f, col in zip(seq, cols):
            profile[col][f] += 1
    return columns, flipped, profile


def aligned_figure(c, k_locus, idx):
    """Aligns a locus, draws its figure and returns its tables."""
    n = len(idx)
    columns, flipped, profile = align_locus(idx)
    width = len(profile)
    rows = idx[np.lexsort((syn_pos[idx], syn_type[idx]))]                                # type → tree order
    drawn = {i: (all_arch[i][::-1] if flipped[i] else all_arch[i]) for i in idx}         # genes in drawing order
    fams = {i: (gene_fam[i][::-1] if flipped[i] else gene_fam[i]) for i in idx}
    share = {i: [profile[col][f] / n for col, f in zip(columns[i], fams[i])] for i in idx}     # fraction of the locus with that family in that column
    cons = np.full((n, width), np.nan)
    key = {"class": c, "locus": k_locus}
    gene_rows = []
    for r, i in enumerate(rows):
        numbers = range(len(drawn[i]), 0, -1) if flipped[i] else range(1, len(drawn[i]) + 1)
        for col, g, f, v in zip(columns[i], numbers, fams[i], share[i]):
            cons[r, col] = v
            gene_rows.append({**key, "data_row": i, "BGC": bgc_label(i), "synteny_type": syn_type[i], "gene": g,
                              "column": col + 1, "family": f, "name": fam_name[f],
                              "pct_BGCs_with_that_family_in_that_column": round(100 * v, 1)})
    bgc_rows = [{**key, "data_row": i, "BGC": bgc_label(i), "synteny_type": syn_type[i],
                 "flipped": flipped[i], "genes": len(drawn[i]), "first_column": columns[i][0] + 1,
                 "last_column": columns[i][-1] + 1} for i in rows]
    occupied = Counter(col for i in idx for col in columns[i])
    top = [genes.most_common(1)[0] for genes in profile]                                 # most frequent family of each column and how many BGCs have it there
    col_rows = [{**key, "column": col + 1, "family": f, "name": fam_name[f], "n_BGC_with_that_family": k,
                 "pct_of_locus": round(100 * k / n, 1), "n_BGC_with_any_gene": occupied[col]} for col, (f, k) in enumerate(top)]
    matched = [top[col][0] == f for i in idx for col, f in zip(columns[i], fams[i])]
    stats = {**key, "BGCs": n, "synteny_types": len(set(syn_type[idx].tolist())), "flipped_BGCs": sum(flipped.values()),
             "columns_50pct": sum(k >= 0.5 * n for _, k in top), "columns_90pct": sum(k >= 0.9 * n for _, k in top),
             "pct_genes_in_their_column": round(100 * float(np.mean(matched)), 1)}
    if n < 2:                                                                            # a single BGC: nothing to align or draw
        return bgc_rows, gene_rows, col_rows, stats

    # synteny types of the locus, in row order: those with few BGCs get no label
    sizes = Counter(syn_type[idx].tolist())
    big = [t for t in sorted(sizes) if sizes[t] >= max(3, 0.04 * n)] or [min(sizes, key=lambda t: (-sizes[t], t))]
    blocks, start = [], 0                                                                # [label or None, first row, last row + 1]
    for t in sorted(sizes):
        if t in big:
            blocks.append([f"type {t} · n = {sizes[t]}", start, start + sizes[t]])
        elif blocks and blocks[-1][0] is None:                                           # consecutive small types: a single block
            blocks[-1][2] = start + sizes[t]
        else:
            blocks.append([None, start, start + sizes[t]])
        start += sizes[t]
    blocks = [(label or f"other types · n = {b - a}", a, b) for label, a, b in blocks]
    rep_of = synteny_types[synteny_types["class"] == c].set_index("type")["representative_row"]
    reps = [int(rep_of[t]) for t in big[:CONFIG["align_representatives"]]]

    # ---- Sizes in inches: the figure grows with the number of columns and BGCs ----
    mat_w = float(np.clip(0.2 * width, 7.0, 16.0))
    col_w = mat_w / width
    # top labels: tilted if columns are wide, vertical if narrow; if they do not all fit, most conserved first
    rotation = 55 if col_w >= 0.195 else 90
    step = int(np.ceil((1.15 * 8.5 / 72) / (col_w * np.sin(np.radians(rotation)))))      # columns between two labels so they do not overlap
    shown = []
    for k, col, f in sorted(((k, col, f) for col, (f, k) in enumerate(top)), key=lambda t: (-t[0], t[1])):
        if k >= CONFIG["align_label_min_fraction"] * n and all(abs(col - other) >= step for other, _ in shown):
            shown.append((col, gene_label(fam_name[f])))
    shown.sort()
    label_h = 0.62 * 8.5 / 72 * max([len(t) for _, t in shown] or [0]) * np.sin(np.radians(rotation)) + 0.2      # height of the labels
    left, right, top_m, bottom = 2.5, 1.9, 1.05, 0.25
    hA, hB = 0.52 * len(reps) + 0.45, max(0.05 * n, min(2.8, 0.25 * n), 1.8)       # minimum height: the colour bar fits
    bar_h, gap = min(2.0, hB), label_h + 0.75
    fig_w, fig_h = left + mat_w + right, top_m + hA + gap + hB + bottom
    fig = plt.figure(figsize=(fig_w, fig_h))
    axA = fig.add_axes([left / fig_w, (bottom + hB + gap) / fig_h, mat_w / fig_w, hA / fig_h])
    axB = fig.add_axes([left / fig_w, bottom / fig_h, mat_w / fig_w, hB / fig_h])
    cax = fig.add_axes([(left + mat_w + 0.3) / fig_w, (bottom + hB - bar_h) / fig_h, 0.16 / fig_w, bar_h / fig_h])

    # ---- A representatives: boxes proportional to each protein; shifted so that their first gene in common with the first one coincides ----
    def gene_starts(i):
        """{column: (start in kb, family)} of every gene of the BGC."""
        x, out = 0.0, {}
        for col, gene, f in zip(columns[i], drawn[i], fams[i]):
            out[col] = (x, f)
            x += gene["length_aa"] * 3e-3 + ALIGN_GAP_KB
        return out

    ref_x = gene_starts(reps[0])
    xmin, xmax = 0.0, 1.0
    for r, i in enumerate(reps):
        xs = gene_starts(i)
        same = [col for col in sorted(xs) if col in ref_x and xs[col][1] == ref_x[col][1]]
        shift = ref_x[same[0]][0] - xs[same[0]][0] if same else 0.0
        for col, gene, v in zip(columns[i], drawn[i], share[i]):
            x, length = xs[col][0] + shift, gene["length_aa"] * 3e-3
            axA.add_patch(FancyBboxPatch((x, -r - 0.3), length, 0.6, boxstyle="round,pad=0,rounding_size=0.06", mutation_aspect=1,
                                         facecolor=CONS_CMAP(v), edgecolor="none"))
            xmin, xmax = min(xmin, x), max(xmax, x + length)
        x0, x1 = min(v[0] for v in xs.values()) + shift, max(v[0] for v in xs.values()) + shift
        axA.plot([x0, x1], [-r, -r], color=AXIS, lw=1, zorder=0)                         # cluster backbone
        axA.text(-0.012, -r, f"type {syn_type[i]} · n = {sizes[syn_type[i]]}" + (" · flipped" if flipped[i] else "") + f"\n{bgc_label(i)}",
                 transform=axA.get_yaxis_transform(), ha="right", va="center", fontsize=8.5, color=INK, linespacing=1.2)
    bar = 5 if xmax - xmin > 40 else 2 if xmax - xmin > 12 else 1                         # scale bar
    axA.plot([xmax - bar, xmax], [-len(reps) + 0.3] * 2, color=INK, lw=1.4)
    axA.text(xmax - bar / 2, -len(reps) + 0.2, f"{bar} kb", ha="center", va="top", fontsize=8.5, color=INK2)
    axA.set_xlim(xmin - 0.01 * (xmax - xmin), xmax + 0.01 * (xmax - xmin))
    axA.set_ylim(-len(reps) - 0.05, 0.6)
    axA.axis("off")

    # ---- B all BGCs of the locus: one row per BGC, one cell per gene ------------
    im = axB.imshow(cons, cmap=CONS_CMAP, vmin=0, vmax=1, aspect="auto", interpolation="nearest", extent=(-0.5, width - 0.5, n - 0.5, -0.5))
    for x in np.arange(0.5, width - 0.5, 1):
        axB.axvline(x, color=SURFACE, lw=0.7)
    for label, a, b in blocks:
        if a > 0:
            axB.axhline(a - 0.5, color=INK, lw=1)
        axB.text(-0.012, (a + b) / 2 - 0.5, label, transform=axB.get_yaxis_transform(), ha="right", va="center", fontsize=9, color=INK)
    axB.set_xticks([col for col, _ in shown])
    axB.set_xticklabels([t for _, t in shown], rotation=rotation, fontsize=8.5, **({"ha": "left", "rotation_mode": "anchor"} if rotation < 90 else {"ha": "center"}))
    axB.xaxis.tick_top()
    axB.tick_params(axis="x", length=0, pad=2)
    axB.set_yticks([])
    axB.set_xlim(-0.5, width - 0.5)
    axB.set_ylim(n - 0.5, -0.5)
    axB.grid(False)
    for spine in axB.spines.values():
        spine.set_visible(False)
    cb = fig.colorbar(im, cax=cax)
    cb.set_label("Fraction of BGCs with that\nfamily in that column", fontsize=9)
    cb.outline.set_visible(False)

    several = int(n_loci[c]) > 1
    fig.text(0.1 / fig_w, 1 - 0.12 / fig_h, f"{c}" + (f" · locus {k_locus}" if several else "") + f": {n} BGCs aligned in order"
             + (f" (of the {counts[c]} in the class)" if several else ""), ha="left", va="top", fontsize=13, fontweight="bold", color=INK)
    fig.text(0.1 / fig_w, 1 - 0.43 / fig_h, f"Each cell is a gene · genes of the same family share a column · darker = more conserved position · empty = gap · "
             + ("1 BGC flipped" if sum(flipped.values()) == 1 else f"{sum(flipped.values())} BGCs flipped"),
             ha="left", va="top", fontsize=10, color=INK2)
    fig.text(0.1 / fig_w, (bottom + hB + gap + hA + 0.08) / fig_h, "A · Representative of each main synteny type (gene sizes to scale)",
             ha="left", va="bottom", fontsize=11, fontweight="bold", color=INK)
    fig.text(0.1 / fig_w, (bottom + hB + label_h + 0.22) / fig_h, f"B · All {n} BGCs (top: family of the columns shared by at least "
             f"{100 * CONFIG['align_label_min_fraction']:.0f} % of the BGCs)", ha="left", va="bottom", fontsize=11, fontweight="bold", color=INK)
    show_and_save(fig, "5.5", f"alignment_{slug(c)}_locus{k_locus}", individual=n < CONFIG["align_min_bgcs_shown"])
    return bgc_rows, gene_rows, col_rows, stats


if COMPARE_OK:
    bgc_rows, gene_rows, col_rows, locus_rows = [], [], [], []
    for c in class_order:
        for k_locus in range(1, int(n_loci[c]) + 1):
            b, g, k, s = aligned_figure(c, k_locus, np.where((y == c) & (locus == k_locus))[0])
            bgc_rows += b
            gene_rows += g
            col_rows += k
            locus_rows.append(s)
    save_table(pd.DataFrame(bgc_rows), "5.5", "alignment_bgcs")
    save_table(pd.DataFrame(gene_rows), "5.5", "alignment_genes")
    save_table(pd.DataFrame(col_rows), "5.5", "columns")
    locus_summary = save_table(pd.DataFrame(locus_rows), "5.5", "summary_by_locus")
    display(locus_summary[locus_summary["BGCs"] >= CONFIG["align_min_bgcs_shown"]].set_index(["class", "locus"]))

---
# Annex · Exported model, summary and generated files

| Section | What it does | Outputs |
|---|---|---|
| A.1 | Saves the final model and shows how to use it on new BGCs | `A.1_svm_model.joblib` |
| A.2 | Gathers the key figures of the whole notebook in one table | `A.2_results_summary.csv` |
| A.3 | Lists every generated file, by folder | `A.3_generated_files.csv` |

## A.1 Exported model and usage example

The `joblib` file holds the fitted pipeline (scaling + SVM), the ordered list of Pfam columns it expects, the classes, the configuration and the library versions. **New data must not be scaled**: scaling is inside the pipeline. Pfam columns missing from a new table are filled with 0 (domain absent).

> `joblib` runs code when loading a file: only open models from trusted sources.

In [ ]:
import platform
import joblib
import sklearn

bundle = {
    "model": final_model,
    "feature_names": selected_features,          # CSV columns it expects, in this order
    "classes": list(final_model.classes_),
    "target_column": TARGET,
    "k_domains": k_optimal,
    "C": C_final,
    "metrics": {"cv_accuracy_train": round(cv_final, 4), "test_accuracy": float(metrics_test.at[0, "accuracy"]),
                "permutation_p": float(p_perm)},
    "config": {k: (list(v) if isinstance(v, (tuple, np.ndarray)) else v) for k, v in CONFIG.items()},
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                 "scikit-learn": sklearn.__version__},
}
model_path = out_path("A.1", "svm_model", "joblib")
joblib.dump(bundle, model_path, compress=3)
print(f"Model saved: {model_path} ({model_path.stat().st_size / 1024:.0f} kB)")

# ---- Usage example: the first 5 CSV rows are used as if they were new BGCs --
# (this only demonstrates the format: those rows may have been in training)
loaded = joblib.load(model_path)
new_bgcs = pd.read_csv(CONFIG["input_csv"], encoding="utf-8-sig", low_memory=False).head(5)
X_new = new_bgcs.reindex(columns=loaded["feature_names"]).fillna(0.0).to_numpy(float)     # same column order; missing = 0
pd.DataFrame({"class_in_csv": new_bgcs[TARGET], "predicted_class": loaded["model"].predict(X_new),
              "probability": loaded["model"].predict_proba(X_new).max(axis=1).round(3)})

## A.2 Results summary

The key figures of each part, with the section where they are computed. This table is the source for any number quoted in an abstract or poster.

In [ ]:
G = globals()
rows = [
    ("1.2", "BGCs analysed (after merging GenBank/RefSeq and excluding rare classes)", len(y)),
    ("1.2", "Genomes", data["genome"].nunique()),
    ("1.3", "Retained classes", f"{len(class_order)}: " + ", ".join(f"{c} ({int((y == c).sum())})" for c in class_order)),
    ("1.3", "Overlapping BGC pairs from the same genome", len(overlaps)),
    ("1.4", "Pfam domains in the matrix", X.shape[1]),
    ("1.4", "Distinct presence/absence profiles", n_profiles),
    ("2.1", "Training / test BGCs", f"{len(idx_train)} / {len(idx_test)}"),
    ("2.2", "Domains used by the model (k)", k_optimal),
    ("2.3", "Chosen C", C_final),
    ("2.5", "Permutation test: p", round(float(p_perm), 4)),
    ("2.6", "Cross-validated accuracy (training)", f"{cv_final:.3f} ± {cv_final_se:.3f}"),
    ("3.1", "Held-out test accuracy (95 % CI)",
     f"{metrics_test.at[0, 'accuracy']:.3f} ({ci_low:.3f}–{ci_high:.3f}) · {n_correct}/{len(y_test)}"),
    ("3.1", "Balanced accuracy / macro F1 / MCC (test)",
     f"{metrics_test.at[0, 'balanced_accuracy']:.3f} / {metrics_test.at[0, 'f1_macro']:.3f} / {metrics_test.at[0, 'mcc']:.3f}"),
    ("3.1", "Training − test gap", round(gap, 3)),
    ("3.1", "Accuracy on test BGCs with / without an identical training profile",
     f"{acc_twin:.3f} (n = {int(has_twin.sum())}) / {acc_new:.3f} (n = {int((~has_twin).sum())})"),
    ("3.4", "Test errors (of which in overlapping BGCs)", f"{int(wrong.sum())} ({n_err_overlap})"),
    ("3.4", "Errors (true → predicted)", "; ".join(f"{t} → {p}" for t, p in zip(y_test[wrong], y_test_pred[wrong])) or "—"),
    ("4.1", "Groups (size)", ", ".join(f"{g} ({group_sizes[g]})" for g in group_names)),
    ("4.1", "Silhouette at the k used / k with the highest silhouette",
     f"{sil.set_index('k').at[k_groups, 'silhouette']:.3f} / {best_k}"),
]
for g in group_names:
    comp = composition.loc[g]
    rows.append(("4.1", f"Composition of {g}", ", ".join(f"{c} {n}" for c, n in comp[comp > 0].sort_values(ascending=False).items())))
for g in group_names:
    top3 = markers[markers["group"] == g].head(3)
    rows.append(("4.2", f"Markers of {g}", "; ".join(
        f"{r['domain']} {r['direction']} ({100 * r['prevalence_group']:.0f} % vs {100 * r['prevalence_rest']:.0f} %)"
        for _, r in top3.iterrows())))
rows.append(("4.2", "Decision tree: balanced accuracy (cross-validation)", f"{tree_cv.mean():.3f}"))
if G.get("ARCH_OK"):
    for g, i in reps:
        rows.append(("4.3", f"Representative of {g}", f"{y[i]} · {bgc_label(i)}"))
    for c in G.get("show_classes", []):
        v = per_bgc.loc[y == c, "modules"]
        rows.append(("4.5", f"NRPS modules per BGC (median, max) · {c}", f"{v.median():g}, {v.max():g}"))
if G.get("SEQ_OK"):
    rows.append(("4.6", "Mean alignment distance within / between classes", f"{intra:.3f} / {inter:.3f}"))
    rows.append(("4.6", "Mantel: sequence versus model domains", f"r = {r_obs:.3f}, p = {p_mantel:.4f}"))

# ---- Part 5 · Cluster comparison ----
rows.append(("5.2", "Domains frequent (≥ 50 % of BGCs) in 2 or more classes", len(shared)))
rows.append(("5.2", "Domains frequent in the most classes (number of classes)",
             f"{', '.join(widest['domain'].head(6))} ({recurrence[FREQ].max()})"))
rows.append(("5.2", "Domains in the most BGCs",
             "; ".join(f"{r['domain']} ({r['n_BGC']} BGCs, {r['pct_BGC']:g} %)" for _, r in recurrence.head(3).iterrows())))
if G.get("ARCH_OK"):
    rows.append(("4.3", "Domain presences of the table that appear in the map",
                 f"{int((P & in_map).sum()):,} / {int(P.sum()):,} ({100 * map_coverage:.1f} %)"))
if G.get("COMPARE_OK"):
    rows.append(("5.3", f"Gene families (similarity ≥ {CONFIG['gene_family_min_similarity']})", f"{len(members):,} ({len(gene_index):,} genes)"))
    rows.append(("5.3", "BGCs with a gene repeated within the same cluster (same family)", f"{n_bgc_rep} / {len(y)}"))
    rows.append(("5.3", "BGCs with 2 or more genes sharing a domain architecture, regardless of sequence",
                 f"{n_same_arch} / {len(y)}"))
    rows.append(("5.3", "Genes repeated in the most BGCs (BGCs; median similarity between copies)",
                 "; ".join(f"{r['gene']} ({r['n_BGC']}; {r['median_similarity']:.2f})" for _, r in gene_summary.head(3).iterrows())))
    rows.append(("5.3", "Cases (BGC, family) with near-identical copies (similarity ≥ 0.9)",
                 f"{len(near)} / {len(repeated_genes)}" + (" · " + ", ".join(f"{g}: {k}" for g, k in near["gene"].value_counts().items()) if len(near) else "")))
    rows.append(("5.3", "BGCs with a domain repeated within one gene (if every tag is counted)",
                 f"{n_bgc_dom} ({n_bgc_dom_runs}) / {len(y)}"))
    rows.append(("5.3", "Domains repeated within one gene in the most BGCs (BGCs; max copies per gene)",
                 "; ".join(f"{r['domain']} ({r['n_BGC']}; {r['max_copies']})" for _, r in domain_summary.head(3).iterrows())))
    rows.append(("5.4", "Mean synteny index within / between classes", f"{syn_within:.3f} / {syn_between:.3f}"))
    rows.append(("5.4", "Pair of different classes with the most synteny", f"{off.idxmax()[0]} – {off.idxmax()[1]} ({off.max():.2f})"))
    rows.append(("5.4", "Loci per class (BGCs in locus 1)",
                 "; ".join(f"{c} {n_loci[c]} ({class_summary.loc[c, 'BGCs_in_locus_1']})" for c in class_order)))
    rows.append(("5.4", "Synteny types per class (BGCs in type 1)",
                 "; ".join(f"{c} {n_types[c]} ({class_summary.loc[c, 'BGCs_in_type_1']})" for c in class_order)))
    first = locus_summary[locus_summary["locus"] == 1].set_index("class")
    rows.append(("5.5", "Columns with the same family in ≥ 90 % of the BGCs, in locus 1 of each class",
                 "; ".join(f"{c} {first.loc[c, 'columns_90pct']}" for c in class_order)))
rows.sort(key=lambda r: [int(x) if x.isdigit() else 99 for x in r[0].split(".")])      # by section

summary_table = save_table(pd.DataFrame(rows, columns=["section", "indicator", "value"]),
                           "A.2", "results_summary")
pd.set_option("display.max_colwidth", 140)
summary_table

## A.3 Generated files

Every file the notebook produced, by folder and type. Each file name starts with the section that produces it.

In [ ]:
files = sorted(p for p in OUT.rglob("*") if p.is_file())
listing = save_table(pd.DataFrame({
    "folder": [p.parent.name for p in files],
    "file": [p.name for p in files],
    "section": [p.name.split("_", 1)[0] for p in files],
    "type": [p.suffix.lstrip(".") for p in files],
    "kB": [round(p.stat().st_size / 1024, 1) for p in files]}), "A.3", "generated_files")
print(f"{len(files)} files in the folder {OUT}/")
listing.pivot_table(index="folder", columns="type", values="kB", aggfunc="count", fill_value=0)